# Task A — Attention Capacity & Interference

**Track:** Attention — Attention Capacity
**Benchmark:** CogAttention v1.0
**Subtasks:** `capacity` (Thread Tracking), `interference` (Interference Chain), `blink` (Attentional Blink)

---

## What This Notebook Does

This notebook benchmarks an LLM's **attention capacity** — how many concurrent information threads it can track, how well it resists proactive interference from prior context, and whether it exhibits attentional blink (missing a second target presented shortly after a first).

### Subtask Breakdown

| Subtask | Paradigm | What It Measures |
|---------|----------|-----------------|
| **Thread Tracking** | Multi-Object Tracking (Pylyshyn & Storm, 2001) | Can the model track N concurrent narrative threads and recall per-thread details? Scales thread count (3→12) and context length across difficulty tiers. |
| **Interference Chain** | Proactive Interference / PI-LLM (Wang & Sun, 2025) | Does earlier context interfere with recall of later facts? The model sees multiple similar-but-conflicting fact chains and must retrieve the most recent version. |
| **Attentional Blink** | Rapid Serial Visual Presentation (RSVP) | When two targets appear in quick succession within a fast stream, can the model detect both? Tests temporal attention gaps. |

### Cognitive Science Grounding

- **Attention capacity** (Pylyshyn & Storm, 2001): Humans can track ~4 objects simultaneously; this benchmark pushes LLMs with 3–12 threads.
- **Proactive interference** (Underwood, 1957; Wang & Sun, 2025): Previously learned information disrupts recall of newer information — a core failure mode in long-context LLMs.
- **Attentional blink** (Raymond et al., 1992): A 200–500ms window after detecting a first target where a second target is often missed.

### Difficulty Scaling

| Level | Thread Tracking | Interference Chain | Attentional Blink |
|-------|----------------|-------------------|-------------------|
| Easy | 3 threads, short context | 2 conflicting chains | Wide target spacing |
| Medium | 5 threads | 3 chains | Moderate spacing |
| Hard | 7 threads | 5 chains, high similarity | Narrow spacing |
| Expert | 10 threads, long context | 7 chains | Minimal spacing |
| Frontier | 12 threads, max context | 10+ chains, near-identical | Adjacent targets |

### Scoring

SDK assertion pass rate = per-element accuracy. Each tracked thread / recalled fact / detected target is a separate assertion for fine-grained scoring.

---

`<!-- COGATTENTION-BENCH-CANARY-C41FF31A0E4F -->`


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 2: Imports + Inline Helpers
# CogAttention — Attention Capacity
# ══════════════════════════════════════════════════════════════════════

import kaggle_benchmarks as kbench

import json
import re

def extract_answer_block(response):
    for pat in [r"ANSWER:\s*(.*)", r"Answer:\s*(.*)", r"answer:\s*(.*)"]:
        match = re.search(pat, response, re.DOTALL | re.IGNORECASE)
        if match:
            return match.group(1).strip()
    return response.strip()

def extract_numbered_answers(response):
    answer_block = extract_answer_block(response)
    results = {}
    matches = re.findall(
        r"(\d+)\s*[.):\-]\s*(.+?)(?=\n\d+\s*[.):\-]|\Z)",
        answer_block, re.DOTALL,
    )
    for num, val in matches:
        results[num] = val.strip().rstrip(".")
    return results

def extract_list_items(response):
    answer_block = extract_answer_block(response)
    bullets = re.findall(r"[-\u2022]\s*(.+?)(?:\n|$)", answer_block)
    if bullets:
        return [b.strip().rstrip(".") for b in bullets]
    numeric_items = re.findall(
        r'[\$]?\d{1,3}(?:,\d{3})*(?:\.\d+)?(?:\s*(?:\xb0[CF]|mg/L|%|\$))?',
        answer_block,
    )
    if numeric_items and len(numeric_items) >= 2:
        return [x.strip() for x in numeric_items]
    if "," in answer_block:
        items = [x.strip().rstrip(".") for x in answer_block.split(",")]
        return [x for x in items if x]
    lines = [l.strip().rstrip(".") for l in answer_block.split("\n") if l.strip()]
    return lines if lines else ([answer_block] if answer_block else [])

def extract_person_item_pairs(response):
    answer_block = extract_answer_block(response)
    results = {}
    for pat in [
        r"[-\u2022]?\s*(\w+)\s*:\s*(.+?)(?:\n|$)",
        r"[-\u2022]?\s*(\w+)\s+holds?\s+(?:a\s+)?(.+?)(?:\n|$)",
    ]:
        matches = re.findall(pat, answer_block, re.IGNORECASE)
        if matches:
            for name, item in matches:
                results[name.strip()] = item.strip().rstrip(".")
            break
    return results

def fuzzy_value_match(predicted, gold):
    pred_clean = re.sub(r"\s+", " ", predicted.strip().lower())
    gold_clean = re.sub(r"\s+", " ", gold.strip().lower())
    if pred_clean == gold_clean:
        return True
    if gold_clean in pred_clean:
        return True
    try:
        pred_num = float(re.sub(r"[,$%\xb0]", "", predicted))
        gold_num = float(re.sub(r"[,$%\xb0]", "", gold))
        return pred_num == gold_num
    except (ValueError, TypeError):
        pass
    return False

def _escape_for_regex(s):
    return re.escape(s).replace(r"\ ", r"\s+")


def run_assertions_capacity(response, gold, kbench):
    for person in gold["people"]:
        gold_item = gold["answers"][person]
        pattern = rf"(?i){re.escape(person)}\s*[:.\\-]\s*.*{_escape_for_regex(gold_item)}"
        kbench.assertions.assert_contains_regex(
            pattern, response,
            expectation=f"{person} should hold '{gold_item}'"
        )


def run_assertions_interference(response, gold, kbench):
    for key in gold["key_names"]:
        gold_val = gold["final_values"][key]
        pattern = rf"(?i){re.escape(key)}\s*[:\-=]\s*.*{_escape_for_regex(gold_val)}"
        kbench.assertions.assert_contains_regex(
            pattern, response,
            expectation=f"Final value of '{key}' should be '{gold_val}'"
        )


def run_assertions_blink(response, gold, kbench):
    for target_key in ["t1", "t2"]:
        gold_val = gold[target_key]
        label = target_key.upper()
        pattern = rf"(?i){re.escape(label)}\s*[:\-=]\s*.*{_escape_for_regex(gold_val)}"
        kbench.assertions.assert_contains_regex(
            pattern, response,
            expectation=f"{label} should be '{gold_val}'"
        )


print("CogAttention helpers loaded")
print(f"Task types: ['capacity', 'interference', 'blink']")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 3: Task Definitions + Embedded Dataset
# ══════════════════════════════════════════════════════════════════════


@kbench.task(name="cogattention_capacity")
def cogattention_capacity(llm, prompt: str, gold_json: str, task_id: str, difficulty: str):
    """CogAttention capacity task."""
    response = llm.prompt(prompt)
    gold = json.loads(gold_json)
    run_assertions_capacity(response, gold, kbench)


@kbench.task(name="cogattention_interference")
def cogattention_interference(llm, prompt: str, gold_json: str, task_id: str, difficulty: str):
    """CogAttention interference task."""
    response = llm.prompt(prompt)
    gold = json.loads(gold_json)
    run_assertions_interference(response, gold, kbench)


@kbench.task(name="cogattention_blink")
def cogattention_blink(llm, prompt: str, gold_json: str, task_id: str, difficulty: str):
    """CogAttention blink task."""
    response = llm.prompt(prompt)
    gold = json.loads(gold_json)
    run_assertions_blink(response, gold, kbench)


# ── Embedded dataset ──────────────────────────────────────────────────
DATASET = json.loads(r'''
[
 {
  "task_id": "capacity_easy_000",
  "task_type": "capacity",
  "difficulty": "Easy",
  "prompt": "You are given 2 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Willa holds a ivory bell\n- Olena holds a ochre shell\n\nSwaps:\n1. Willa and Olena swap items.\n2. Willa and Olena swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Willa\": \"ivory bell\", \"Olena\": \"ochre shell\"}, \"people\": [\"Willa\", \"Olena\"]}"
 },
 {
  "task_id": "capacity_easy_001",
  "task_type": "capacity",
  "difficulty": "Easy",
  "prompt": "You are given 2 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Viktor holds a jade shell\n- Xander holds a onyx pendant\n\nSwaps:\n1. Viktor and Xander swap items.\n2. Viktor and Xander swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Viktor\": \"jade shell\", \"Xander\": \"onyx pendant\"}, \"people\": [\"Viktor\", \"Xander\"]}"
 },
 {
  "task_id": "capacity_easy_002",
  "task_type": "capacity",
  "difficulty": "Easy",
  "prompt": "You are given 2 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Zain holds a russet shell\n- Kaia holds a teal coin\n\nSwaps:\n1. Zain and Kaia swap items.\n2. Zain and Kaia swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Zain\": \"russet shell\", \"Kaia\": \"teal coin\"}, \"people\": [\"Zain\", \"Kaia\"]}"
 },
 {
  "task_id": "capacity_easy_003",
  "task_type": "capacity",
  "difficulty": "Easy",
  "prompt": "You are given 2 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Femi holds a slate flask\n- Soren holds a amber dice\n\nSwaps:\n1. Femi and Soren swap items.\n2. Femi and Soren swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Femi\": \"slate flask\", \"Soren\": \"amber dice\"}, \"people\": [\"Femi\", \"Soren\"]}"
 },
 {
  "task_id": "capacity_easy_004",
  "task_type": "capacity",
  "difficulty": "Easy",
  "prompt": "You are given 2 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Dariush holds a bronze key\n- Ines holds a green compass\n\nSwaps:\n1. Dariush and Ines swap items.\n2. Dariush and Ines swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Dariush\": \"bronze key\", \"Ines\": \"green compass\"}, \"people\": [\"Dariush\", \"Ines\"]}"
 },
 {
  "task_id": "capacity_easy_005",
  "task_type": "capacity",
  "difficulty": "Easy",
  "prompt": "You are given 2 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Ines holds a russet locket\n- Amara holds a silver bell\n\nSwaps:\n1. Ines and Amara swap items.\n2. Ines and Amara swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Ines\": \"russet locket\", \"Amara\": \"silver bell\"}, \"people\": [\"Ines\", \"Amara\"]}"
 },
 {
  "task_id": "capacity_easy_006",
  "task_type": "capacity",
  "difficulty": "Easy",
  "prompt": "You are given 2 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Wren holds a copper chalice\n- Yuki holds a green dagger\n\nSwaps:\n1. Wren and Yuki swap items.\n2. Wren and Yuki swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Wren\": \"copper chalice\", \"Yuki\": \"green dagger\"}, \"people\": [\"Wren\", \"Yuki\"]}"
 },
 {
  "task_id": "capacity_easy_007",
  "task_type": "capacity",
  "difficulty": "Easy",
  "prompt": "You are given 2 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Tariq holds a pearl stone\n- Magnus holds a slate lantern\n\nSwaps:\n1. Tariq and Magnus swap items.\n2. Tariq and Magnus swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Tariq\": \"pearl stone\", \"Magnus\": \"slate lantern\"}, \"people\": [\"Tariq\", \"Magnus\"]}"
 },
 {
  "task_id": "capacity_medium_008",
  "task_type": "capacity",
  "difficulty": "Medium",
  "prompt": "You are given 3 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Maren holds a bronze mask\n- Xander holds a amber dice\n- Tariq holds a blue mirror\n\nSwaps:\n1. Maren and Tariq swap items.\n2. Xander and Tariq swap items.\n3. Maren and Xander swap items.\n4. Maren and Tariq swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Maren\": \"amber dice\", \"Xander\": \"blue mirror\", \"Tariq\": \"bronze mask\"}, \"people\": [\"Maren\", \"Xander\", \"Tariq\"]}"
 },
 {
  "task_id": "capacity_medium_009",
  "task_type": "capacity",
  "difficulty": "Medium",
  "prompt": "You are given 3 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Tala holds a cobalt chalice\n- Paloma holds a russet candle\n- Soren holds a onyx locket\n\nSwaps:\n1. Paloma and Soren swap items.\n2. Tala and Paloma swap items.\n3. Tala and Soren swap items.\n4. Tala and Paloma swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Tala\": \"cobalt chalice\", \"Paloma\": \"russet candle\", \"Soren\": \"onyx locket\"}, \"people\": [\"Tala\", \"Paloma\", \"Soren\"]}"
 },
 {
  "task_id": "capacity_medium_010",
  "task_type": "capacity",
  "difficulty": "Medium",
  "prompt": "You are given 3 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Olena holds a amber dice\n- Kenji holds a onyx scroll\n- Zora holds a bronze pendant\n\nSwaps:\n1. Olena and Zora swap items.\n2. Kenji and Zora swap items.\n3. Olena and Kenji swap items.\n4. Kenji and Zora swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Olena\": \"amber dice\", \"Kenji\": \"onyx scroll\", \"Zora\": \"bronze pendant\"}, \"people\": [\"Olena\", \"Kenji\", \"Zora\"]}"
 },
 {
  "task_id": "capacity_medium_011",
  "task_type": "capacity",
  "difficulty": "Medium",
  "prompt": "You are given 3 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Joelle holds a teal locket\n- Amara holds a silver dagger\n- Willa holds a green coin\n\nSwaps:\n1. Amara and Willa swap items.\n2. Joelle and Willa swap items.\n3. Joelle and Amara swap items.\n4. Amara and Willa swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Joelle\": \"green coin\", \"Amara\": \"teal locket\", \"Willa\": \"silver dagger\"}, \"people\": [\"Joelle\", \"Amara\", \"Willa\"]}"
 },
 {
  "task_id": "capacity_medium_012",
  "task_type": "capacity",
  "difficulty": "Medium",
  "prompt": "You are given 3 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Orla holds a copper shell\n- Elio holds a onyx candle\n- Amara holds a green coin\n\nSwaps:\n1. Orla and Amara swap items.\n2. Orla and Elio swap items.\n3. Elio and Amara swap items.\n4. Orla and Amara swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Orla\": \"green coin\", \"Elio\": \"copper shell\", \"Amara\": \"onyx candle\"}, \"people\": [\"Orla\", \"Elio\", \"Amara\"]}"
 },
 {
  "task_id": "capacity_medium_013",
  "task_type": "capacity",
  "difficulty": "Medium",
  "prompt": "You are given 3 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Zora holds a indigo key\n- Nico holds a cobalt coin\n- Runa holds a bronze dice\n\nSwaps:\n1. Zora and Runa swap items.\n2. Zora and Nico swap items.\n3. Nico and Runa swap items.\n4. Zora and Nico swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Zora\": \"indigo key\", \"Nico\": \"cobalt coin\", \"Runa\": \"bronze dice\"}, \"people\": [\"Zora\", \"Nico\", \"Runa\"]}"
 },
 {
  "task_id": "capacity_medium_014",
  "task_type": "capacity",
  "difficulty": "Medium",
  "prompt": "You are given 3 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Dmitri holds a pearl key\n- Dariush holds a indigo coin\n- Colette holds a green ring\n\nSwaps:\n1. Dmitri and Dariush swap items.\n2. Dariush and Colette swap items.\n3. Dmitri and Dariush swap items.\n4. Dmitri and Colette swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Dmitri\": \"pearl key\", \"Dariush\": \"indigo coin\", \"Colette\": \"green ring\"}, \"people\": [\"Dmitri\", \"Dariush\", \"Colette\"]}"
 },
 {
  "task_id": "capacity_medium_015",
  "task_type": "capacity",
  "difficulty": "Medium",
  "prompt": "You are given 3 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Freya holds a silver dice\n- Femi holds a crimson stone\n- Joelle holds a green ring\n\nSwaps:\n1. Femi and Joelle swap items.\n2. Freya and Joelle swap items.\n3. Femi and Joelle swap items.\n4. Freya and Femi swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Freya\": \"silver dice\", \"Femi\": \"crimson stone\", \"Joelle\": \"green ring\"}, \"people\": [\"Freya\", \"Femi\", \"Joelle\"]}"
 },
 {
  "task_id": "capacity_hard_016",
  "task_type": "capacity",
  "difficulty": "Hard",
  "prompt": "You are given 4 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Greta holds a pearl shell\n- Adaeze holds a ochre coin\n- Nalini holds a cobalt mirror\n- Kaia holds a coral stone\n\nSwaps:\n1. Adaeze and Nalini swap items.\n2. Greta and Kaia swap items.\n3. Adaeze and Nalini swap items.\n4. Nalini and Kaia swap items.\n5. Greta and Adaeze swap items.\n6. Nalini and Kaia swap items.\n7. Adaeze and Nalini swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Greta\": \"ochre coin\", \"Adaeze\": \"cobalt mirror\", \"Nalini\": \"coral stone\", \"Kaia\": \"pearl shell\"}, \"people\": [\"Greta\", \"Adaeze\", \"Nalini\", \"Kaia\"]}"
 },
 {
  "task_id": "capacity_hard_017",
  "task_type": "capacity",
  "difficulty": "Hard",
  "prompt": "You are given 4 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Kaia holds a jade feather\n- Dmitri holds a golden lantern\n- Idris holds a copper locket\n- Maren holds a ochre ring\n\nSwaps:\n1. Dmitri and Idris swap items.\n2. Kaia and Maren swap items.\n3. Kaia and Idris swap items.\n4. Idris and Maren swap items.\n5. Kaia and Idris swap items.\n6. Idris and Maren swap items.\n7. Kaia and Maren swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Kaia\": \"golden lantern\", \"Dmitri\": \"copper locket\", \"Idris\": \"ochre ring\", \"Maren\": \"jade feather\"}, \"people\": [\"Kaia\", \"Dmitri\", \"Idris\", \"Maren\"]}"
 },
 {
  "task_id": "capacity_hard_018",
  "task_type": "capacity",
  "difficulty": "Hard",
  "prompt": "You are given 4 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Paloma holds a copper book\n- Wren holds a jade mirror\n- Lumi holds a green dagger\n- Joaquin holds a indigo compass\n\nSwaps:\n1. Paloma and Lumi swap items.\n2. Paloma and Wren swap items.\n3. Paloma and Lumi swap items.\n4. Wren and Lumi swap items.\n5. Lumi and Joaquin swap items.\n6. Paloma and Lumi swap items.\n7. Wren and Lumi swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Paloma\": \"indigo compass\", \"Wren\": \"copper book\", \"Lumi\": \"jade mirror\", \"Joaquin\": \"green dagger\"}, \"people\": [\"Paloma\", \"Wren\", \"Lumi\", \"Joaquin\"]}"
 },
 {
  "task_id": "capacity_hard_019",
  "task_type": "capacity",
  "difficulty": "Hard",
  "prompt": "You are given 4 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Idris holds a pearl locket\n- Celine holds a crimson mirror\n- Kenji holds a red shell\n- Lumi holds a copper coin\n\nSwaps:\n1. Idris and Kenji swap items.\n2. Celine and Lumi swap items.\n3. Celine and Kenji swap items.\n4. Celine and Lumi swap items.\n5. Idris and Celine swap items.\n6. Idris and Kenji swap items.\n7. Kenji and Lumi swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Idris\": \"copper coin\", \"Celine\": \"red shell\", \"Kenji\": \"pearl locket\", \"Lumi\": \"crimson mirror\"}, \"people\": [\"Idris\", \"Celine\", \"Kenji\", \"Lumi\"]}"
 },
 {
  "task_id": "capacity_hard_020",
  "task_type": "capacity",
  "difficulty": "Hard",
  "prompt": "You are given 4 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Dmitri holds a onyx flask\n- Gael holds a ochre shell\n- Sigrid holds a cobalt pendant\n- Tala holds a crimson chalice\n\nSwaps:\n1. Dmitri and Gael swap items.\n2. Dmitri and Tala swap items.\n3. Dmitri and Gael swap items.\n4. Gael and Tala swap items.\n5. Gael and Sigrid swap items.\n6. Dmitri and Tala swap items.\n7. Dmitri and Gael swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Dmitri\": \"cobalt pendant\", \"Gael\": \"crimson chalice\", \"Sigrid\": \"ochre shell\", \"Tala\": \"onyx flask\"}, \"people\": [\"Dmitri\", \"Gael\", \"Sigrid\", \"Tala\"]}"
 },
 {
  "task_id": "capacity_hard_021",
  "task_type": "capacity",
  "difficulty": "Hard",
  "prompt": "You are given 4 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Freya holds a ochre flask\n- Gael holds a crimson feather\n- Sigrid holds a russet lantern\n- Amara holds a silver chalice\n\nSwaps:\n1. Gael and Amara swap items.\n2. Gael and Sigrid swap items.\n3. Freya and Gael swap items.\n4. Gael and Amara swap items.\n5. Freya and Sigrid swap items.\n6. Freya and Amara swap items.\n7. Freya and Gael swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Freya\": \"crimson feather\", \"Gael\": \"ochre flask\", \"Sigrid\": \"russet lantern\", \"Amara\": \"silver chalice\"}, \"people\": [\"Freya\", \"Gael\", \"Sigrid\", \"Amara\"]}"
 },
 {
  "task_id": "capacity_hard_022",
  "task_type": "capacity",
  "difficulty": "Hard",
  "prompt": "You are given 4 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Nalini holds a cobalt lantern\n- Soren holds a amber locket\n- Adaeze holds a bronze ring\n- Leif holds a russet mask\n\nSwaps:\n1. Adaeze and Leif swap items.\n2. Nalini and Leif swap items.\n3. Adaeze and Leif swap items.\n4. Nalini and Adaeze swap items.\n5. Soren and Leif swap items.\n6. Soren and Adaeze swap items.\n7. Nalini and Soren swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Nalini\": \"bronze ring\", \"Soren\": \"cobalt lantern\", \"Adaeze\": \"russet mask\", \"Leif\": \"amber locket\"}, \"people\": [\"Nalini\", \"Soren\", \"Adaeze\", \"Leif\"]}"
 },
 {
  "task_id": "capacity_hard_023",
  "task_type": "capacity",
  "difficulty": "Hard",
  "prompt": "You are given 4 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Willa holds a teal mirror\n- Paloma holds a onyx pendant\n- Maren holds a green locket\n- Viktor holds a pearl shell\n\nSwaps:\n1. Willa and Paloma swap items.\n2. Willa and Maren swap items.\n3. Paloma and Maren swap items.\n4. Willa and Maren swap items.\n5. Maren and Viktor swap items.\n6. Paloma and Viktor swap items.\n7. Paloma and Maren swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Willa\": \"teal mirror\", \"Paloma\": \"pearl shell\", \"Maren\": \"green locket\", \"Viktor\": \"onyx pendant\"}, \"people\": [\"Willa\", \"Paloma\", \"Maren\", \"Viktor\"]}"
 },
 {
  "task_id": "capacity_expert_024",
  "task_type": "capacity",
  "difficulty": "Expert",
  "prompt": "You are given 5 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Lumi holds a ochre coin\n- Joaquin holds a red dice\n- Femi holds a ivory shell\n- Dmitri holds a silver flask\n- Willa holds a indigo lantern\n\nSwaps:\n1. Lumi and Joaquin swap items.\n2. Joaquin and Dmitri swap items.\n3. Femi and Dmitri swap items.\n4. Dmitri and Willa swap items.\n5. Joaquin and Willa swap items.\n6. Lumi and Dmitri swap items.\n7. Joaquin and Dmitri swap items.\n8. Femi and Willa swap items.\n9. Dmitri and Willa swap items.\n10. Lumi and Femi swap items.\n11. Lumi and Willa swap items.\n12. Femi and Dmitri swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Lumi\": \"ivory shell\", \"Joaquin\": \"red dice\", \"Femi\": \"ochre coin\", \"Dmitri\": \"indigo lantern\", \"Willa\": \"silver flask\"}, \"people\": [\"Lumi\", \"Joaquin\", \"Femi\", \"Dmitri\", \"Willa\"]}"
 },
 {
  "task_id": "capacity_expert_025",
  "task_type": "capacity",
  "difficulty": "Expert",
  "prompt": "You are given 5 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Colette holds a green stone\n- Paloma holds a ochre mask\n- Zora holds a crimson flask\n- Adaeze holds a cobalt scroll\n- Bram holds a russet bell\n\nSwaps:\n1. Paloma and Adaeze swap items.\n2. Colette and Zora swap items.\n3. Colette and Bram swap items.\n4. Zora and Adaeze swap items.\n5. Zora and Bram swap items.\n6. Adaeze and Bram swap items.\n7. Colette and Adaeze swap items.\n8. Paloma and Bram swap items.\n9. Colette and Adaeze swap items.\n10. Adaeze and Bram swap items.\n11. Zora and Adaeze swap items.\n12. Paloma and Adaeze swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Colette\": \"russet bell\", \"Paloma\": \"crimson flask\", \"Zora\": \"cobalt scroll\", \"Adaeze\": \"green stone\", \"Bram\": \"ochre mask\"}, \"people\": [\"Colette\", \"Paloma\", \"Zora\", \"Adaeze\", \"Bram\"]}"
 },
 {
  "task_id": "capacity_expert_026",
  "task_type": "capacity",
  "difficulty": "Expert",
  "prompt": "You are given 5 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Colette holds a copper chalice\n- Joelle holds a crimson key\n- Wren holds a bronze mirror\n- Freya holds a onyx shell\n- Bashir holds a indigo lantern\n\nSwaps:\n1. Joelle and Bashir swap items.\n2. Joelle and Freya swap items.\n3. Wren and Freya swap items.\n4. Joelle and Wren swap items.\n5. Joelle and Freya swap items.\n6. Colette and Wren swap items.\n7. Joelle and Freya swap items.\n8. Colette and Joelle swap items.\n9. Joelle and Wren swap items.\n10. Freya and Bashir swap items.\n11. Wren and Freya swap items.\n12. Colette and Freya swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Colette\": \"onyx shell\", \"Joelle\": \"copper chalice\", \"Wren\": \"crimson key\", \"Freya\": \"indigo lantern\", \"Bashir\": \"bronze mirror\"}, \"people\": [\"Colette\", \"Joelle\", \"Wren\", \"Freya\", \"Bashir\"]}"
 },
 {
  "task_id": "capacity_expert_027",
  "task_type": "capacity",
  "difficulty": "Expert",
  "prompt": "You are given 5 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Tala holds a green feather\n- Tariq holds a ivory book\n- Uma holds a russet dagger\n- Hana holds a red coin\n- Wren holds a slate dice\n\nSwaps:\n1. Tariq and Hana swap items.\n2. Uma and Hana swap items.\n3. Uma and Wren swap items.\n4. Tala and Hana swap items.\n5. Tala and Tariq swap items.\n6. Tariq and Uma swap items.\n7. Uma and Wren swap items.\n8. Uma and Hana swap items.\n9. Tariq and Uma swap items.\n10. Tala and Uma swap items.\n11. Tala and Wren swap items.\n12. Tariq and Uma swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Tala\": \"russet dagger\", \"Tariq\": \"red coin\", \"Uma\": \"green feather\", \"Hana\": \"ivory book\", \"Wren\": \"slate dice\"}, \"people\": [\"Tala\", \"Tariq\", \"Uma\", \"Hana\", \"Wren\"]}"
 },
 {
  "task_id": "capacity_expert_028",
  "task_type": "capacity",
  "difficulty": "Expert",
  "prompt": "You are given 5 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Leif holds a slate feather\n- Ines holds a teal pendant\n- Olena holds a amber ring\n- Vesna holds a russet mirror\n- Joelle holds a copper dagger\n\nSwaps:\n1. Leif and Olena swap items.\n2. Leif and Vesna swap items.\n3. Vesna and Joelle swap items.\n4. Ines and Joelle swap items.\n5. Leif and Ines swap items.\n6. Ines and Joelle swap items.\n7. Leif and Vesna swap items.\n8. Leif and Olena swap items.\n9. Leif and Ines swap items.\n10. Leif and Vesna swap items.\n11. Olena and Joelle swap items.\n12. Vesna and Joelle swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Leif\": \"amber ring\", \"Ines\": \"slate feather\", \"Olena\": \"russet mirror\", \"Vesna\": \"copper dagger\", \"Joelle\": \"teal pendant\"}, \"people\": [\"Leif\", \"Ines\", \"Olena\", \"Vesna\", \"Joelle\"]}"
 },
 {
  "task_id": "capacity_expert_029",
  "task_type": "capacity",
  "difficulty": "Expert",
  "prompt": "You are given 5 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Gael holds a silver dagger\n- Bashir holds a onyx stone\n- Ines holds a red bell\n- Willa holds a ochre mask\n- Zora holds a pearl scroll\n\nSwaps:\n1. Bashir and Ines swap items.\n2. Gael and Willa swap items.\n3. Bashir and Willa swap items.\n4. Bashir and Ines swap items.\n5. Gael and Bashir swap items.\n6. Willa and Zora swap items.\n7. Gael and Bashir swap items.\n8. Willa and Zora swap items.\n9. Gael and Willa swap items.\n10. Willa and Zora swap items.\n11. Gael and Ines swap items.\n12. Willa and Zora swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Gael\": \"silver dagger\", \"Bashir\": \"onyx stone\", \"Ines\": \"red bell\", \"Willa\": \"ochre mask\", \"Zora\": \"pearl scroll\"}, \"people\": [\"Gael\", \"Bashir\", \"Ines\", \"Willa\", \"Zora\"]}"
 },
 {
  "task_id": "capacity_expert_030",
  "task_type": "capacity",
  "difficulty": "Expert",
  "prompt": "You are given 5 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Freya holds a ivory dice\n- Runa holds a amber compass\n- Greta holds a blue locket\n- Paloma holds a silver stone\n- Elio holds a teal coin\n\nSwaps:\n1. Freya and Paloma swap items.\n2. Paloma and Elio swap items.\n3. Runa and Elio swap items.\n4. Greta and Paloma swap items.\n5. Freya and Paloma swap items.\n6. Freya and Greta swap items.\n7. Greta and Elio swap items.\n8. Freya and Elio swap items.\n9. Freya and Paloma swap items.\n10. Freya and Elio swap items.\n11. Freya and Greta swap items.\n12. Runa and Greta swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Freya\": \"amber compass\", \"Runa\": \"teal coin\", \"Greta\": \"ivory dice\", \"Paloma\": \"blue locket\", \"Elio\": \"silver stone\"}, \"people\": [\"Freya\", \"Runa\", \"Greta\", \"Paloma\", \"Elio\"]}"
 },
 {
  "task_id": "capacity_expert_031",
  "task_type": "capacity",
  "difficulty": "Expert",
  "prompt": "You are given 5 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Zain holds a pearl key\n- Uma holds a coral locket\n- Willa holds a red mirror\n- Hana holds a slate dice\n- Joelle holds a ochre bell\n\nSwaps:\n1. Uma and Joelle swap items.\n2. Zain and Uma swap items.\n3. Zain and Willa swap items.\n4. Hana and Joelle swap items.\n5. Zain and Willa swap items.\n6. Willa and Joelle swap items.\n7. Zain and Joelle swap items.\n8. Zain and Uma swap items.\n9. Zain and Joelle swap items.\n10. Willa and Joelle swap items.\n11. Uma and Willa swap items.\n12. Zain and Joelle swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Zain\": \"slate dice\", \"Uma\": \"pearl key\", \"Willa\": \"red mirror\", \"Hana\": \"coral locket\", \"Joelle\": \"ochre bell\"}, \"people\": [\"Zain\", \"Uma\", \"Willa\", \"Hana\", \"Joelle\"]}"
 },
 {
  "task_id": "capacity_frontier_032",
  "task_type": "capacity",
  "difficulty": "Frontier",
  "prompt": "You are given 8 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Magnus holds a golden key\n- Freya holds a teal coin\n- Femi holds a slate dice\n- Kaia holds a crimson stone\n- Nico holds a green bell\n- Kenji holds a indigo mirror\n- Sigrid holds a cobalt shell\n- Hana holds a russet lantern\n\nSwaps:\n1. Femi and Hana swap items.\n2. Magnus and Nico swap items.\n3. Magnus and Kenji swap items.\n4. Magnus and Kaia swap items.\n5. Magnus and Kenji swap items.\n6. Magnus and Femi swap items.\n7. Femi and Kenji swap items.\n8. Magnus and Nico swap items.\n9. Nico and Hana swap items.\n10. Magnus and Kaia swap items.\n11. Nico and Kenji swap items.\n12. Freya and Nico swap items.\n13. Magnus and Kenji swap items.\n14. Magnus and Sigrid swap items.\n15. Freya and Hana swap items.\n16. Freya and Kaia swap items.\n17. Nico and Hana swap items.\n18. Magnus and Kaia swap items.\n19. Magnus and Sigrid swap items.\n20. Freya and Femi swap items.\n21. Freya and Sigrid swap items.\n22. Nico and Hana swap items.\n23. Kaia and Nico swap items.\n24. Magnus and Sigrid swap items.\n25. Freya and Femi swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Magnus\": \"crimson stone\", \"Freya\": \"golden key\", \"Femi\": \"russet lantern\", \"Kaia\": \"teal coin\", \"Nico\": \"cobalt shell\", \"Kenji\": \"indigo mirror\", \"Sigrid\": \"slate dice\", \"Hana\": \"green bell\"}, \"people\": [\"Magnus\", \"Freya\", \"Femi\", \"Kaia\", \"Nico\", \"Kenji\", \"Sigrid\", \"Hana\"]}"
 },
 {
  "task_id": "capacity_frontier_033",
  "task_type": "capacity",
  "difficulty": "Frontier",
  "prompt": "You are given 8 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Soren holds a russet chalice\n- Zain holds a coral candle\n- Maren holds a blue locket\n- Nalini holds a pearl flask\n- Gael holds a golden shell\n- Zora holds a cobalt mask\n- Leif holds a copper mirror\n- Joaquin holds a crimson coin\n\nSwaps:\n1. Zain and Zora swap items.\n2. Soren and Leif swap items.\n3. Zain and Maren swap items.\n4. Zain and Gael swap items.\n5. Nalini and Leif swap items.\n6. Zain and Zora swap items.\n7. Soren and Leif swap items.\n8. Nalini and Zora swap items.\n9. Zain and Gael swap items.\n10. Zain and Joaquin swap items.\n11. Gael and Joaquin swap items.\n12. Soren and Nalini swap items.\n13. Soren and Leif swap items.\n14. Soren and Joaquin swap items.\n15. Gael and Joaquin swap items.\n16. Soren and Leif swap items.\n17. Zain and Joaquin swap items.\n18. Nalini and Joaquin swap items.\n19. Maren and Joaquin swap items.\n20. Zain and Zora swap items.\n21. Soren and Leif swap items.\n22. Zain and Nalini swap items.\n23. Nalini and Joaquin swap items.\n24. Zora and Joaquin swap items.\n25. Soren and Nalini swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Soren\": \"cobalt mask\", \"Zain\": \"crimson coin\", \"Maren\": \"pearl flask\", \"Nalini\": \"coral candle\", \"Gael\": \"copper mirror\", \"Zora\": \"russet chalice\", \"Leif\": \"golden shell\", \"Joaquin\": \"blue locket\"}, \"people\": [\"Soren\", \"Zain\", \"Maren\", \"Nalini\", \"Gael\", \"Zora\", \"Leif\", \"Joaquin\"]}"
 },
 {
  "task_id": "capacity_frontier_034",
  "task_type": "capacity",
  "difficulty": "Frontier",
  "prompt": "You are given 8 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Bram holds a ivory ring\n- Bashir holds a russet dice\n- Sigrid holds a green chalice\n- Joelle holds a cobalt locket\n- Dmitri holds a red bell\n- Ines holds a pearl stone\n- Priya holds a golden dagger\n- Yara holds a slate book\n\nSwaps:\n1. Sigrid and Dmitri swap items.\n2. Bashir and Yara swap items.\n3. Bashir and Priya swap items.\n4. Sigrid and Yara swap items.\n5. Sigrid and Dmitri swap items.\n6. Bram and Bashir swap items.\n7. Joelle and Yara swap items.\n8. Bram and Joelle swap items.\n9. Dmitri and Priya swap items.\n10. Bram and Sigrid swap items.\n11. Priya and Yara swap items.\n12. Bram and Bashir swap items.\n13. Bram and Priya swap items.\n14. Sigrid and Yara swap items.\n15. Dmitri and Ines swap items.\n16. Joelle and Dmitri swap items.\n17. Sigrid and Joelle swap items.\n18. Sigrid and Yara swap items.\n19. Sigrid and Priya swap items.\n20. Bram and Ines swap items.\n21. Priya and Yara swap items.\n22. Joelle and Dmitri swap items.\n23. Joelle and Ines swap items.\n24. Bram and Yara swap items.\n25. Bashir and Ines swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Bram\": \"red bell\", \"Bashir\": \"golden dagger\", \"Sigrid\": \"ivory ring\", \"Joelle\": \"cobalt locket\", \"Dmitri\": \"russet dice\", \"Ines\": \"green chalice\", \"Priya\": \"pearl stone\", \"Yara\": \"slate book\"}, \"people\": [\"Bram\", \"Bashir\", \"Sigrid\", \"Joelle\", \"Dmitri\", \"Ines\", \"Priya\", \"Yara\"]}"
 },
 {
  "task_id": "capacity_frontier_035",
  "task_type": "capacity",
  "difficulty": "Frontier",
  "prompt": "You are given 8 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Maren holds a green mask\n- Joaquin holds a teal candle\n- Runa holds a coral bell\n- Lumi holds a amber compass\n- Qadir holds a cobalt locket\n- Femi holds a bronze pendant\n- Adaeze holds a ochre shell\n- Ugo holds a russet stone\n\nSwaps:\n1. Lumi and Adaeze swap items.\n2. Joaquin and Ugo swap items.\n3. Runa and Femi swap items.\n4. Qadir and Adaeze swap items.\n5. Femi and Adaeze swap items.\n6. Lumi and Qadir swap items.\n7. Joaquin and Ugo swap items.\n8. Runa and Adaeze swap items.\n9. Maren and Ugo swap items.\n10. Joaquin and Femi swap items.\n11. Joaquin and Qadir swap items.\n12. Maren and Femi swap items.\n13. Lumi and Femi swap items.\n14. Runa and Femi swap items.\n15. Runa and Lumi swap items.\n16. Maren and Lumi swap items.\n17. Joaquin and Femi swap items.\n18. Qadir and Ugo swap items.\n19. Maren and Runa swap items.\n20. Adaeze and Ugo swap items.\n21. Joaquin and Femi swap items.\n22. Runa and Qadir swap items.\n23. Runa and Femi swap items.\n24. Maren and Adaeze swap items.\n25. Joaquin and Lumi swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Maren\": \"cobalt locket\", \"Joaquin\": \"teal candle\", \"Runa\": \"coral bell\", \"Lumi\": \"ochre shell\", \"Qadir\": \"amber compass\", \"Femi\": \"green mask\", \"Adaeze\": \"russet stone\", \"Ugo\": \"bronze pendant\"}, \"people\": [\"Maren\", \"Joaquin\", \"Runa\", \"Lumi\", \"Qadir\", \"Femi\", \"Adaeze\", \"Ugo\"]}"
 },
 {
  "task_id": "capacity_frontier_036",
  "task_type": "capacity",
  "difficulty": "Frontier",
  "prompt": "You are given 8 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Paloma holds a jade stone\n- Tala holds a russet book\n- Tariq holds a pearl dagger\n- Nico holds a indigo compass\n- Femi holds a silver ring\n- Yara holds a ochre shell\n- Celine holds a slate coin\n- Qadir holds a onyx key\n\nSwaps:\n1. Paloma and Nico swap items.\n2. Tala and Qadir swap items.\n3. Yara and Qadir swap items.\n4. Paloma and Qadir swap items.\n5. Nico and Celine swap items.\n6. Nico and Qadir swap items.\n7. Paloma and Tariq swap items.\n8. Paloma and Yara swap items.\n9. Paloma and Nico swap items.\n10. Femi and Yara swap items.\n11. Yara and Celine swap items.\n12. Tala and Tariq swap items.\n13. Tala and Femi swap items.\n14. Femi and Yara swap items.\n15. Tariq and Celine swap items.\n16. Paloma and Tala swap items.\n17. Tariq and Nico swap items.\n18. Nico and Qadir swap items.\n19. Nico and Yara swap items.\n20. Tariq and Yara swap items.\n21. Yara and Qadir swap items.\n22. Paloma and Yara swap items.\n23. Tala and Tariq swap items.\n24. Tariq and Nico swap items.\n25. Tala and Tariq swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Paloma\": \"silver ring\", \"Tala\": \"ochre shell\", \"Tariq\": \"slate coin\", \"Nico\": \"indigo compass\", \"Femi\": \"jade stone\", \"Yara\": \"pearl dagger\", \"Celine\": \"onyx key\", \"Qadir\": \"russet book\"}, \"people\": [\"Paloma\", \"Tala\", \"Tariq\", \"Nico\", \"Femi\", \"Yara\", \"Celine\", \"Qadir\"]}"
 },
 {
  "task_id": "capacity_frontier_037",
  "task_type": "capacity",
  "difficulty": "Frontier",
  "prompt": "You are given 8 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Adaeze holds a russet dagger\n- Ines holds a silver scroll\n- Ugo holds a crimson mirror\n- Qadir holds a green shell\n- Tala holds a ivory book\n- Nico holds a jade locket\n- Haruto holds a golden stone\n- Zora holds a slate ring\n\nSwaps:\n1. Ugo and Haruto swap items.\n2. Ines and Haruto swap items.\n3. Nico and Zora swap items.\n4. Adaeze and Ugo swap items.\n5. Ines and Nico swap items.\n6. Adaeze and Qadir swap items.\n7. Tala and Haruto swap items.\n8. Ines and Ugo swap items.\n9. Tala and Zora swap items.\n10. Qadir and Tala swap items.\n11. Haruto and Zora swap items.\n12. Tala and Zora swap items.\n13. Ines and Qadir swap items.\n14. Ines and Haruto swap items.\n15. Qadir and Nico swap items.\n16. Nico and Haruto swap items.\n17. Haruto and Zora swap items.\n18. Adaeze and Ines swap items.\n19. Ugo and Tala swap items.\n20. Ugo and Zora swap items.\n21. Haruto and Zora swap items.\n22. Adaeze and Tala swap items.\n23. Qadir and Nico swap items.\n24. Haruto and Zora swap items.\n25. Ines and Tala swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Adaeze\": \"slate ring\", \"Ines\": \"silver scroll\", \"Ugo\": \"russet dagger\", \"Qadir\": \"jade locket\", \"Tala\": \"green shell\", \"Nico\": \"crimson mirror\", \"Haruto\": \"golden stone\", \"Zora\": \"ivory book\"}, \"people\": [\"Adaeze\", \"Ines\", \"Ugo\", \"Qadir\", \"Tala\", \"Nico\", \"Haruto\", \"Zora\"]}"
 },
 {
  "task_id": "capacity_frontier_038",
  "task_type": "capacity",
  "difficulty": "Frontier",
  "prompt": "You are given 8 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Idris holds a red key\n- Sigrid holds a indigo candle\n- Soren holds a cobalt compass\n- Yara holds a green stone\n- Paloma holds a copper scroll\n- Yuki holds a amber flask\n- Runa holds a slate lantern\n- Magnus holds a pearl pendant\n\nSwaps:\n1. Paloma and Magnus swap items.\n2. Yara and Paloma swap items.\n3. Paloma and Runa swap items.\n4. Sigrid and Yara swap items.\n5. Idris and Runa swap items.\n6. Yuki and Magnus swap items.\n7. Runa and Magnus swap items.\n8. Soren and Yuki swap items.\n9. Sigrid and Magnus swap items.\n10. Idris and Soren swap items.\n11. Sigrid and Yuki swap items.\n12. Yuki and Magnus swap items.\n13. Soren and Yara swap items.\n14. Sigrid and Paloma swap items.\n15. Paloma and Magnus swap items.\n16. Sigrid and Yuki swap items.\n17. Idris and Sigrid swap items.\n18. Idris and Yuki swap items.\n19. Soren and Yuki swap items.\n20. Yara and Paloma swap items.\n21. Idris and Soren swap items.\n22. Yara and Yuki swap items.\n23. Sigrid and Paloma swap items.\n24. Runa and Magnus swap items.\n25. Idris and Paloma swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Idris\": \"copper scroll\", \"Sigrid\": \"green stone\", \"Soren\": \"slate lantern\", \"Yara\": \"indigo candle\", \"Paloma\": \"pearl pendant\", \"Yuki\": \"red key\", \"Runa\": \"cobalt compass\", \"Magnus\": \"amber flask\"}, \"people\": [\"Idris\", \"Sigrid\", \"Soren\", \"Yara\", \"Paloma\", \"Yuki\", \"Runa\", \"Magnus\"]}"
 },
 {
  "task_id": "capacity_frontier_039",
  "task_type": "capacity",
  "difficulty": "Frontier",
  "prompt": "You are given 8 people, each holding a unique item. After a series of swaps, report who holds each item.\n\nStarting positions:\n- Zain holds a amber mirror\n- Dmitri holds a blue feather\n- Nico holds a slate scroll\n- Colette holds a green book\n- Uma holds a ochre bell\n- Elio holds a bronze lantern\n- Maren holds a golden flask\n- Bram holds a coral shell\n\nSwaps:\n1. Nico and Uma swap items.\n2. Nico and Elio swap items.\n3. Colette and Uma swap items.\n4. Dmitri and Uma swap items.\n5. Zain and Uma swap items.\n6. Zain and Dmitri swap items.\n7. Nico and Colette swap items.\n8. Dmitri and Nico swap items.\n9. Uma and Bram swap items.\n10. Colette and Uma swap items.\n11. Zain and Bram swap items.\n12. Dmitri and Nico swap items.\n13. Nico and Elio swap items.\n14. Zain and Elio swap items.\n15. Dmitri and Bram swap items.\n16. Dmitri and Colette swap items.\n17. Dmitri and Nico swap items.\n18. Nico and Bram swap items.\n19. Uma and Elio swap items.\n20. Nico and Maren swap items.\n21. Colette and Maren swap items.\n22. Maren and Bram swap items.\n23. Zain and Bram swap items.\n24. Maren and Bram swap items.\n25. Nico and Colette swap items.\n\nAfter all swaps, list each person and their current item.\nFormat your answer EXACTLY as:\nANSWER:\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]\n- [Person]: [item]",
  "gold_json": "{\"answers\": {\"Zain\": \"green book\", \"Dmitri\": \"ochre bell\", \"Nico\": \"blue feather\", \"Colette\": \"golden flask\", \"Uma\": \"amber mirror\", \"Elio\": \"bronze lantern\", \"Maren\": \"slate scroll\", \"Bram\": \"coral shell\"}, \"people\": [\"Zain\", \"Dmitri\", \"Nico\", \"Colette\", \"Uma\", \"Elio\", \"Maren\", \"Bram\"]}"
 },
 {
  "task_id": "interference_easy_000",
  "task_type": "interference",
  "difficulty": "Easy",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: delegate is now set to 'Kotor'\n  Update: delegate is now set to 'Gdansk'\n  Update: delegate is now set to 'Plovdiv'\n\nWhat is the FINAL value of each record?\nANSWER:\n- delegate: [final value]",
  "gold_json": "{\"final_values\": {\"delegate\": \"Plovdiv\"}, \"key_names\": [\"delegate\"]}"
 },
 {
  "task_id": "interference_easy_001",
  "task_type": "interference",
  "difficulty": "Easy",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: registry is now set to '574'\n  Update: registry is now set to '943'\n  Update: registry is now set to '850'\n\nWhat is the FINAL value of each record?\nANSWER:\n- registry: [final value]",
  "gold_json": "{\"final_values\": {\"registry\": \"850\"}, \"key_names\": [\"registry\"]}"
 },
 {
  "task_id": "interference_easy_002",
  "task_type": "interference",
  "difficulty": "Easy",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: assignment is now set to 'Mandalay'\n  Update: assignment is now set to 'Plovdiv'\n  Update: assignment is now set to 'Jaipur'\n\nWhat is the FINAL value of each record?\nANSWER:\n- assignment: [final value]",
  "gold_json": "{\"final_values\": {\"assignment\": \"Jaipur\"}, \"key_names\": [\"assignment\"]}"
 },
 {
  "task_id": "interference_easy_003",
  "task_type": "interference",
  "difficulty": "Easy",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: coordinator is now set to 'Zain'\n  Update: coordinator is now set to 'Bashir'\n  Update: coordinator is now set to 'Willa'\n\nWhat is the FINAL value of each record?\nANSWER:\n- coordinator: [final value]",
  "gold_json": "{\"final_values\": {\"coordinator\": \"Willa\"}, \"key_names\": [\"coordinator\"]}"
 },
 {
  "task_id": "interference_easy_004",
  "task_type": "interference",
  "difficulty": "Easy",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: dispatch is now set to 'Zanzibar'\n  Update: dispatch is now set to 'Gdansk'\n  Update: dispatch is now set to 'Reykjavik'\n\nWhat is the FINAL value of each record?\nANSWER:\n- dispatch: [final value]",
  "gold_json": "{\"final_values\": {\"dispatch\": \"Reykjavik\"}, \"key_names\": [\"dispatch\"]}"
 },
 {
  "task_id": "interference_easy_005",
  "task_type": "interference",
  "difficulty": "Easy",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: delegate is now set to 'Tallinn'\n  Update: delegate is now set to 'Jaipur'\n  Update: delegate is now set to 'Oulu'\n\nWhat is the FINAL value of each record?\nANSWER:\n- delegate: [final value]",
  "gold_json": "{\"final_values\": {\"delegate\": \"Oulu\"}, \"key_names\": [\"delegate\"]}"
 },
 {
  "task_id": "interference_easy_006",
  "task_type": "interference",
  "difficulty": "Easy",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: assignment is now set to '627'\n  Update: assignment is now set to '694'\n  Update: assignment is now set to '455'\n\nWhat is the FINAL value of each record?\nANSWER:\n- assignment: [final value]",
  "gold_json": "{\"final_values\": {\"assignment\": \"455\"}, \"key_names\": [\"assignment\"]}"
 },
 {
  "task_id": "interference_easy_007",
  "task_type": "interference",
  "difficulty": "Easy",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: coordinator is now set to 'Reykjavik'\n  Update: coordinator is now set to 'Recife'\n  Update: coordinator is now set to 'Oulu'\n\nWhat is the FINAL value of each record?\nANSWER:\n- coordinator: [final value]",
  "gold_json": "{\"final_values\": {\"coordinator\": \"Oulu\"}, \"key_names\": [\"coordinator\"]}"
 },
 {
  "task_id": "interference_medium_008",
  "task_type": "interference",
  "difficulty": "Medium",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: coordinator is now set to 'Ulaanbaatar'\n  Update: delegate is now set to 'Jaipur'\n  Update: coordinator is now set to 'Jaipur'\n  Update: delegate is now set to 'Kotor'\n  Update: coordinator is now set to 'Fez'\n  Update: delegate is now set to 'Cusco'\n  Update: coordinator is now set to 'Tallinn'\n  Update: delegate is now set to 'Jaipur'\n  Update: coordinator is now set to 'Ulaanbaatar'\n  Update: delegate is now set to 'Tbilisi'\n  Update: coordinator is now set to 'Cartagena'\n  Update: delegate is now set to 'Tallinn'\n\nWhat is the FINAL value of each record?\nANSWER:\n- coordinator: [final value]\n- delegate: [final value]",
  "gold_json": "{\"final_values\": {\"coordinator\": \"Cartagena\", \"delegate\": \"Tallinn\"}, \"key_names\": [\"coordinator\", \"delegate\"]}"
 },
 {
  "task_id": "interference_medium_009",
  "task_type": "interference",
  "difficulty": "Medium",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: reference is now set to 'Valetta'\n  Update: assignment is now set to 'Cusco'\n  Update: reference is now set to 'Kumasi'\n  Update: assignment is now set to 'Kumasi'\n  Update: reference is now set to 'Fez'\n  Update: assignment is now set to 'Oulu'\n  Update: reference is now set to 'Cartagena'\n  Update: assignment is now set to 'Gdansk'\n  Update: reference is now set to 'Ulaanbaatar'\n  Update: assignment is now set to 'Fez'\n  Update: reference is now set to 'Mandalay'\n  Update: assignment is now set to 'Gdansk'\n\nWhat is the FINAL value of each record?\nANSWER:\n- reference: [final value]\n- assignment: [final value]",
  "gold_json": "{\"final_values\": {\"reference\": \"Mandalay\", \"assignment\": \"Gdansk\"}, \"key_names\": [\"reference\", \"assignment\"]}"
 },
 {
  "task_id": "interference_medium_010",
  "task_type": "interference",
  "difficulty": "Medium",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: registry is now set to '398'\n  Update: reference is now set to '662'\n  Update: registry is now set to '187'\n  Update: reference is now set to '985'\n  Update: registry is now set to '513'\n  Update: reference is now set to '142'\n  Update: registry is now set to '388'\n  Update: reference is now set to '849'\n  Update: registry is now set to '308'\n  Update: reference is now set to '158'\n  Update: registry is now set to '805'\n  Update: reference is now set to '297'\n\nWhat is the FINAL value of each record?\nANSWER:\n- registry: [final value]\n- reference: [final value]",
  "gold_json": "{\"final_values\": {\"registry\": \"805\", \"reference\": \"297\"}, \"key_names\": [\"registry\", \"reference\"]}"
 },
 {
  "task_id": "interference_medium_011",
  "task_type": "interference",
  "difficulty": "Medium",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: coordinator is now set to 'Maren'\n  Update: delegate is now set to 'Elara'\n  Update: coordinator is now set to 'Wren'\n  Update: delegate is now set to 'Lumi'\n  Update: coordinator is now set to 'Tariq'\n  Update: delegate is now set to 'Adaeze'\n  Update: coordinator is now set to 'Joelle'\n  Update: delegate is now set to 'Bashir'\n  Update: coordinator is now set to 'Ugo'\n  Update: delegate is now set to 'Colette'\n  Update: coordinator is now set to 'Dmitri'\n  Update: delegate is now set to 'Magnus'\n\nWhat is the FINAL value of each record?\nANSWER:\n- coordinator: [final value]\n- delegate: [final value]",
  "gold_json": "{\"final_values\": {\"coordinator\": \"Dmitri\", \"delegate\": \"Magnus\"}, \"key_names\": [\"coordinator\", \"delegate\"]}"
 },
 {
  "task_id": "interference_medium_012",
  "task_type": "interference",
  "difficulty": "Medium",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: reference is now set to '681'\n  Update: registry is now set to '630'\n  Update: reference is now set to '844'\n  Update: registry is now set to '326'\n  Update: reference is now set to '441'\n  Update: registry is now set to '611'\n  Update: reference is now set to '175'\n  Update: registry is now set to '103'\n  Update: reference is now set to '867'\n  Update: registry is now set to '175'\n  Update: reference is now set to '542'\n  Update: registry is now set to '238'\n\nWhat is the FINAL value of each record?\nANSWER:\n- reference: [final value]\n- registry: [final value]",
  "gold_json": "{\"final_values\": {\"reference\": \"542\", \"registry\": \"238\"}, \"key_names\": [\"reference\", \"registry\"]}"
 },
 {
  "task_id": "interference_medium_013",
  "task_type": "interference",
  "difficulty": "Medium",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: destination is now set to 'Colette'\n  Update: reference is now set to 'Freya'\n  Update: destination is now set to 'Freya'\n  Update: reference is now set to 'Bashir'\n  Update: destination is now set to 'Maren'\n  Update: reference is now set to 'Soren'\n  Update: destination is now set to 'Magnus'\n  Update: reference is now set to 'Ravi'\n  Update: destination is now set to 'Adaeze'\n  Update: reference is now set to 'Vesna'\n  Update: destination is now set to 'Gael'\n  Update: reference is now set to 'Idris'\n\nWhat is the FINAL value of each record?\nANSWER:\n- destination: [final value]\n- reference: [final value]",
  "gold_json": "{\"final_values\": {\"destination\": \"Gael\", \"reference\": \"Idris\"}, \"key_names\": [\"destination\", \"reference\"]}"
 },
 {
  "task_id": "interference_medium_014",
  "task_type": "interference",
  "difficulty": "Medium",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: coordinator is now set to '526'\n  Update: dispatch is now set to '209'\n  Update: coordinator is now set to '646'\n  Update: dispatch is now set to '159'\n  Update: coordinator is now set to '885'\n  Update: dispatch is now set to '985'\n  Update: coordinator is now set to '640'\n  Update: dispatch is now set to '507'\n  Update: coordinator is now set to '823'\n  Update: dispatch is now set to '366'\n  Update: coordinator is now set to '530'\n  Update: dispatch is now set to '553'\n\nWhat is the FINAL value of each record?\nANSWER:\n- coordinator: [final value]\n- dispatch: [final value]",
  "gold_json": "{\"final_values\": {\"coordinator\": \"530\", \"dispatch\": \"553\"}, \"key_names\": [\"coordinator\", \"dispatch\"]}"
 },
 {
  "task_id": "interference_medium_015",
  "task_type": "interference",
  "difficulty": "Medium",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: dispatch is now set to 'Vesna'\n  Update: delegate is now set to 'Haruto'\n  Update: dispatch is now set to 'Soren'\n  Update: delegate is now set to 'Tala'\n  Update: dispatch is now set to 'Kaia'\n  Update: delegate is now set to 'Priya'\n  Update: dispatch is now set to 'Nico'\n  Update: delegate is now set to 'Ines'\n  Update: dispatch is now set to 'Zora'\n  Update: delegate is now set to 'Soren'\n  Update: dispatch is now set to 'Tariq'\n  Update: delegate is now set to 'Elara'\n\nWhat is the FINAL value of each record?\nANSWER:\n- dispatch: [final value]\n- delegate: [final value]",
  "gold_json": "{\"final_values\": {\"dispatch\": \"Tariq\", \"delegate\": \"Elara\"}, \"key_names\": [\"dispatch\", \"delegate\"]}"
 },
 {
  "task_id": "interference_hard_016",
  "task_type": "interference",
  "difficulty": "Hard",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: dispatch is now set to 'Nalini'\n  Update: delegate is now set to 'Qadir'\n  Update: assignment is now set to 'Gael'\n  Update: dispatch is now set to 'Uma'\n  Update: delegate is now set to 'Tariq'\n  Update: assignment is now set to 'Ravi'\n  Update: dispatch is now set to 'Dariush'\n  Update: delegate is now set to 'Hana'\n  Update: assignment is now set to 'Elara'\n  Update: dispatch is now set to 'Yuki'\n  Update: delegate is now set to 'Olena'\n  Update: assignment is now set to 'Uma'\n  Update: dispatch is now set to 'Vesna'\n  Update: delegate is now set to 'Amara'\n  Update: assignment is now set to 'Sigrid'\n  Update: dispatch is now set to 'Lumi'\n  Update: delegate is now set to 'Ugo'\n  Update: assignment is now set to 'Viktor'\n  Update: dispatch is now set to 'Bashir'\n  Update: delegate is now set to 'Colette'\n  Update: assignment is now set to 'Orla'\n  Update: dispatch is now set to 'Uma'\n  Update: delegate is now set to 'Maren'\n  Update: assignment is now set to 'Freya'\n  Update: dispatch is now set to 'Tariq'\n  Update: delegate is now set to 'Adaeze'\n  Update: assignment is now set to 'Elara'\n  Update: dispatch is now set to 'Femi'\n  Update: delegate is now set to 'Freya'\n  Update: assignment is now set to 'Vesna'\n  Update: dispatch is now set to 'Sigrid'\n  Update: delegate is now set to 'Dmitri'\n  Update: assignment is now set to 'Ugo'\n  Update: dispatch is now set to 'Joelle'\n  Update: delegate is now set to 'Freya'\n  Update: assignment is now set to 'Nico'\n\nWhat is the FINAL value of each record?\nANSWER:\n- dispatch: [final value]\n- delegate: [final value]\n- assignment: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was 'Bashir' ever assigned to dispatch? [Yes/No]",
  "gold_json": "{\"final_values\": {\"dispatch\": \"Joelle\", \"delegate\": \"Freya\", \"assignment\": \"Nico\"}, \"key_names\": [\"dispatch\", \"delegate\", \"assignment\"]}"
 },
 {
  "task_id": "interference_hard_017",
  "task_type": "interference",
  "difficulty": "Hard",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: assignment is now set to 'Orla'\n  Update: dispatch is now set to 'Dariush'\n  Update: contact is now set to 'Vesna'\n  Update: assignment is now set to 'Wren'\n  Update: dispatch is now set to 'Colette'\n  Update: contact is now set to 'Kenji'\n  Update: assignment is now set to 'Vesna'\n  Update: dispatch is now set to 'Nico'\n  Update: contact is now set to 'Haruto'\n  Update: assignment is now set to 'Viktor'\n  Update: dispatch is now set to 'Wren'\n  Update: contact is now set to 'Nalini'\n  Update: assignment is now set to 'Leif'\n  Update: dispatch is now set to 'Yuki'\n  Update: contact is now set to 'Tala'\n  Update: assignment is now set to 'Sigrid'\n  Update: dispatch is now set to 'Joaquin'\n  Update: contact is now set to 'Celine'\n  Update: assignment is now set to 'Zora'\n  Update: dispatch is now set to 'Kenji'\n  Update: contact is now set to 'Haruto'\n  Update: assignment is now set to 'Femi'\n  Update: dispatch is now set to 'Runa'\n  Update: contact is now set to 'Xander'\n  Update: assignment is now set to 'Yuki'\n  Update: dispatch is now set to 'Priya'\n  Update: contact is now set to 'Priya'\n  Update: assignment is now set to 'Joelle'\n  Update: dispatch is now set to 'Elara'\n  Update: contact is now set to 'Vesna'\n  Update: assignment is now set to 'Dmitri'\n  Update: dispatch is now set to 'Leif'\n  Update: contact is now set to 'Dariush'\n  Update: assignment is now set to 'Lumi'\n  Update: dispatch is now set to 'Viktor'\n  Update: contact is now set to 'Uma'\n\nWhat is the FINAL value of each record?\nANSWER:\n- assignment: [final value]\n- dispatch: [final value]\n- contact: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was 'Wren' ever assigned to assignment? [Yes/No]",
  "gold_json": "{\"final_values\": {\"assignment\": \"Lumi\", \"dispatch\": \"Viktor\", \"contact\": \"Uma\"}, \"key_names\": [\"assignment\", \"dispatch\", \"contact\"]}"
 },
 {
  "task_id": "interference_hard_018",
  "task_type": "interference",
  "difficulty": "Hard",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: registry is now set to 'Kumasi'\n  Update: contact is now set to 'Luang Prabang'\n  Update: destination is now set to 'Fez'\n  Update: registry is now set to 'Zanzibar'\n  Update: contact is now set to 'Plovdiv'\n  Update: destination is now set to 'Valetta'\n  Update: registry is now set to 'Luang Prabang'\n  Update: contact is now set to 'Kumasi'\n  Update: destination is now set to 'Oulu'\n  Update: registry is now set to 'Trieste'\n  Update: contact is now set to 'Ulaanbaatar'\n  Update: destination is now set to 'Tallinn'\n  Update: registry is now set to 'Ulaanbaatar'\n  Update: contact is now set to 'Jaipur'\n  Update: destination is now set to 'Kotor'\n  Update: registry is now set to 'Tbilisi'\n  Update: contact is now set to 'Oulu'\n  Update: destination is now set to 'Jaipur'\n  Update: registry is now set to 'Zanzibar'\n  Update: contact is now set to 'Luang Prabang'\n  Update: destination is now set to 'Tallinn'\n  Update: registry is now set to 'Bruges'\n  Update: contact is now set to 'Reykjavik'\n  Update: destination is now set to 'Zanzibar'\n  Update: registry is now set to 'Zanzibar'\n  Update: contact is now set to 'Jaipur'\n  Update: destination is now set to 'Bruges'\n  Update: registry is now set to 'Cartagena'\n  Update: contact is now set to 'Valetta'\n  Update: destination is now set to 'Kumasi'\n  Update: registry is now set to 'Recife'\n  Update: contact is now set to 'Gdansk'\n  Update: destination is now set to 'Tbilisi'\n  Update: registry is now set to 'Kotor'\n  Update: contact is now set to 'Cartagena'\n  Update: destination is now set to 'Reykjavik'\n\nWhat is the FINAL value of each record?\nANSWER:\n- registry: [final value]\n- contact: [final value]\n- destination: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was 'Zanzibar' ever assigned to registry? [Yes/No]",
  "gold_json": "{\"final_values\": {\"registry\": \"Kotor\", \"contact\": \"Cartagena\", \"destination\": \"Reykjavik\"}, \"key_names\": [\"registry\", \"contact\", \"destination\"]}"
 },
 {
  "task_id": "interference_hard_019",
  "task_type": "interference",
  "difficulty": "Hard",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: location is now set to '247'\n  Update: assignment is now set to '283'\n  Update: registry is now set to '856'\n  Update: location is now set to '958'\n  Update: assignment is now set to '368'\n  Update: registry is now set to '170'\n  Update: location is now set to '908'\n  Update: assignment is now set to '475'\n  Update: registry is now set to '755'\n  Update: location is now set to '803'\n  Update: assignment is now set to '945'\n  Update: registry is now set to '881'\n  Update: location is now set to '131'\n  Update: assignment is now set to '912'\n  Update: registry is now set to '764'\n  Update: location is now set to '461'\n  Update: assignment is now set to '993'\n  Update: registry is now set to '409'\n  Update: location is now set to '170'\n  Update: assignment is now set to '737'\n  Update: registry is now set to '970'\n  Update: location is now set to '147'\n  Update: assignment is now set to '478'\n  Update: registry is now set to '422'\n  Update: location is now set to '762'\n  Update: assignment is now set to '556'\n  Update: registry is now set to '232'\n  Update: location is now set to '484'\n  Update: assignment is now set to '435'\n  Update: registry is now set to '339'\n  Update: location is now set to '206'\n  Update: assignment is now set to '278'\n  Update: registry is now set to '321'\n  Update: location is now set to '354'\n  Update: assignment is now set to '897'\n  Update: registry is now set to '464'\n\nWhat is the FINAL value of each record?\nANSWER:\n- location: [final value]\n- assignment: [final value]\n- registry: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was '484' ever assigned to location? [Yes/No]",
  "gold_json": "{\"final_values\": {\"location\": \"354\", \"assignment\": \"897\", \"registry\": \"464\"}, \"key_names\": [\"location\", \"assignment\", \"registry\"]}"
 },
 {
  "task_id": "interference_hard_020",
  "task_type": "interference",
  "difficulty": "Hard",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: contact is now set to 'Ines'\n  Update: location is now set to 'Vesna'\n  Update: coordinator is now set to 'Orla'\n  Update: contact is now set to 'Dmitri'\n  Update: location is now set to 'Kaia'\n  Update: coordinator is now set to 'Uma'\n  Update: contact is now set to 'Zora'\n  Update: location is now set to 'Tariq'\n  Update: coordinator is now set to 'Dmitri'\n  Update: contact is now set to 'Soren'\n  Update: location is now set to 'Lumi'\n  Update: coordinator is now set to 'Dariush'\n  Update: contact is now set to 'Bram'\n  Update: location is now set to 'Idris'\n  Update: coordinator is now set to 'Greta'\n  Update: contact is now set to 'Freya'\n  Update: location is now set to 'Ines'\n  Update: coordinator is now set to 'Dmitri'\n  Update: contact is now set to 'Dariush'\n  Update: location is now set to 'Elara'\n  Update: coordinator is now set to 'Kenji'\n  Update: contact is now set to 'Bram'\n  Update: location is now set to 'Bashir'\n  Update: coordinator is now set to 'Xander'\n  Update: contact is now set to 'Zain'\n  Update: location is now set to 'Dmitri'\n  Update: coordinator is now set to 'Idris'\n  Update: contact is now set to 'Xander'\n  Update: location is now set to 'Adaeze'\n  Update: coordinator is now set to 'Bashir'\n  Update: contact is now set to 'Viktor'\n  Update: location is now set to 'Orla'\n  Update: coordinator is now set to 'Celine'\n  Update: contact is now set to 'Zain'\n  Update: location is now set to 'Freya'\n  Update: coordinator is now set to 'Nico'\n\nWhat is the FINAL value of each record?\nANSWER:\n- contact: [final value]\n- location: [final value]\n- coordinator: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was 'Dmitri' ever assigned to contact? [Yes/No]",
  "gold_json": "{\"final_values\": {\"contact\": \"Zain\", \"location\": \"Freya\", \"coordinator\": \"Nico\"}, \"key_names\": [\"contact\", \"location\", \"coordinator\"]}"
 },
 {
  "task_id": "interference_hard_021",
  "task_type": "interference",
  "difficulty": "Hard",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: liaison is now set to 'Kotor'\n  Update: reference is now set to 'Tallinn'\n  Update: location is now set to 'Valetta'\n  Update: liaison is now set to 'Reykjavik'\n  Update: reference is now set to 'Reykjavik'\n  Update: location is now set to 'Tallinn'\n  Update: liaison is now set to 'Kumasi'\n  Update: reference is now set to 'Luang Prabang'\n  Update: location is now set to 'Luang Prabang'\n  Update: liaison is now set to 'Ulaanbaatar'\n  Update: reference is now set to 'Valetta'\n  Update: location is now set to 'Tbilisi'\n  Update: liaison is now set to 'Kumasi'\n  Update: reference is now set to 'Cartagena'\n  Update: location is now set to 'Zanzibar'\n  Update: liaison is now set to 'Cusco'\n  Update: reference is now set to 'Gdansk'\n  Update: location is now set to 'Mandalay'\n  Update: liaison is now set to 'Plovdiv'\n  Update: reference is now set to 'Fez'\n  Update: location is now set to 'Trieste'\n  Update: liaison is now set to 'Jaipur'\n  Update: reference is now set to 'Zanzibar'\n  Update: location is now set to 'Valetta'\n  Update: liaison is now set to 'Tbilisi'\n  Update: reference is now set to 'Recife'\n  Update: location is now set to 'Jaipur'\n  Update: liaison is now set to 'Recife'\n  Update: reference is now set to 'Jaipur'\n  Update: location is now set to 'Valetta'\n  Update: liaison is now set to 'Zanzibar'\n  Update: reference is now set to 'Kumasi'\n  Update: location is now set to 'Luang Prabang'\n  Update: liaison is now set to 'Mandalay'\n  Update: reference is now set to 'Kotor'\n  Update: location is now set to 'Plovdiv'\n\nWhat is the FINAL value of each record?\nANSWER:\n- liaison: [final value]\n- reference: [final value]\n- location: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was 'Cusco' ever assigned to liaison? [Yes/No]",
  "gold_json": "{\"final_values\": {\"liaison\": \"Mandalay\", \"reference\": \"Kotor\", \"location\": \"Plovdiv\"}, \"key_names\": [\"liaison\", \"reference\", \"location\"]}"
 },
 {
  "task_id": "interference_hard_022",
  "task_type": "interference",
  "difficulty": "Hard",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: delegate is now set to 'Valetta'\n  Update: dispatch is now set to 'Mandalay'\n  Update: assignment is now set to 'Reykjavik'\n  Update: delegate is now set to 'Cartagena'\n  Update: dispatch is now set to 'Plovdiv'\n  Update: assignment is now set to 'Fez'\n  Update: delegate is now set to 'Tbilisi'\n  Update: dispatch is now set to 'Cusco'\n  Update: assignment is now set to 'Gdansk'\n  Update: delegate is now set to 'Luang Prabang'\n  Update: dispatch is now set to 'Recife'\n  Update: assignment is now set to 'Bruges'\n  Update: delegate is now set to 'Valetta'\n  Update: dispatch is now set to 'Kumasi'\n  Update: assignment is now set to 'Cartagena'\n  Update: delegate is now set to 'Luang Prabang'\n  Update: dispatch is now set to 'Recife'\n  Update: assignment is now set to 'Cusco'\n  Update: delegate is now set to 'Tbilisi'\n  Update: dispatch is now set to 'Cartagena'\n  Update: assignment is now set to 'Valetta'\n  Update: delegate is now set to 'Tallinn'\n  Update: dispatch is now set to 'Trieste'\n  Update: assignment is now set to 'Gdansk'\n  Update: delegate is now set to 'Luang Prabang'\n  Update: dispatch is now set to 'Ulaanbaatar'\n  Update: assignment is now set to 'Oulu'\n  Update: delegate is now set to 'Trieste'\n  Update: dispatch is now set to 'Recife'\n  Update: assignment is now set to 'Tbilisi'\n  Update: delegate is now set to 'Recife'\n  Update: dispatch is now set to 'Oulu'\n  Update: assignment is now set to 'Plovdiv'\n  Update: delegate is now set to 'Plovdiv'\n  Update: dispatch is now set to 'Mandalay'\n  Update: assignment is now set to 'Bruges'\n\nWhat is the FINAL value of each record?\nANSWER:\n- delegate: [final value]\n- dispatch: [final value]\n- assignment: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was 'Tbilisi' ever assigned to delegate? [Yes/No]",
  "gold_json": "{\"final_values\": {\"delegate\": \"Plovdiv\", \"dispatch\": \"Mandalay\", \"assignment\": \"Bruges\"}, \"key_names\": [\"delegate\", \"dispatch\", \"assignment\"]}"
 },
 {
  "task_id": "interference_hard_023",
  "task_type": "interference",
  "difficulty": "Hard",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: delegate is now set to 'Sigrid'\n  Update: dispatch is now set to 'Runa'\n  Update: contact is now set to 'Elio'\n  Update: delegate is now set to 'Uma'\n  Update: dispatch is now set to 'Greta'\n  Update: contact is now set to 'Yara'\n  Update: delegate is now set to 'Runa'\n  Update: dispatch is now set to 'Maren'\n  Update: contact is now set to 'Yuki'\n  Update: delegate is now set to 'Soren'\n  Update: dispatch is now set to 'Hana'\n  Update: contact is now set to 'Nalini'\n  Update: delegate is now set to 'Paloma'\n  Update: dispatch is now set to 'Qadir'\n  Update: contact is now set to 'Olena'\n  Update: delegate is now set to 'Elara'\n  Update: dispatch is now set to 'Olena'\n  Update: contact is now set to 'Viktor'\n  Update: delegate is now set to 'Bashir'\n  Update: dispatch is now set to 'Paloma'\n  Update: contact is now set to 'Orla'\n  Update: delegate is now set to 'Dariush'\n  Update: dispatch is now set to 'Sigrid'\n  Update: contact is now set to 'Femi'\n  Update: delegate is now set to 'Tariq'\n  Update: dispatch is now set to 'Priya'\n  Update: contact is now set to 'Joaquin'\n  Update: delegate is now set to 'Colette'\n  Update: dispatch is now set to 'Sigrid'\n  Update: contact is now set to 'Celine'\n  Update: delegate is now set to 'Maren'\n  Update: dispatch is now set to 'Olena'\n  Update: contact is now set to 'Gael'\n  Update: delegate is now set to 'Tala'\n  Update: dispatch is now set to 'Amara'\n  Update: contact is now set to 'Amara'\n\nWhat is the FINAL value of each record?\nANSWER:\n- delegate: [final value]\n- dispatch: [final value]\n- contact: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was 'Paloma' ever assigned to delegate? [Yes/No]",
  "gold_json": "{\"final_values\": {\"delegate\": \"Tala\", \"dispatch\": \"Amara\", \"contact\": \"Amara\"}, \"key_names\": [\"delegate\", \"dispatch\", \"contact\"]}"
 },
 {
  "task_id": "interference_expert_024",
  "task_type": "interference",
  "difficulty": "Expert",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: assignment is now set to '901'\n  Update: liaison is now set to '728'\n  Update: location is now set to '422'\n  Update: registry is now set to '440'\n  Update: assignment is now set to '961'\n  Update: liaison is now set to '927'\n  Update: location is now set to '758'\n  Update: registry is now set to '518'\n  Update: assignment is now set to '885'\n  Update: liaison is now set to '787'\n  Update: location is now set to '566'\n  Update: registry is now set to '111'\n  Update: assignment is now set to '931'\n  Update: liaison is now set to '832'\n  Update: location is now set to '863'\n  Update: registry is now set to '850'\n  Update: assignment is now set to '775'\n  Update: liaison is now set to '178'\n  Update: location is now set to '576'\n  Update: registry is now set to '264'\n  Update: assignment is now set to '621'\n  Update: liaison is now set to '872'\n  Update: location is now set to '662'\n  Update: registry is now set to '282'\n  Update: assignment is now set to '423'\n  Update: liaison is now set to '924'\n  Update: location is now set to '427'\n  Update: registry is now set to '419'\n  Update: assignment is now set to '138'\n  Update: liaison is now set to '560'\n  Update: location is now set to '393'\n  Update: registry is now set to '252'\n  Update: assignment is now set to '352'\n  Update: liaison is now set to '774'\n  Update: location is now set to '130'\n  Update: registry is now set to '855'\n  Update: assignment is now set to '741'\n  Update: liaison is now set to '470'\n  Update: location is now set to '505'\n  Update: registry is now set to '761'\n  Update: assignment is now set to '631'\n  Update: liaison is now set to '523'\n  Update: location is now set to '647'\n  Update: registry is now set to '594'\n  Update: assignment is now set to '518'\n  Update: liaison is now set to '875'\n  Update: location is now set to '169'\n  Update: registry is now set to '907'\n  Update: assignment is now set to '491'\n  Update: liaison is now set to '598'\n  Update: location is now set to '189'\n  Update: registry is now set to '508'\n  Update: assignment is now set to '450'\n  Update: liaison is now set to '121'\n  Update: location is now set to '708'\n  Update: registry is now set to '678'\n  Update: assignment is now set to '877'\n  Update: liaison is now set to '823'\n  Update: location is now set to '192'\n  Update: registry is now set to '831'\n  Update: assignment is now set to '833'\n  Update: liaison is now set to '533'\n  Update: location is now set to '330'\n  Update: registry is now set to '344'\n  Update: assignment is now set to '335'\n  Update: liaison is now set to '279'\n  Update: location is now set to '552'\n  Update: registry is now set to '614'\n  Update: assignment is now set to '268'\n  Update: liaison is now set to '209'\n  Update: location is now set to '627'\n  Update: registry is now set to '522'\n  Update: assignment is now set to '559'\n  Update: liaison is now set to '399'\n  Update: location is now set to '355'\n  Update: registry is now set to '742'\n  Update: assignment is now set to '608'\n  Update: liaison is now set to '291'\n  Update: location is now set to '633'\n  Update: registry is now set to '697'\n  Update: assignment is now set to '884'\n  Update: liaison is now set to '765'\n  Update: location is now set to '482'\n  Update: registry is now set to '871'\n  Update: assignment is now set to '448'\n  Update: liaison is now set to '812'\n  Update: location is now set to '496'\n  Update: registry is now set to '781'\n  Update: assignment is now set to '791'\n  Update: liaison is now set to '713'\n  Update: location is now set to '274'\n  Update: registry is now set to '767'\n  Update: assignment is now set to '894'\n  Update: liaison is now set to '132'\n  Update: location is now set to '491'\n  Update: registry is now set to '188'\n  Update: assignment is now set to '209'\n  Update: liaison is now set to '765'\n  Update: location is now set to '267'\n  Update: registry is now set to '951'\n\nWhat is the FINAL value of each record?\nANSWER:\n- assignment: [final value]\n- liaison: [final value]\n- location: [final value]\n- registry: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was '518' ever assigned to assignment? [Yes/No]\nV2. Was '872' ever assigned to liaison? [Yes/No]",
  "gold_json": "{\"final_values\": {\"assignment\": \"209\", \"liaison\": \"765\", \"location\": \"267\", \"registry\": \"951\"}, \"key_names\": [\"assignment\", \"liaison\", \"location\", \"registry\"]}"
 },
 {
  "task_id": "interference_expert_025",
  "task_type": "interference",
  "difficulty": "Expert",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: registry is now set to 'Olena'\n  Update: liaison is now set to 'Runa'\n  Update: dispatch is now set to 'Idris'\n  Update: assignment is now set to 'Lumi'\n  Update: registry is now set to 'Freya'\n  Update: liaison is now set to 'Olena'\n  Update: dispatch is now set to 'Nico'\n  Update: assignment is now set to 'Vesna'\n  Update: registry is now set to 'Zora'\n  Update: liaison is now set to 'Elara'\n  Update: dispatch is now set to 'Femi'\n  Update: assignment is now set to 'Ravi'\n  Update: registry is now set to 'Haruto'\n  Update: liaison is now set to 'Ravi'\n  Update: dispatch is now set to 'Adaeze'\n  Update: assignment is now set to 'Adaeze'\n  Update: registry is now set to 'Zain'\n  Update: liaison is now set to 'Adaeze'\n  Update: dispatch is now set to 'Wren'\n  Update: assignment is now set to 'Zora'\n  Update: registry is now set to 'Maren'\n  Update: liaison is now set to 'Joaquin'\n  Update: dispatch is now set to 'Celine'\n  Update: assignment is now set to 'Joelle'\n  Update: registry is now set to 'Gael'\n  Update: liaison is now set to 'Colette'\n  Update: dispatch is now set to 'Hana'\n  Update: assignment is now set to 'Qadir'\n  Update: registry is now set to 'Yuki'\n  Update: liaison is now set to 'Leif'\n  Update: dispatch is now set to 'Yuki'\n  Update: assignment is now set to 'Wren'\n  Update: registry is now set to 'Joaquin'\n  Update: liaison is now set to 'Elio'\n  Update: dispatch is now set to 'Yara'\n  Update: assignment is now set to 'Colette'\n  Update: registry is now set to 'Joelle'\n  Update: liaison is now set to 'Zain'\n  Update: dispatch is now set to 'Bram'\n  Update: assignment is now set to 'Femi'\n  Update: registry is now set to 'Joaquin'\n  Update: liaison is now set to 'Dariush'\n  Update: dispatch is now set to 'Leif'\n  Update: assignment is now set to 'Yuki'\n  Update: registry is now set to 'Magnus'\n  Update: liaison is now set to 'Viktor'\n  Update: dispatch is now set to 'Greta'\n  Update: assignment is now set to 'Hana'\n  Update: registry is now set to 'Celine'\n  Update: liaison is now set to 'Bashir'\n  Update: dispatch is now set to 'Magnus'\n  Update: assignment is now set to 'Ravi'\n  Update: registry is now set to 'Idris'\n  Update: liaison is now set to 'Femi'\n  Update: dispatch is now set to 'Runa'\n  Update: assignment is now set to 'Uma'\n  Update: registry is now set to 'Qadir'\n  Update: liaison is now set to 'Xander'\n  Update: dispatch is now set to 'Zora'\n  Update: assignment is now set to 'Elio'\n  Update: registry is now set to 'Leif'\n  Update: liaison is now set to 'Zain'\n  Update: dispatch is now set to 'Femi'\n  Update: assignment is now set to 'Tariq'\n  Update: registry is now set to 'Adaeze'\n  Update: liaison is now set to 'Gael'\n  Update: dispatch is now set to 'Nalini'\n  Update: assignment is now set to 'Uma'\n  Update: registry is now set to 'Hana'\n  Update: liaison is now set to 'Lumi'\n  Update: dispatch is now set to 'Haruto'\n  Update: assignment is now set to 'Kenji'\n  Update: registry is now set to 'Xander'\n  Update: liaison is now set to 'Tariq'\n  Update: dispatch is now set to 'Kenji'\n  Update: assignment is now set to 'Joaquin'\n  Update: registry is now set to 'Ravi'\n  Update: liaison is now set to 'Xander'\n  Update: dispatch is now set to 'Priya'\n  Update: assignment is now set to 'Kaia'\n  Update: registry is now set to 'Tala'\n  Update: liaison is now set to 'Elio'\n  Update: dispatch is now set to 'Bashir'\n  Update: assignment is now set to 'Zain'\n  Update: registry is now set to 'Gael'\n  Update: liaison is now set to 'Tala'\n  Update: dispatch is now set to 'Tala'\n  Update: assignment is now set to 'Tala'\n  Update: registry is now set to 'Freya'\n  Update: liaison is now set to 'Paloma'\n  Update: dispatch is now set to 'Yuki'\n  Update: assignment is now set to 'Joelle'\n  Update: registry is now set to 'Haruto'\n  Update: liaison is now set to 'Freya'\n  Update: dispatch is now set to 'Dariush'\n  Update: assignment is now set to 'Celine'\n  Update: registry is now set to 'Ines'\n  Update: liaison is now set to 'Viktor'\n  Update: dispatch is now set to 'Dmitri'\n  Update: assignment is now set to 'Dmitri'\n\nWhat is the FINAL value of each record?\nANSWER:\n- registry: [final value]\n- liaison: [final value]\n- dispatch: [final value]\n- assignment: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was 'Freya' ever assigned to registry? [Yes/No]\nV2. Was 'Leif' ever assigned to liaison? [Yes/No]",
  "gold_json": "{\"final_values\": {\"registry\": \"Ines\", \"liaison\": \"Viktor\", \"dispatch\": \"Dmitri\", \"assignment\": \"Dmitri\"}, \"key_names\": [\"registry\", \"liaison\", \"dispatch\", \"assignment\"]}"
 },
 {
  "task_id": "interference_expert_026",
  "task_type": "interference",
  "difficulty": "Expert",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: reference is now set to 'Soren'\n  Update: registry is now set to 'Olena'\n  Update: contact is now set to 'Bashir'\n  Update: location is now set to 'Hana'\n  Update: reference is now set to 'Amara'\n  Update: registry is now set to 'Yara'\n  Update: contact is now set to 'Tariq'\n  Update: location is now set to 'Priya'\n  Update: reference is now set to 'Yara'\n  Update: registry is now set to 'Zain'\n  Update: contact is now set to 'Ines'\n  Update: location is now set to 'Sigrid'\n  Update: reference is now set to 'Wren'\n  Update: registry is now set to 'Orla'\n  Update: contact is now set to 'Hana'\n  Update: location is now set to 'Olena'\n  Update: reference is now set to 'Greta'\n  Update: registry is now set to 'Greta'\n  Update: contact is now set to 'Elara'\n  Update: location is now set to 'Elara'\n  Update: reference is now set to 'Kaia'\n  Update: registry is now set to 'Colette'\n  Update: contact is now set to 'Freya'\n  Update: location is now set to 'Colette'\n  Update: reference is now set to 'Colette'\n  Update: registry is now set to 'Kaia'\n  Update: contact is now set to 'Colette'\n  Update: location is now set to 'Amara'\n  Update: reference is now set to 'Vesna'\n  Update: registry is now set to 'Magnus'\n  Update: contact is now set to 'Vesna'\n  Update: location is now set to 'Elio'\n  Update: reference is now set to 'Femi'\n  Update: registry is now set to 'Gael'\n  Update: contact is now set to 'Runa'\n  Update: location is now set to 'Lumi'\n  Update: reference is now set to 'Nalini'\n  Update: registry is now set to 'Dmitri'\n  Update: contact is now set to 'Bram'\n  Update: location is now set to 'Paloma'\n  Update: reference is now set to 'Joaquin'\n  Update: registry is now set to 'Vesna'\n  Update: contact is now set to 'Wren'\n  Update: location is now set to 'Olena'\n  Update: reference is now set to 'Uma'\n  Update: registry is now set to 'Gael'\n  Update: contact is now set to 'Runa'\n  Update: location is now set to 'Elio'\n  Update: reference is now set to 'Idris'\n  Update: registry is now set to 'Soren'\n  Update: contact is now set to 'Gael'\n  Update: location is now set to 'Lumi'\n  Update: reference is now set to 'Dariush'\n  Update: registry is now set to 'Vesna'\n  Update: contact is now set to 'Qadir'\n  Update: location is now set to 'Dariush'\n  Update: reference is now set to 'Amara'\n  Update: registry is now set to 'Lumi'\n  Update: contact is now set to 'Joelle'\n  Update: location is now set to 'Bashir'\n  Update: reference is now set to 'Dariush'\n  Update: registry is now set to 'Viktor'\n  Update: contact is now set to 'Uma'\n  Update: location is now set to 'Soren'\n  Update: reference is now set to 'Bram'\n  Update: registry is now set to 'Zain'\n  Update: contact is now set to 'Runa'\n  Update: location is now set to 'Lumi'\n  Update: reference is now set to 'Zain'\n  Update: registry is now set to 'Orla'\n  Update: contact is now set to 'Adaeze'\n  Update: location is now set to 'Ines'\n  Update: reference is now set to 'Nico'\n  Update: registry is now set to 'Paloma'\n  Update: contact is now set to 'Paloma'\n  Update: location is now set to 'Wren'\n  Update: reference is now set to 'Adaeze'\n  Update: registry is now set to 'Bashir'\n  Update: contact is now set to 'Nalini'\n  Update: location is now set to 'Olena'\n  Update: reference is now set to 'Ugo'\n  Update: registry is now set to 'Colette'\n  Update: contact is now set to 'Amara'\n  Update: location is now set to 'Kaia'\n  Update: reference is now set to 'Leif'\n  Update: registry is now set to 'Hana'\n  Update: contact is now set to 'Zora'\n  Update: location is now set to 'Lumi'\n  Update: reference is now set to 'Xander'\n  Update: registry is now set to 'Nalini'\n  Update: contact is now set to 'Hana'\n  Update: location is now set to 'Kenji'\n  Update: reference is now set to 'Lumi'\n  Update: registry is now set to 'Hana'\n  Update: contact is now set to 'Kaia'\n  Update: location is now set to 'Dmitri'\n  Update: reference is now set to 'Joaquin'\n  Update: registry is now set to 'Vesna'\n  Update: contact is now set to 'Tala'\n  Update: location is now set to 'Willa'\n\nWhat is the FINAL value of each record?\nANSWER:\n- reference: [final value]\n- registry: [final value]\n- contact: [final value]\n- location: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was 'Adaeze' ever assigned to reference? [Yes/No]\nV2. Was 'Colette' ever assigned to registry? [Yes/No]",
  "gold_json": "{\"final_values\": {\"reference\": \"Joaquin\", \"registry\": \"Vesna\", \"contact\": \"Tala\", \"location\": \"Willa\"}, \"key_names\": [\"reference\", \"registry\", \"contact\", \"location\"]}"
 },
 {
  "task_id": "interference_expert_027",
  "task_type": "interference",
  "difficulty": "Expert",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: delegate is now set to '436'\n  Update: destination is now set to '126'\n  Update: location is now set to '405'\n  Update: dispatch is now set to '960'\n  Update: delegate is now set to '663'\n  Update: destination is now set to '552'\n  Update: location is now set to '199'\n  Update: dispatch is now set to '923'\n  Update: delegate is now set to '731'\n  Update: destination is now set to '212'\n  Update: location is now set to '882'\n  Update: dispatch is now set to '819'\n  Update: delegate is now set to '895'\n  Update: destination is now set to '492'\n  Update: location is now set to '239'\n  Update: dispatch is now set to '918'\n  Update: delegate is now set to '554'\n  Update: destination is now set to '129'\n  Update: location is now set to '866'\n  Update: dispatch is now set to '106'\n  Update: delegate is now set to '466'\n  Update: destination is now set to '112'\n  Update: location is now set to '556'\n  Update: dispatch is now set to '341'\n  Update: delegate is now set to '705'\n  Update: destination is now set to '909'\n  Update: location is now set to '389'\n  Update: dispatch is now set to '759'\n  Update: delegate is now set to '221'\n  Update: destination is now set to '304'\n  Update: location is now set to '173'\n  Update: dispatch is now set to '324'\n  Update: delegate is now set to '469'\n  Update: destination is now set to '278'\n  Update: location is now set to '167'\n  Update: dispatch is now set to '147'\n  Update: delegate is now set to '771'\n  Update: destination is now set to '702'\n  Update: location is now set to '481'\n  Update: dispatch is now set to '932'\n  Update: delegate is now set to '100'\n  Update: destination is now set to '377'\n  Update: location is now set to '139'\n  Update: dispatch is now set to '103'\n  Update: delegate is now set to '788'\n  Update: destination is now set to '486'\n  Update: location is now set to '418'\n  Update: dispatch is now set to '166'\n  Update: delegate is now set to '213'\n  Update: destination is now set to '514'\n  Update: location is now set to '995'\n  Update: dispatch is now set to '547'\n  Update: delegate is now set to '704'\n  Update: destination is now set to '589'\n  Update: location is now set to '762'\n  Update: dispatch is now set to '939'\n  Update: delegate is now set to '470'\n  Update: destination is now set to '146'\n  Update: location is now set to '588'\n  Update: dispatch is now set to '343'\n  Update: delegate is now set to '848'\n  Update: destination is now set to '980'\n  Update: location is now set to '640'\n  Update: dispatch is now set to '408'\n  Update: delegate is now set to '995'\n  Update: destination is now set to '957'\n  Update: location is now set to '276'\n  Update: dispatch is now set to '793'\n  Update: delegate is now set to '180'\n  Update: destination is now set to '674'\n  Update: location is now set to '699'\n  Update: dispatch is now set to '345'\n  Update: delegate is now set to '837'\n  Update: destination is now set to '643'\n  Update: location is now set to '562'\n  Update: dispatch is now set to '181'\n  Update: delegate is now set to '207'\n  Update: destination is now set to '110'\n  Update: location is now set to '514'\n  Update: dispatch is now set to '880'\n  Update: delegate is now set to '954'\n  Update: destination is now set to '586'\n  Update: location is now set to '894'\n  Update: dispatch is now set to '949'\n  Update: delegate is now set to '930'\n  Update: destination is now set to '961'\n  Update: location is now set to '669'\n  Update: dispatch is now set to '641'\n  Update: delegate is now set to '356'\n  Update: destination is now set to '438'\n  Update: location is now set to '195'\n  Update: dispatch is now set to '103'\n  Update: delegate is now set to '157'\n  Update: destination is now set to '708'\n  Update: location is now set to '520'\n  Update: dispatch is now set to '384'\n  Update: delegate is now set to '900'\n  Update: destination is now set to '751'\n  Update: location is now set to '344'\n  Update: dispatch is now set to '260'\n\nWhat is the FINAL value of each record?\nANSWER:\n- delegate: [final value]\n- destination: [final value]\n- location: [final value]\n- dispatch: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was '848' ever assigned to delegate? [Yes/No]\nV2. Was '304' ever assigned to destination? [Yes/No]",
  "gold_json": "{\"final_values\": {\"delegate\": \"900\", \"destination\": \"751\", \"location\": \"344\", \"dispatch\": \"260\"}, \"key_names\": [\"delegate\", \"destination\", \"location\", \"dispatch\"]}"
 },
 {
  "task_id": "interference_expert_028",
  "task_type": "interference",
  "difficulty": "Expert",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: delegate is now set to '595'\n  Update: assignment is now set to '768'\n  Update: coordinator is now set to '792'\n  Update: reference is now set to '413'\n  Update: delegate is now set to '117'\n  Update: assignment is now set to '241'\n  Update: coordinator is now set to '984'\n  Update: reference is now set to '431'\n  Update: delegate is now set to '138'\n  Update: assignment is now set to '608'\n  Update: coordinator is now set to '287'\n  Update: reference is now set to '709'\n  Update: delegate is now set to '902'\n  Update: assignment is now set to '281'\n  Update: coordinator is now set to '722'\n  Update: reference is now set to '561'\n  Update: delegate is now set to '141'\n  Update: assignment is now set to '434'\n  Update: coordinator is now set to '512'\n  Update: reference is now set to '422'\n  Update: delegate is now set to '502'\n  Update: assignment is now set to '848'\n  Update: coordinator is now set to '470'\n  Update: reference is now set to '217'\n  Update: delegate is now set to '961'\n  Update: assignment is now set to '463'\n  Update: coordinator is now set to '929'\n  Update: reference is now set to '189'\n  Update: delegate is now set to '806'\n  Update: assignment is now set to '151'\n  Update: coordinator is now set to '290'\n  Update: reference is now set to '714'\n  Update: delegate is now set to '392'\n  Update: assignment is now set to '333'\n  Update: coordinator is now set to '720'\n  Update: reference is now set to '115'\n  Update: delegate is now set to '310'\n  Update: assignment is now set to '623'\n  Update: coordinator is now set to '935'\n  Update: reference is now set to '910'\n  Update: delegate is now set to '680'\n  Update: assignment is now set to '954'\n  Update: coordinator is now set to '913'\n  Update: reference is now set to '382'\n  Update: delegate is now set to '643'\n  Update: assignment is now set to '801'\n  Update: coordinator is now set to '487'\n  Update: reference is now set to '366'\n  Update: delegate is now set to '365'\n  Update: assignment is now set to '867'\n  Update: coordinator is now set to '376'\n  Update: reference is now set to '690'\n  Update: delegate is now set to '576'\n  Update: assignment is now set to '688'\n  Update: coordinator is now set to '552'\n  Update: reference is now set to '887'\n  Update: delegate is now set to '202'\n  Update: assignment is now set to '226'\n  Update: coordinator is now set to '127'\n  Update: reference is now set to '921'\n  Update: delegate is now set to '781'\n  Update: assignment is now set to '847'\n  Update: coordinator is now set to '720'\n  Update: reference is now set to '149'\n  Update: delegate is now set to '397'\n  Update: assignment is now set to '750'\n  Update: coordinator is now set to '706'\n  Update: reference is now set to '733'\n  Update: delegate is now set to '400'\n  Update: assignment is now set to '583'\n  Update: coordinator is now set to '733'\n  Update: reference is now set to '189'\n  Update: delegate is now set to '205'\n  Update: assignment is now set to '187'\n  Update: coordinator is now set to '484'\n  Update: reference is now set to '383'\n  Update: delegate is now set to '484'\n  Update: assignment is now set to '335'\n  Update: coordinator is now set to '842'\n  Update: reference is now set to '331'\n  Update: delegate is now set to '639'\n  Update: assignment is now set to '545'\n  Update: coordinator is now set to '844'\n  Update: reference is now set to '256'\n  Update: delegate is now set to '677'\n  Update: assignment is now set to '451'\n  Update: coordinator is now set to '166'\n  Update: reference is now set to '360'\n  Update: delegate is now set to '188'\n  Update: assignment is now set to '488'\n  Update: coordinator is now set to '134'\n  Update: reference is now set to '139'\n  Update: delegate is now set to '215'\n  Update: assignment is now set to '354'\n  Update: coordinator is now set to '699'\n  Update: reference is now set to '184'\n  Update: delegate is now set to '759'\n  Update: assignment is now set to '534'\n  Update: coordinator is now set to '110'\n  Update: reference is now set to '364'\n\nWhat is the FINAL value of each record?\nANSWER:\n- delegate: [final value]\n- assignment: [final value]\n- coordinator: [final value]\n- reference: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was '502' ever assigned to delegate? [Yes/No]\nV2. Was '583' ever assigned to assignment? [Yes/No]",
  "gold_json": "{\"final_values\": {\"delegate\": \"759\", \"assignment\": \"534\", \"coordinator\": \"110\", \"reference\": \"364\"}, \"key_names\": [\"delegate\", \"assignment\", \"coordinator\", \"reference\"]}"
 },
 {
  "task_id": "interference_expert_029",
  "task_type": "interference",
  "difficulty": "Expert",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: destination is now set to 'Jaipur'\n  Update: liaison is now set to 'Luang Prabang'\n  Update: coordinator is now set to 'Cartagena'\n  Update: reference is now set to 'Ulaanbaatar'\n  Update: destination is now set to 'Luang Prabang'\n  Update: liaison is now set to 'Tallinn'\n  Update: coordinator is now set to 'Valetta'\n  Update: reference is now set to 'Tallinn'\n  Update: destination is now set to 'Ulaanbaatar'\n  Update: liaison is now set to 'Cusco'\n  Update: coordinator is now set to 'Kotor'\n  Update: reference is now set to 'Recife'\n  Update: destination is now set to 'Tbilisi'\n  Update: liaison is now set to 'Recife'\n  Update: coordinator is now set to 'Recife'\n  Update: reference is now set to 'Tbilisi'\n  Update: destination is now set to 'Oulu'\n  Update: liaison is now set to 'Gdansk'\n  Update: coordinator is now set to 'Tallinn'\n  Update: reference is now set to 'Reykjavik'\n  Update: destination is now set to 'Tbilisi'\n  Update: liaison is now set to 'Kotor'\n  Update: coordinator is now set to 'Valetta'\n  Update: reference is now set to 'Recife'\n  Update: destination is now set to 'Luang Prabang'\n  Update: liaison is now set to 'Fez'\n  Update: coordinator is now set to 'Gdansk'\n  Update: reference is now set to 'Ulaanbaatar'\n  Update: destination is now set to 'Oulu'\n  Update: liaison is now set to 'Oulu'\n  Update: coordinator is now set to 'Cartagena'\n  Update: reference is now set to 'Oulu'\n  Update: destination is now set to 'Kotor'\n  Update: liaison is now set to 'Tbilisi'\n  Update: coordinator is now set to 'Oulu'\n  Update: reference is now set to 'Tbilisi'\n  Update: destination is now set to 'Valetta'\n  Update: liaison is now set to 'Kumasi'\n  Update: coordinator is now set to 'Plovdiv'\n  Update: reference is now set to 'Recife'\n  Update: destination is now set to 'Cartagena'\n  Update: liaison is now set to 'Kotor'\n  Update: coordinator is now set to 'Gdansk'\n  Update: reference is now set to 'Tallinn'\n  Update: destination is now set to 'Tallinn'\n  Update: liaison is now set to 'Luang Prabang'\n  Update: coordinator is now set to 'Kotor'\n  Update: reference is now set to 'Mandalay'\n  Update: destination is now set to 'Reykjavik'\n  Update: liaison is now set to 'Valetta'\n  Update: coordinator is now set to 'Valetta'\n  Update: reference is now set to 'Jaipur'\n  Update: destination is now set to 'Luang Prabang'\n  Update: liaison is now set to 'Kumasi'\n  Update: coordinator is now set to 'Trieste'\n  Update: reference is now set to 'Kotor'\n  Update: destination is now set to 'Mandalay'\n  Update: liaison is now set to 'Valetta'\n  Update: coordinator is now set to 'Mandalay'\n  Update: reference is now set to 'Kumasi'\n  Update: destination is now set to 'Tbilisi'\n  Update: liaison is now set to 'Recife'\n  Update: coordinator is now set to 'Cartagena'\n  Update: reference is now set to 'Jaipur'\n  Update: destination is now set to 'Zanzibar'\n  Update: liaison is now set to 'Fez'\n  Update: coordinator is now set to 'Trieste'\n  Update: reference is now set to 'Recife'\n  Update: destination is now set to 'Cartagena'\n  Update: liaison is now set to 'Luang Prabang'\n  Update: coordinator is now set to 'Valetta'\n  Update: reference is now set to 'Cusco'\n  Update: destination is now set to 'Plovdiv'\n  Update: liaison is now set to 'Ulaanbaatar'\n  Update: coordinator is now set to 'Luang Prabang'\n  Update: reference is now set to 'Trieste'\n  Update: destination is now set to 'Trieste'\n  Update: liaison is now set to 'Trieste'\n  Update: coordinator is now set to 'Mandalay'\n  Update: reference is now set to 'Mandalay'\n  Update: destination is now set to 'Bruges'\n  Update: liaison is now set to 'Recife'\n  Update: coordinator is now set to 'Cartagena'\n  Update: reference is now set to 'Plovdiv'\n  Update: destination is now set to 'Mandalay'\n  Update: liaison is now set to 'Plovdiv'\n  Update: coordinator is now set to 'Plovdiv'\n  Update: reference is now set to 'Kotor'\n  Update: destination is now set to 'Cusco'\n  Update: liaison is now set to 'Recife'\n  Update: coordinator is now set to 'Recife'\n  Update: reference is now set to 'Recife'\n  Update: destination is now set to 'Mandalay'\n  Update: liaison is now set to 'Luang Prabang'\n  Update: coordinator is now set to 'Cusco'\n  Update: reference is now set to 'Gdansk'\n  Update: destination is now set to 'Valetta'\n  Update: liaison is now set to 'Ulaanbaatar'\n  Update: coordinator is now set to 'Mandalay'\n  Update: reference is now set to 'Trieste'\n\nWhat is the FINAL value of each record?\nANSWER:\n- destination: [final value]\n- liaison: [final value]\n- coordinator: [final value]\n- reference: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was 'Luang Prabang' ever assigned to destination? [Yes/No]\nV2. Was 'Tbilisi' ever assigned to liaison? [Yes/No]",
  "gold_json": "{\"final_values\": {\"destination\": \"Valetta\", \"liaison\": \"Ulaanbaatar\", \"coordinator\": \"Mandalay\", \"reference\": \"Trieste\"}, \"key_names\": [\"destination\", \"liaison\", \"coordinator\", \"reference\"]}"
 },
 {
  "task_id": "interference_expert_030",
  "task_type": "interference",
  "difficulty": "Expert",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: destination is now set to '602'\n  Update: dispatch is now set to '528'\n  Update: liaison is now set to '805'\n  Update: contact is now set to '477'\n  Update: destination is now set to '839'\n  Update: dispatch is now set to '257'\n  Update: liaison is now set to '685'\n  Update: contact is now set to '338'\n  Update: destination is now set to '638'\n  Update: dispatch is now set to '956'\n  Update: liaison is now set to '374'\n  Update: contact is now set to '655'\n  Update: destination is now set to '855'\n  Update: dispatch is now set to '463'\n  Update: liaison is now set to '857'\n  Update: contact is now set to '698'\n  Update: destination is now set to '959'\n  Update: dispatch is now set to '533'\n  Update: liaison is now set to '454'\n  Update: contact is now set to '489'\n  Update: destination is now set to '423'\n  Update: dispatch is now set to '682'\n  Update: liaison is now set to '371'\n  Update: contact is now set to '720'\n  Update: destination is now set to '585'\n  Update: dispatch is now set to '524'\n  Update: liaison is now set to '485'\n  Update: contact is now set to '818'\n  Update: destination is now set to '954'\n  Update: dispatch is now set to '691'\n  Update: liaison is now set to '264'\n  Update: contact is now set to '484'\n  Update: destination is now set to '387'\n  Update: dispatch is now set to '791'\n  Update: liaison is now set to '275'\n  Update: contact is now set to '962'\n  Update: destination is now set to '283'\n  Update: dispatch is now set to '887'\n  Update: liaison is now set to '126'\n  Update: contact is now set to '908'\n  Update: destination is now set to '713'\n  Update: dispatch is now set to '125'\n  Update: liaison is now set to '970'\n  Update: contact is now set to '906'\n  Update: destination is now set to '721'\n  Update: dispatch is now set to '234'\n  Update: liaison is now set to '448'\n  Update: contact is now set to '387'\n  Update: destination is now set to '801'\n  Update: dispatch is now set to '712'\n  Update: liaison is now set to '138'\n  Update: contact is now set to '296'\n  Update: destination is now set to '750'\n  Update: dispatch is now set to '198'\n  Update: liaison is now set to '546'\n  Update: contact is now set to '618'\n  Update: destination is now set to '499'\n  Update: dispatch is now set to '185'\n  Update: liaison is now set to '877'\n  Update: contact is now set to '956'\n  Update: destination is now set to '219'\n  Update: dispatch is now set to '393'\n  Update: liaison is now set to '106'\n  Update: contact is now set to '595'\n  Update: destination is now set to '140'\n  Update: dispatch is now set to '928'\n  Update: liaison is now set to '218'\n  Update: contact is now set to '736'\n  Update: destination is now set to '742'\n  Update: dispatch is now set to '112'\n  Update: liaison is now set to '269'\n  Update: contact is now set to '819'\n  Update: destination is now set to '200'\n  Update: dispatch is now set to '558'\n  Update: liaison is now set to '704'\n  Update: contact is now set to '825'\n  Update: destination is now set to '439'\n  Update: dispatch is now set to '127'\n  Update: liaison is now set to '454'\n  Update: contact is now set to '786'\n  Update: destination is now set to '614'\n  Update: dispatch is now set to '302'\n  Update: liaison is now set to '278'\n  Update: contact is now set to '154'\n  Update: destination is now set to '569'\n  Update: dispatch is now set to '844'\n  Update: liaison is now set to '118'\n  Update: contact is now set to '916'\n  Update: destination is now set to '839'\n  Update: dispatch is now set to '886'\n  Update: liaison is now set to '393'\n  Update: contact is now set to '823'\n  Update: destination is now set to '842'\n  Update: dispatch is now set to '557'\n  Update: liaison is now set to '235'\n  Update: contact is now set to '828'\n  Update: destination is now set to '861'\n  Update: dispatch is now set to '358'\n  Update: liaison is now set to '416'\n  Update: contact is now set to '254'\n\nWhat is the FINAL value of each record?\nANSWER:\n- destination: [final value]\n- dispatch: [final value]\n- liaison: [final value]\n- contact: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was '842' ever assigned to destination? [Yes/No]\nV2. Was '185' ever assigned to dispatch? [Yes/No]",
  "gold_json": "{\"final_values\": {\"destination\": \"861\", \"dispatch\": \"358\", \"liaison\": \"416\", \"contact\": \"254\"}, \"key_names\": [\"destination\", \"dispatch\", \"liaison\", \"contact\"]}"
 },
 {
  "task_id": "interference_expert_031",
  "task_type": "interference",
  "difficulty": "Expert",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: reference is now set to '760'\n  Update: liaison is now set to '916'\n  Update: registry is now set to '586'\n  Update: delegate is now set to '172'\n  Update: reference is now set to '542'\n  Update: liaison is now set to '393'\n  Update: registry is now set to '131'\n  Update: delegate is now set to '544'\n  Update: reference is now set to '751'\n  Update: liaison is now set to '973'\n  Update: registry is now set to '990'\n  Update: delegate is now set to '477'\n  Update: reference is now set to '947'\n  Update: liaison is now set to '692'\n  Update: registry is now set to '149'\n  Update: delegate is now set to '378'\n  Update: reference is now set to '799'\n  Update: liaison is now set to '485'\n  Update: registry is now set to '389'\n  Update: delegate is now set to '107'\n  Update: reference is now set to '854'\n  Update: liaison is now set to '137'\n  Update: registry is now set to '802'\n  Update: delegate is now set to '973'\n  Update: reference is now set to '514'\n  Update: liaison is now set to '708'\n  Update: registry is now set to '479'\n  Update: delegate is now set to '212'\n  Update: reference is now set to '453'\n  Update: liaison is now set to '185'\n  Update: registry is now set to '629'\n  Update: delegate is now set to '528'\n  Update: reference is now set to '416'\n  Update: liaison is now set to '766'\n  Update: registry is now set to '640'\n  Update: delegate is now set to '474'\n  Update: reference is now set to '867'\n  Update: liaison is now set to '240'\n  Update: registry is now set to '720'\n  Update: delegate is now set to '444'\n  Update: reference is now set to '171'\n  Update: liaison is now set to '790'\n  Update: registry is now set to '709'\n  Update: delegate is now set to '621'\n  Update: reference is now set to '496'\n  Update: liaison is now set to '144'\n  Update: registry is now set to '502'\n  Update: delegate is now set to '563'\n  Update: reference is now set to '735'\n  Update: liaison is now set to '601'\n  Update: registry is now set to '280'\n  Update: delegate is now set to '633'\n  Update: reference is now set to '446'\n  Update: liaison is now set to '217'\n  Update: registry is now set to '935'\n  Update: delegate is now set to '982'\n  Update: reference is now set to '470'\n  Update: liaison is now set to '221'\n  Update: registry is now set to '340'\n  Update: delegate is now set to '562'\n  Update: reference is now set to '362'\n  Update: liaison is now set to '494'\n  Update: registry is now set to '461'\n  Update: delegate is now set to '402'\n  Update: reference is now set to '767'\n  Update: liaison is now set to '274'\n  Update: registry is now set to '833'\n  Update: delegate is now set to '611'\n  Update: reference is now set to '995'\n  Update: liaison is now set to '127'\n  Update: registry is now set to '772'\n  Update: delegate is now set to '819'\n  Update: reference is now set to '464'\n  Update: liaison is now set to '488'\n  Update: registry is now set to '179'\n  Update: delegate is now set to '448'\n  Update: reference is now set to '354'\n  Update: liaison is now set to '901'\n  Update: registry is now set to '551'\n  Update: delegate is now set to '740'\n  Update: reference is now set to '626'\n  Update: liaison is now set to '457'\n  Update: registry is now set to '534'\n  Update: delegate is now set to '515'\n  Update: reference is now set to '841'\n  Update: liaison is now set to '915'\n  Update: registry is now set to '753'\n  Update: delegate is now set to '171'\n  Update: reference is now set to '184'\n  Update: liaison is now set to '928'\n  Update: registry is now set to '108'\n  Update: delegate is now set to '710'\n  Update: reference is now set to '201'\n  Update: liaison is now set to '286'\n  Update: registry is now set to '389'\n  Update: delegate is now set to '180'\n  Update: reference is now set to '689'\n  Update: liaison is now set to '831'\n  Update: registry is now set to '325'\n  Update: delegate is now set to '444'\n\nWhat is the FINAL value of each record?\nANSWER:\n- reference: [final value]\n- liaison: [final value]\n- registry: [final value]\n- delegate: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was '841' ever assigned to reference? [Yes/No]\nV2. Was '127' ever assigned to liaison? [Yes/No]",
  "gold_json": "{\"final_values\": {\"reference\": \"689\", \"liaison\": \"831\", \"registry\": \"325\", \"delegate\": \"444\"}, \"key_names\": [\"reference\", \"liaison\", \"registry\", \"delegate\"]}"
 },
 {
  "task_id": "interference_frontier_032",
  "task_type": "interference",
  "difficulty": "Frontier",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: registry is now set to 'Tallinn'\n  Update: coordinator is now set to 'Zanzibar'\n  Update: location is now set to 'Tbilisi'\n  Update: reference is now set to 'Kotor'\n  Update: delegate is now set to 'Mandalay'\n  Update: dispatch is now set to 'Gdansk'\n  Update: contact is now set to 'Gdansk'\n  Update: destination is now set to 'Gdansk'\n  Update: registry is now set to 'Tbilisi'\n  Update: coordinator is now set to 'Recife'\n  Update: location is now set to 'Cusco'\n  Update: reference is now set to 'Oulu'\n  Update: delegate is now set to 'Reykjavik'\n  Update: dispatch is now set to 'Tbilisi'\n  Update: contact is now set to 'Jaipur'\n  Update: destination is now set to 'Plovdiv'\n  Update: registry is now set to 'Kumasi'\n  Update: coordinator is now set to 'Ulaanbaatar'\n  Update: location is now set to 'Luang Prabang'\n  Update: reference is now set to 'Bruges'\n  Update: delegate is now set to 'Zanzibar'\n  Update: dispatch is now set to 'Recife'\n  Update: contact is now set to 'Bruges'\n  Update: destination is now set to 'Tallinn'\n  Update: registry is now set to 'Tallinn'\n  Update: coordinator is now set to 'Valetta'\n  Update: location is now set to 'Plovdiv'\n  Update: reference is now set to 'Tbilisi'\n  Update: delegate is now set to 'Kumasi'\n  Update: dispatch is now set to 'Trieste'\n  Update: contact is now set to 'Reykjavik'\n  Update: destination is now set to 'Trieste'\n  Update: registry is now set to 'Mandalay'\n  Update: coordinator is now set to 'Recife'\n  Update: location is now set to 'Kotor'\n  Update: reference is now set to 'Mandalay'\n  Update: delegate is now set to 'Cusco'\n  Update: dispatch is now set to 'Plovdiv'\n  Update: contact is now set to 'Bruges'\n  Update: destination is now set to 'Kotor'\n  Update: registry is now set to 'Tbilisi'\n  Update: coordinator is now set to 'Tallinn'\n  Update: location is now set to 'Oulu'\n  Update: reference is now set to 'Kotor'\n  Update: delegate is now set to 'Reykjavik'\n  Update: dispatch is now set to 'Luang Prabang'\n  Update: contact is now set to 'Recife'\n  Update: destination is now set to 'Cusco'\n  Update: registry is now set to 'Cartagena'\n  Update: coordinator is now set to 'Gdansk'\n  Update: location is now set to 'Luang Prabang'\n  Update: reference is now set to 'Valetta'\n  Update: delegate is now set to 'Oulu'\n  Update: dispatch is now set to 'Kumasi'\n  Update: contact is now set to 'Zanzibar'\n  Update: destination is now set to 'Gdansk'\n  Update: registry is now set to 'Kumasi'\n  Update: coordinator is now set to 'Bruges'\n  Update: location is now set to 'Reykjavik'\n  Update: reference is now set to 'Oulu'\n  Update: delegate is now set to 'Tbilisi'\n  Update: dispatch is now set to 'Luang Prabang'\n  Update: contact is now set to 'Cartagena'\n  Update: destination is now set to 'Kotor'\n  Update: registry is now set to 'Fez'\n  Update: coordinator is now set to 'Zanzibar'\n  Update: location is now set to 'Trieste'\n  Update: reference is now set to 'Mandalay'\n  Update: delegate is now set to 'Oulu'\n  Update: dispatch is now set to 'Cartagena'\n  Update: contact is now set to 'Luang Prabang'\n  Update: destination is now set to 'Zanzibar'\n  Update: registry is now set to 'Oulu'\n  Update: coordinator is now set to 'Kotor'\n  Update: location is now set to 'Tbilisi'\n  Update: reference is now set to 'Recife'\n  Update: delegate is now set to 'Reykjavik'\n  Update: dispatch is now set to 'Zanzibar'\n  Update: contact is now set to 'Zanzibar'\n  Update: destination is now set to 'Kotor'\n  Update: registry is now set to 'Gdansk'\n  Update: coordinator is now set to 'Gdansk'\n  Update: location is now set to 'Gdansk'\n  Update: reference is now set to 'Luang Prabang'\n  Update: delegate is now set to 'Cusco'\n  Update: dispatch is now set to 'Reykjavik'\n  Update: contact is now set to 'Cartagena'\n  Update: destination is now set to 'Valetta'\n  Update: registry is now set to 'Bruges'\n  Update: coordinator is now set to 'Trieste'\n  Update: location is now set to 'Zanzibar'\n  Update: reference is now set to 'Tallinn'\n  Update: delegate is now set to 'Valetta'\n  Update: dispatch is now set to 'Kumasi'\n  Update: contact is now set to 'Mandalay'\n  Update: destination is now set to 'Kumasi'\n  Update: registry is now set to 'Oulu'\n  Update: coordinator is now set to 'Recife'\n  Update: location is now set to 'Oulu'\n  Update: reference is now set to 'Tbilisi'\n  Update: delegate is now set to 'Kotor'\n  Update: dispatch is now set to 'Oulu'\n  Update: contact is now set to 'Jaipur'\n  Update: destination is now set to 'Valetta'\n  Update: registry is now set to 'Gdansk'\n  Update: coordinator is now set to 'Fez'\n  Update: location is now set to 'Ulaanbaatar'\n  Update: reference is now set to 'Gdansk'\n  Update: delegate is now set to 'Luang Prabang'\n  Update: dispatch is now set to 'Bruges'\n  Update: contact is now set to 'Gdansk'\n  Update: destination is now set to 'Tbilisi'\n  Update: registry is now set to 'Bruges'\n  Update: coordinator is now set to 'Mandalay'\n  Update: location is now set to 'Plovdiv'\n  Update: reference is now set to 'Jaipur'\n  Update: delegate is now set to 'Oulu'\n  Update: dispatch is now set to 'Kotor'\n  Update: contact is now set to 'Reykjavik'\n  Update: destination is now set to 'Kumasi'\n  Update: registry is now set to 'Cartagena'\n  Update: coordinator is now set to 'Zanzibar'\n  Update: location is now set to 'Tbilisi'\n  Update: reference is now set to 'Mandalay'\n  Update: delegate is now set to 'Tbilisi'\n  Update: dispatch is now set to 'Ulaanbaatar'\n  Update: contact is now set to 'Jaipur'\n  Update: destination is now set to 'Recife'\n  Update: registry is now set to 'Mandalay'\n  Update: coordinator is now set to 'Trieste'\n  Update: location is now set to 'Mandalay'\n  Update: reference is now set to 'Zanzibar'\n  Update: delegate is now set to 'Plovdiv'\n  Update: dispatch is now set to 'Zanzibar'\n  Update: contact is now set to 'Oulu'\n  Update: destination is now set to 'Reykjavik'\n  Update: registry is now set to 'Ulaanbaatar'\n  Update: coordinator is now set to 'Kumasi'\n  Update: location is now set to 'Plovdiv'\n  Update: reference is now set to 'Oulu'\n  Update: delegate is now set to 'Luang Prabang'\n  Update: dispatch is now set to 'Bruges'\n  Update: contact is now set to 'Cartagena'\n  Update: destination is now set to 'Fez'\n  Update: registry is now set to 'Jaipur'\n  Update: coordinator is now set to 'Fez'\n  Update: location is now set to 'Trieste'\n  Update: reference is now set to 'Cartagena'\n  Update: delegate is now set to 'Fez'\n  Update: dispatch is now set to 'Trieste'\n  Update: contact is now set to 'Reykjavik'\n  Update: destination is now set to 'Valetta'\n  Update: registry is now set to 'Bruges'\n  Update: coordinator is now set to 'Valetta'\n  Update: location is now set to 'Cusco'\n  Update: reference is now set to 'Cusco'\n  Update: delegate is now set to 'Recife'\n  Update: dispatch is now set to 'Zanzibar'\n  Update: contact is now set to 'Zanzibar'\n  Update: destination is now set to 'Cusco'\n  Update: registry is now set to 'Mandalay'\n  Update: coordinator is now set to 'Cusco'\n  Update: location is now set to 'Gdansk'\n  Update: reference is now set to 'Ulaanbaatar'\n  Update: delegate is now set to 'Tbilisi'\n  Update: dispatch is now set to 'Reykjavik'\n  Update: contact is now set to 'Gdansk'\n  Update: destination is now set to 'Cartagena'\n  Update: registry is now set to 'Oulu'\n  Update: coordinator is now set to 'Plovdiv'\n  Update: location is now set to 'Reykjavik'\n  Update: reference is now set to 'Cartagena'\n  Update: delegate is now set to 'Trieste'\n  Update: dispatch is now set to 'Gdansk'\n  Update: contact is now set to 'Tbilisi'\n  Update: destination is now set to 'Oulu'\n  Update: registry is now set to 'Fez'\n  Update: coordinator is now set to 'Oulu'\n  Update: location is now set to 'Fez'\n  Update: reference is now set to 'Fez'\n  Update: delegate is now set to 'Fez'\n  Update: dispatch is now set to 'Trieste'\n  Update: contact is now set to 'Oulu'\n  Update: destination is now set to 'Fez'\n  Update: registry is now set to 'Valetta'\n  Update: coordinator is now set to 'Kumasi'\n  Update: location is now set to 'Jaipur'\n  Update: reference is now set to 'Kumasi'\n  Update: delegate is now set to 'Ulaanbaatar'\n  Update: dispatch is now set to 'Gdansk'\n  Update: contact is now set to 'Tallinn'\n  Update: destination is now set to 'Ulaanbaatar'\n  Update: registry is now set to 'Fez'\n  Update: coordinator is now set to 'Cartagena'\n  Update: location is now set to 'Luang Prabang'\n  Update: reference is now set to 'Tallinn'\n  Update: delegate is now set to 'Jaipur'\n  Update: dispatch is now set to 'Luang Prabang'\n  Update: contact is now set to 'Valetta'\n  Update: destination is now set to 'Plovdiv'\n  Update: registry is now set to 'Mandalay'\n  Update: coordinator is now set to 'Cusco'\n  Update: location is now set to 'Ulaanbaatar'\n  Update: reference is now set to 'Fez'\n  Update: delegate is now set to 'Valetta'\n  Update: dispatch is now set to 'Valetta'\n  Update: contact is now set to 'Recife'\n  Update: destination is now set to 'Cartagena'\n  Update: registry is now set to 'Recife'\n  Update: coordinator is now set to 'Oulu'\n  Update: location is now set to 'Bruges'\n  Update: reference is now set to 'Cusco'\n  Update: delegate is now set to 'Bruges'\n  Update: dispatch is now set to 'Cusco'\n  Update: contact is now set to 'Gdansk'\n  Update: destination is now set to 'Luang Prabang'\n  Update: registry is now set to 'Trieste'\n  Update: coordinator is now set to 'Trieste'\n  Update: location is now set to 'Trieste'\n  Update: reference is now set to 'Plovdiv'\n  Update: delegate is now set to 'Oulu'\n  Update: dispatch is now set to 'Mandalay'\n  Update: contact is now set to 'Zanzibar'\n  Update: destination is now set to 'Oulu'\n  Update: registry is now set to 'Kotor'\n  Update: coordinator is now set to 'Zanzibar'\n  Update: location is now set to 'Cartagena'\n  Update: reference is now set to 'Gdansk'\n  Update: delegate is now set to 'Reykjavik'\n  Update: dispatch is now set to 'Kotor'\n  Update: contact is now set to 'Mandalay'\n  Update: destination is now set to 'Cusco'\n  Update: registry is now set to 'Luang Prabang'\n  Update: coordinator is now set to 'Valetta'\n  Update: location is now set to 'Oulu'\n  Update: reference is now set to 'Valetta'\n  Update: delegate is now set to 'Bruges'\n  Update: dispatch is now set to 'Bruges'\n  Update: contact is now set to 'Trieste'\n  Update: destination is now set to 'Gdansk'\n  Update: registry is now set to 'Zanzibar'\n  Update: coordinator is now set to 'Cusco'\n  Update: location is now set to 'Kotor'\n  Update: reference is now set to 'Plovdiv'\n  Update: delegate is now set to 'Tbilisi'\n  Update: dispatch is now set to 'Kotor'\n  Update: contact is now set to 'Tbilisi'\n  Update: destination is now set to 'Kumasi'\n  Update: registry is now set to 'Recife'\n  Update: coordinator is now set to 'Recife'\n  Update: location is now set to 'Ulaanbaatar'\n  Update: reference is now set to 'Gdansk'\n  Update: delegate is now set to 'Kumasi'\n  Update: dispatch is now set to 'Cartagena'\n  Update: contact is now set to 'Reykjavik'\n  Update: destination is now set to 'Gdansk'\n  Update: registry is now set to 'Kotor'\n  Update: coordinator is now set to 'Bruges'\n  Update: location is now set to 'Bruges'\n  Update: reference is now set to 'Jaipur'\n  Update: delegate is now set to 'Zanzibar'\n  Update: dispatch is now set to 'Fez'\n  Update: contact is now set to 'Kotor'\n  Update: destination is now set to 'Mandalay'\n  Update: registry is now set to 'Recife'\n  Update: coordinator is now set to 'Ulaanbaatar'\n  Update: location is now set to 'Oulu'\n  Update: reference is now set to 'Gdansk'\n  Update: delegate is now set to 'Kumasi'\n  Update: dispatch is now set to 'Mandalay'\n  Update: contact is now set to 'Tallinn'\n  Update: destination is now set to 'Tbilisi'\n  Update: registry is now set to 'Valetta'\n  Update: coordinator is now set to 'Tbilisi'\n  Update: location is now set to 'Ulaanbaatar'\n  Update: reference is now set to 'Reykjavik'\n  Update: delegate is now set to 'Tbilisi'\n  Update: dispatch is now set to 'Cusco'\n  Update: contact is now set to 'Zanzibar'\n  Update: destination is now set to 'Trieste'\n  Update: registry is now set to 'Plovdiv'\n  Update: coordinator is now set to 'Mandalay'\n  Update: location is now set to 'Trieste'\n  Update: reference is now set to 'Recife'\n  Update: delegate is now set to 'Cartagena'\n  Update: dispatch is now set to 'Recife'\n  Update: contact is now set to 'Ulaanbaatar'\n  Update: destination is now set to 'Kumasi'\n  Update: registry is now set to 'Zanzibar'\n  Update: coordinator is now set to 'Trieste'\n  Update: location is now set to 'Fez'\n  Update: reference is now set to 'Trieste'\n  Update: delegate is now set to 'Fez'\n  Update: dispatch is now set to 'Kumasi'\n  Update: contact is now set to 'Tbilisi'\n  Update: destination is now set to 'Gdansk'\n  Update: registry is now set to 'Ulaanbaatar'\n  Update: coordinator is now set to 'Oulu'\n  Update: location is now set to 'Kotor'\n  Update: reference is now set to 'Plovdiv'\n  Update: delegate is now set to 'Kumasi'\n  Update: dispatch is now set to 'Plovdiv'\n  Update: contact is now set to 'Kotor'\n  Update: destination is now set to 'Fez'\n  Update: registry is now set to 'Plovdiv'\n  Update: coordinator is now set to 'Gdansk'\n  Update: location is now set to 'Jaipur'\n  Update: reference is now set to 'Valetta'\n  Update: delegate is now set to 'Cusco'\n  Update: dispatch is now set to 'Luang Prabang'\n  Update: contact is now set to 'Bruges'\n  Update: destination is now set to 'Luang Prabang'\n  Update: registry is now set to 'Recife'\n  Update: coordinator is now set to 'Cartagena'\n  Update: location is now set to 'Zanzibar'\n  Update: reference is now set to 'Fez'\n  Update: delegate is now set to 'Tbilisi'\n  Update: dispatch is now set to 'Mandalay'\n  Update: contact is now set to 'Ulaanbaatar'\n  Update: destination is now set to 'Ulaanbaatar'\n  Update: registry is now set to 'Oulu'\n  Update: coordinator is now set to 'Ulaanbaatar'\n  Update: location is now set to 'Kotor'\n  Update: reference is now set to 'Kotor'\n  Update: delegate is now set to 'Cartagena'\n  Update: dispatch is now set to 'Zanzibar'\n  Update: contact is now set to 'Cusco'\n  Update: destination is now set to 'Trieste'\n  Update: registry is now set to 'Kotor'\n  Update: coordinator is now set to 'Tbilisi'\n  Update: location is now set to 'Gdansk'\n  Update: reference is now set to 'Cusco'\n  Update: delegate is now set to 'Valetta'\n  Update: dispatch is now set to 'Recife'\n  Update: contact is now set to 'Trieste'\n  Update: destination is now set to 'Luang Prabang'\n  Update: registry is now set to 'Luang Prabang'\n  Update: coordinator is now set to 'Oulu'\n  Update: location is now set to 'Mandalay'\n  Update: reference is now set to 'Reykjavik'\n  Update: delegate is now set to 'Jaipur'\n  Update: dispatch is now set to 'Luang Prabang'\n  Update: contact is now set to 'Reykjavik'\n  Update: destination is now set to 'Plovdiv'\n  Update: registry is now set to 'Cartagena'\n  Update: coordinator is now set to 'Cusco'\n  Update: location is now set to 'Gdansk'\n  Update: reference is now set to 'Plovdiv'\n  Update: delegate is now set to 'Gdansk'\n  Update: dispatch is now set to 'Cartagena'\n  Update: contact is now set to 'Valetta'\n  Update: destination is now set to 'Kumasi'\n  Update: registry is now set to 'Ulaanbaatar'\n  Update: coordinator is now set to 'Gdansk'\n  Update: location is now set to 'Cartagena'\n  Update: reference is now set to 'Tallinn'\n  Update: delegate is now set to 'Oulu'\n  Update: dispatch is now set to 'Recife'\n  Update: contact is now set to 'Kumasi'\n  Update: destination is now set to 'Valetta'\n  Update: registry is now set to 'Reykjavik'\n  Update: coordinator is now set to 'Kotor'\n  Update: location is now set to 'Plovdiv'\n  Update: reference is now set to 'Oulu'\n  Update: delegate is now set to 'Tbilisi'\n  Update: dispatch is now set to 'Luang Prabang'\n  Update: contact is now set to 'Jaipur'\n  Update: destination is now set to 'Cartagena'\n  Update: registry is now set to 'Fez'\n  Update: coordinator is now set to 'Zanzibar'\n  Update: location is now set to 'Fez'\n  Update: reference is now set to 'Jaipur'\n  Update: delegate is now set to 'Jaipur'\n  Update: dispatch is now set to 'Kumasi'\n  Update: contact is now set to 'Tbilisi'\n  Update: destination is now set to 'Zanzibar'\n  Update: registry is now set to 'Cartagena'\n  Update: coordinator is now set to 'Trieste'\n  Update: location is now set to 'Gdansk'\n  Update: reference is now set to 'Kumasi'\n  Update: delegate is now set to 'Kotor'\n  Update: dispatch is now set to 'Trieste'\n  Update: contact is now set to 'Ulaanbaatar'\n  Update: destination is now set to 'Bruges'\n  Update: registry is now set to 'Tallinn'\n  Update: coordinator is now set to 'Tallinn'\n  Update: location is now set to 'Mandalay'\n  Update: reference is now set to 'Reykjavik'\n  Update: delegate is now set to 'Luang Prabang'\n  Update: dispatch is now set to 'Oulu'\n  Update: contact is now set to 'Reykjavik'\n  Update: destination is now set to 'Trieste'\n  Update: registry is now set to 'Trieste'\n  Update: coordinator is now set to 'Gdansk'\n  Update: location is now set to 'Tallinn'\n  Update: reference is now set to 'Valetta'\n  Update: delegate is now set to 'Oulu'\n  Update: dispatch is now set to 'Cusco'\n  Update: contact is now set to 'Luang Prabang'\n  Update: destination is now set to 'Reykjavik'\n\nWhat is the FINAL value of each record?\nANSWER:\n- registry: [final value]\n- coordinator: [final value]\n- location: [final value]\n- reference: [final value]\n- delegate: [final value]\n- dispatch: [final value]\n- contact: [final value]\n- destination: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was 'Valetta' ever assigned to registry? [Yes/No]\nV2. Was 'Cusco' ever assigned to coordinator? [Yes/No]\nV3. Was 'Trieste' ever assigned to location? [Yes/No]\nV4. Was 'Tallinn' ever assigned to reference? [Yes/No]",
  "gold_json": "{\"final_values\": {\"registry\": \"Trieste\", \"coordinator\": \"Gdansk\", \"location\": \"Tallinn\", \"reference\": \"Valetta\", \"delegate\": \"Oulu\", \"dispatch\": \"Cusco\", \"contact\": \"Luang Prabang\", \"destination\": \"Reykjavik\"}, \"key_names\": [\"registry\", \"coordinator\", \"location\", \"reference\", \"delegate\", \"dispatch\", \"contact\", \"destination\"]}"
 },
 {
  "task_id": "interference_frontier_033",
  "task_type": "interference",
  "difficulty": "Frontier",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: registry is now set to 'Luang Prabang'\n  Update: dispatch is now set to 'Zanzibar'\n  Update: reference is now set to 'Cartagena'\n  Update: coordinator is now set to 'Trieste'\n  Update: destination is now set to 'Zanzibar'\n  Update: location is now set to 'Valetta'\n  Update: delegate is now set to 'Tbilisi'\n  Update: contact is now set to 'Luang Prabang'\n  Update: registry is now set to 'Mandalay'\n  Update: dispatch is now set to 'Tallinn'\n  Update: reference is now set to 'Valetta'\n  Update: coordinator is now set to 'Tallinn'\n  Update: destination is now set to 'Kotor'\n  Update: location is now set to 'Kumasi'\n  Update: delegate is now set to 'Valetta'\n  Update: contact is now set to 'Trieste'\n  Update: registry is now set to 'Ulaanbaatar'\n  Update: dispatch is now set to 'Trieste'\n  Update: reference is now set to 'Cartagena'\n  Update: coordinator is now set to 'Oulu'\n  Update: destination is now set to 'Zanzibar'\n  Update: location is now set to 'Ulaanbaatar'\n  Update: delegate is now set to 'Cartagena'\n  Update: contact is now set to 'Kotor'\n  Update: registry is now set to 'Kumasi'\n  Update: dispatch is now set to 'Luang Prabang'\n  Update: reference is now set to 'Gdansk'\n  Update: coordinator is now set to 'Kotor'\n  Update: destination is now set to 'Cusco'\n  Update: location is now set to 'Gdansk'\n  Update: delegate is now set to 'Gdansk'\n  Update: contact is now set to 'Cusco'\n  Update: registry is now set to 'Plovdiv'\n  Update: dispatch is now set to 'Jaipur'\n  Update: reference is now set to 'Tallinn'\n  Update: coordinator is now set to 'Fez'\n  Update: destination is now set to 'Ulaanbaatar'\n  Update: location is now set to 'Reykjavik'\n  Update: delegate is now set to 'Trieste'\n  Update: contact is now set to 'Tbilisi'\n  Update: registry is now set to 'Kumasi'\n  Update: dispatch is now set to 'Mandalay'\n  Update: reference is now set to 'Mandalay'\n  Update: coordinator is now set to 'Gdansk'\n  Update: destination is now set to 'Bruges'\n  Update: location is now set to 'Kumasi'\n  Update: delegate is now set to 'Ulaanbaatar'\n  Update: contact is now set to 'Oulu'\n  Update: registry is now set to 'Trieste'\n  Update: dispatch is now set to 'Valetta'\n  Update: reference is now set to 'Kotor'\n  Update: coordinator is now set to 'Cusco'\n  Update: destination is now set to 'Zanzibar'\n  Update: location is now set to 'Cusco'\n  Update: delegate is now set to 'Cartagena'\n  Update: contact is now set to 'Recife'\n  Update: registry is now set to 'Mandalay'\n  Update: dispatch is now set to 'Jaipur'\n  Update: reference is now set to 'Trieste'\n  Update: coordinator is now set to 'Ulaanbaatar'\n  Update: destination is now set to 'Trieste'\n  Update: location is now set to 'Valetta'\n  Update: delegate is now set to 'Plovdiv'\n  Update: contact is now set to 'Mandalay'\n  Update: registry is now set to 'Jaipur'\n  Update: dispatch is now set to 'Valetta'\n  Update: reference is now set to 'Kotor'\n  Update: coordinator is now set to 'Tallinn'\n  Update: destination is now set to 'Recife'\n  Update: location is now set to 'Oulu'\n  Update: delegate is now set to 'Gdansk'\n  Update: contact is now set to 'Tallinn'\n  Update: registry is now set to 'Bruges'\n  Update: dispatch is now set to 'Recife'\n  Update: reference is now set to 'Mandalay'\n  Update: coordinator is now set to 'Luang Prabang'\n  Update: destination is now set to 'Kotor'\n  Update: location is now set to 'Mandalay'\n  Update: delegate is now set to 'Jaipur'\n  Update: contact is now set to 'Luang Prabang'\n  Update: registry is now set to 'Jaipur'\n  Update: dispatch is now set to 'Mandalay'\n  Update: reference is now set to 'Cusco'\n  Update: coordinator is now set to 'Cartagena'\n  Update: destination is now set to 'Jaipur'\n  Update: location is now set to 'Reykjavik'\n  Update: delegate is now set to 'Tbilisi'\n  Update: contact is now set to 'Jaipur'\n  Update: registry is now set to 'Cartagena'\n  Update: dispatch is now set to 'Kumasi'\n  Update: reference is now set to 'Luang Prabang'\n  Update: coordinator is now set to 'Luang Prabang'\n  Update: destination is now set to 'Recife'\n  Update: location is now set to 'Ulaanbaatar'\n  Update: delegate is now set to 'Tallinn'\n  Update: contact is now set to 'Oulu'\n  Update: registry is now set to 'Ulaanbaatar'\n  Update: dispatch is now set to 'Reykjavik'\n  Update: reference is now set to 'Kotor'\n  Update: coordinator is now set to 'Zanzibar'\n  Update: destination is now set to 'Jaipur'\n  Update: location is now set to 'Fez'\n  Update: delegate is now set to 'Trieste'\n  Update: contact is now set to 'Cusco'\n  Update: registry is now set to 'Jaipur'\n  Update: dispatch is now set to 'Recife'\n  Update: reference is now set to 'Zanzibar'\n  Update: coordinator is now set to 'Kumasi'\n  Update: destination is now set to 'Zanzibar'\n  Update: location is now set to 'Valetta'\n  Update: delegate is now set to 'Zanzibar'\n  Update: contact is now set to 'Oulu'\n  Update: registry is now set to 'Recife'\n  Update: dispatch is now set to 'Valetta'\n  Update: reference is now set to 'Reykjavik'\n  Update: coordinator is now set to 'Mandalay'\n  Update: destination is now set to 'Kotor'\n  Update: location is now set to 'Zanzibar'\n  Update: delegate is now set to 'Luang Prabang'\n  Update: contact is now set to 'Recife'\n  Update: registry is now set to 'Zanzibar'\n  Update: dispatch is now set to 'Tbilisi'\n  Update: reference is now set to 'Plovdiv'\n  Update: coordinator is now set to 'Tbilisi'\n  Update: destination is now set to 'Valetta'\n  Update: location is now set to 'Plovdiv'\n  Update: delegate is now set to 'Reykjavik'\n  Update: contact is now set to 'Tallinn'\n  Update: registry is now set to 'Kumasi'\n  Update: dispatch is now set to 'Zanzibar'\n  Update: reference is now set to 'Trieste'\n  Update: coordinator is now set to 'Recife'\n  Update: destination is now set to 'Fez'\n  Update: location is now set to 'Tbilisi'\n  Update: delegate is now set to 'Kumasi'\n  Update: contact is now set to 'Kotor'\n  Update: registry is now set to 'Tallinn'\n  Update: dispatch is now set to 'Recife'\n  Update: reference is now set to 'Kotor'\n  Update: coordinator is now set to 'Cusco'\n  Update: destination is now set to 'Kotor'\n  Update: location is now set to 'Plovdiv'\n  Update: delegate is now set to 'Luang Prabang'\n  Update: contact is now set to 'Valetta'\n  Update: registry is now set to 'Fez'\n  Update: dispatch is now set to 'Luang Prabang'\n  Update: reference is now set to 'Fez'\n  Update: coordinator is now set to 'Ulaanbaatar'\n  Update: destination is now set to 'Tbilisi'\n  Update: location is now set to 'Kotor'\n  Update: delegate is now set to 'Jaipur'\n  Update: contact is now set to 'Trieste'\n  Update: registry is now set to 'Luang Prabang'\n  Update: dispatch is now set to 'Jaipur'\n  Update: reference is now set to 'Gdansk'\n  Update: coordinator is now set to 'Recife'\n  Update: destination is now set to 'Kumasi'\n  Update: location is now set to 'Recife'\n  Update: delegate is now set to 'Cusco'\n  Update: contact is now set to 'Luang Prabang'\n  Update: registry is now set to 'Kumasi'\n  Update: dispatch is now set to 'Tbilisi'\n  Update: reference is now set to 'Recife'\n  Update: coordinator is now set to 'Jaipur'\n  Update: destination is now set to 'Ulaanbaatar'\n  Update: location is now set to 'Cusco'\n  Update: delegate is now set to 'Tbilisi'\n  Update: contact is now set to 'Tbilisi'\n  Update: registry is now set to 'Luang Prabang'\n  Update: dispatch is now set to 'Jaipur'\n  Update: reference is now set to 'Valetta'\n  Update: coordinator is now set to 'Mandalay'\n  Update: destination is now set to 'Reykjavik'\n  Update: location is now set to 'Mandalay'\n  Update: delegate is now set to 'Ulaanbaatar'\n  Update: contact is now set to 'Jaipur'\n  Update: registry is now set to 'Cusco'\n  Update: dispatch is now set to 'Recife'\n  Update: reference is now set to 'Oulu'\n  Update: coordinator is now set to 'Plovdiv'\n  Update: destination is now set to 'Bruges'\n  Update: location is now set to 'Recife'\n  Update: delegate is now set to 'Plovdiv'\n  Update: contact is now set to 'Kumasi'\n  Update: registry is now set to 'Zanzibar'\n  Update: dispatch is now set to 'Valetta'\n  Update: reference is now set to 'Tbilisi'\n  Update: coordinator is now set to 'Kumasi'\n  Update: destination is now set to 'Ulaanbaatar'\n  Update: location is now set to 'Tallinn'\n  Update: delegate is now set to 'Recife'\n  Update: contact is now set to 'Reykjavik'\n  Update: registry is now set to 'Cusco'\n  Update: dispatch is now set to 'Tbilisi'\n  Update: reference is now set to 'Tallinn'\n  Update: coordinator is now set to 'Mandalay'\n  Update: destination is now set to 'Kotor'\n  Update: location is now set to 'Cusco'\n  Update: delegate is now set to 'Tallinn'\n  Update: contact is now set to 'Zanzibar'\n  Update: registry is now set to 'Recife'\n  Update: dispatch is now set to 'Gdansk'\n  Update: reference is now set to 'Kumasi'\n  Update: coordinator is now set to 'Fez'\n  Update: destination is now set to 'Gdansk'\n  Update: location is now set to 'Jaipur'\n  Update: delegate is now set to 'Valetta'\n  Update: contact is now set to 'Cusco'\n  Update: registry is now set to 'Ulaanbaatar'\n  Update: dispatch is now set to 'Valetta'\n  Update: reference is now set to 'Gdansk'\n  Update: coordinator is now set to 'Gdansk'\n  Update: destination is now set to 'Kumasi'\n  Update: location is now set to 'Kotor'\n  Update: delegate is now set to 'Recife'\n  Update: contact is now set to 'Fez'\n  Update: registry is now set to 'Fez'\n  Update: dispatch is now set to 'Reykjavik'\n  Update: reference is now set to 'Zanzibar'\n  Update: coordinator is now set to 'Kotor'\n  Update: destination is now set to 'Mandalay'\n  Update: location is now set to 'Reykjavik'\n  Update: delegate is now set to 'Bruges'\n  Update: contact is now set to 'Trieste'\n  Update: registry is now set to 'Oulu'\n  Update: dispatch is now set to 'Ulaanbaatar'\n  Update: reference is now set to 'Recife'\n  Update: coordinator is now set to 'Trieste'\n  Update: destination is now set to 'Kotor'\n  Update: location is now set to 'Kotor'\n  Update: delegate is now set to 'Recife'\n  Update: contact is now set to 'Tbilisi'\n  Update: registry is now set to 'Ulaanbaatar'\n  Update: dispatch is now set to 'Kumasi'\n  Update: reference is now set to 'Bruges'\n  Update: coordinator is now set to 'Oulu'\n  Update: destination is now set to 'Jaipur'\n  Update: location is now set to 'Jaipur'\n  Update: delegate is now set to 'Bruges'\n  Update: contact is now set to 'Plovdiv'\n  Update: registry is now set to 'Tbilisi'\n  Update: dispatch is now set to 'Gdansk'\n  Update: reference is now set to 'Plovdiv'\n  Update: coordinator is now set to 'Reykjavik'\n  Update: destination is now set to 'Luang Prabang'\n  Update: location is now set to 'Plovdiv'\n  Update: delegate is now set to 'Fez'\n  Update: contact is now set to 'Tallinn'\n  Update: registry is now set to 'Valetta'\n  Update: dispatch is now set to 'Zanzibar'\n  Update: reference is now set to 'Cartagena'\n  Update: coordinator is now set to 'Gdansk'\n  Update: destination is now set to 'Kumasi'\n  Update: location is now set to 'Mandalay'\n  Update: delegate is now set to 'Bruges'\n  Update: contact is now set to 'Recife'\n  Update: registry is now set to 'Fez'\n  Update: dispatch is now set to 'Tbilisi'\n  Update: reference is now set to 'Tallinn'\n  Update: coordinator is now set to 'Kotor'\n  Update: destination is now set to 'Recife'\n  Update: location is now set to 'Bruges'\n  Update: delegate is now set to 'Tallinn'\n  Update: contact is now set to 'Zanzibar'\n  Update: registry is now set to 'Recife'\n  Update: dispatch is now set to 'Tallinn'\n  Update: reference is now set to 'Cusco'\n  Update: coordinator is now set to 'Valetta'\n  Update: destination is now set to 'Trieste'\n  Update: location is now set to 'Kotor'\n  Update: delegate is now set to 'Cartagena'\n  Update: contact is now set to 'Valetta'\n  Update: registry is now set to 'Mandalay'\n  Update: dispatch is now set to 'Recife'\n  Update: reference is now set to 'Oulu'\n  Update: coordinator is now set to 'Bruges'\n  Update: destination is now set to 'Cartagena'\n  Update: location is now set to 'Recife'\n  Update: delegate is now set to 'Cusco'\n  Update: contact is now set to 'Cartagena'\n  Update: registry is now set to 'Jaipur'\n  Update: dispatch is now set to 'Bruges'\n  Update: reference is now set to 'Kotor'\n  Update: coordinator is now set to 'Ulaanbaatar'\n  Update: destination is now set to 'Zanzibar'\n  Update: location is now set to 'Cusco'\n  Update: delegate is now set to 'Tallinn'\n  Update: contact is now set to 'Reykjavik'\n  Update: registry is now set to 'Cartagena'\n  Update: dispatch is now set to 'Tallinn'\n  Update: reference is now set to 'Valetta'\n  Update: coordinator is now set to 'Jaipur'\n  Update: destination is now set to 'Tallinn'\n  Update: location is now set to 'Kotor'\n  Update: delegate is now set to 'Zanzibar'\n  Update: contact is now set to 'Gdansk'\n  Update: registry is now set to 'Cusco'\n  Update: dispatch is now set to 'Ulaanbaatar'\n  Update: reference is now set to 'Oulu'\n  Update: coordinator is now set to 'Cusco'\n  Update: destination is now set to 'Plovdiv'\n  Update: location is now set to 'Gdansk'\n  Update: delegate is now set to 'Oulu'\n  Update: contact is now set to 'Reykjavik'\n  Update: registry is now set to 'Ulaanbaatar'\n  Update: dispatch is now set to 'Tallinn'\n  Update: reference is now set to 'Kotor'\n  Update: coordinator is now set to 'Kumasi'\n  Update: destination is now set to 'Cusco'\n  Update: location is now set to 'Reykjavik'\n  Update: delegate is now set to 'Trieste'\n  Update: contact is now set to 'Ulaanbaatar'\n  Update: registry is now set to 'Oulu'\n  Update: dispatch is now set to 'Valetta'\n  Update: reference is now set to 'Tbilisi'\n  Update: coordinator is now set to 'Mandalay'\n  Update: destination is now set to 'Tbilisi'\n  Update: location is now set to 'Tbilisi'\n  Update: delegate is now set to 'Ulaanbaatar'\n  Update: contact is now set to 'Mandalay'\n  Update: registry is now set to 'Recife'\n  Update: dispatch is now set to 'Tallinn'\n  Update: reference is now set to 'Valetta'\n  Update: coordinator is now set to 'Tallinn'\n  Update: destination is now set to 'Valetta'\n  Update: location is now set to 'Kotor'\n  Update: delegate is now set to 'Fez'\n  Update: contact is now set to 'Bruges'\n  Update: registry is now set to 'Valetta'\n  Update: dispatch is now set to 'Trieste'\n  Update: reference is now set to 'Gdansk'\n  Update: coordinator is now set to 'Trieste'\n  Update: destination is now set to 'Luang Prabang'\n  Update: location is now set to 'Ulaanbaatar'\n  Update: delegate is now set to 'Tallinn'\n  Update: contact is now set to 'Trieste'\n  Update: registry is now set to 'Zanzibar'\n  Update: dispatch is now set to 'Gdansk'\n  Update: reference is now set to 'Mandalay'\n  Update: coordinator is now set to 'Cusco'\n  Update: destination is now set to 'Zanzibar'\n  Update: location is now set to 'Luang Prabang'\n  Update: delegate is now set to 'Jaipur'\n  Update: contact is now set to 'Valetta'\n  Update: registry is now set to 'Mandalay'\n  Update: dispatch is now set to 'Tbilisi'\n  Update: reference is now set to 'Cusco'\n  Update: coordinator is now set to 'Cartagena'\n  Update: destination is now set to 'Cusco'\n  Update: location is now set to 'Plovdiv'\n  Update: delegate is now set to 'Kumasi'\n  Update: contact is now set to 'Zanzibar'\n  Update: registry is now set to 'Valetta'\n  Update: dispatch is now set to 'Luang Prabang'\n  Update: reference is now set to 'Jaipur'\n  Update: coordinator is now set to 'Oulu'\n  Update: destination is now set to 'Oulu'\n  Update: location is now set to 'Mandalay'\n  Update: delegate is now set to 'Gdansk'\n  Update: contact is now set to 'Gdansk'\n  Update: registry is now set to 'Ulaanbaatar'\n  Update: dispatch is now set to 'Kotor'\n  Update: reference is now set to 'Tbilisi'\n  Update: coordinator is now set to 'Luang Prabang'\n  Update: destination is now set to 'Cartagena'\n  Update: location is now set to 'Gdansk'\n  Update: delegate is now set to 'Mandalay'\n  Update: contact is now set to 'Trieste'\n  Update: registry is now set to 'Oulu'\n  Update: dispatch is now set to 'Recife'\n  Update: reference is now set to 'Valetta'\n  Update: coordinator is now set to 'Plovdiv'\n  Update: destination is now set to 'Plovdiv'\n  Update: location is now set to 'Zanzibar'\n  Update: delegate is now set to 'Cusco'\n  Update: contact is now set to 'Kumasi'\n  Update: registry is now set to 'Gdansk'\n  Update: dispatch is now set to 'Kumasi'\n  Update: reference is now set to 'Tallinn'\n  Update: coordinator is now set to 'Gdansk'\n  Update: destination is now set to 'Tbilisi'\n  Update: location is now set to 'Tallinn'\n  Update: delegate is now set to 'Luang Prabang'\n  Update: contact is now set to 'Valetta'\n  Update: registry is now set to 'Valetta'\n  Update: dispatch is now set to 'Zanzibar'\n  Update: reference is now set to 'Kotor'\n  Update: coordinator is now set to 'Zanzibar'\n  Update: destination is now set to 'Ulaanbaatar'\n  Update: location is now set to 'Kumasi'\n  Update: delegate is now set to 'Mandalay'\n  Update: contact is now set to 'Trieste'\n  Update: registry is now set to 'Kumasi'\n  Update: dispatch is now set to 'Gdansk'\n  Update: reference is now set to 'Jaipur'\n  Update: coordinator is now set to 'Cusco'\n  Update: destination is now set to 'Mandalay'\n  Update: location is now set to 'Tbilisi'\n  Update: delegate is now set to 'Fez'\n  Update: contact is now set to 'Kotor'\n\nWhat is the FINAL value of each record?\nANSWER:\n- registry: [final value]\n- dispatch: [final value]\n- reference: [final value]\n- coordinator: [final value]\n- destination: [final value]\n- location: [final value]\n- delegate: [final value]\n- contact: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was 'Cusco' ever assigned to registry? [Yes/No]\nV2. Was 'Tbilisi' ever assigned to dispatch? [Yes/No]\nV3. Was 'Oulu' ever assigned to reference? [Yes/No]\nV4. Was 'Valetta' ever assigned to coordinator? [Yes/No]",
  "gold_json": "{\"final_values\": {\"registry\": \"Kumasi\", \"dispatch\": \"Gdansk\", \"reference\": \"Jaipur\", \"coordinator\": \"Cusco\", \"destination\": \"Mandalay\", \"location\": \"Tbilisi\", \"delegate\": \"Fez\", \"contact\": \"Kotor\"}, \"key_names\": [\"registry\", \"dispatch\", \"reference\", \"coordinator\", \"destination\", \"location\", \"delegate\", \"contact\"]}"
 },
 {
  "task_id": "interference_frontier_034",
  "task_type": "interference",
  "difficulty": "Frontier",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: coordinator is now set to '498'\n  Update: liaison is now set to '737'\n  Update: location is now set to '205'\n  Update: dispatch is now set to '927'\n  Update: reference is now set to '212'\n  Update: delegate is now set to '571'\n  Update: contact is now set to '473'\n  Update: assignment is now set to '114'\n  Update: coordinator is now set to '810'\n  Update: liaison is now set to '548'\n  Update: location is now set to '562'\n  Update: dispatch is now set to '801'\n  Update: reference is now set to '330'\n  Update: delegate is now set to '343'\n  Update: contact is now set to '686'\n  Update: assignment is now set to '862'\n  Update: coordinator is now set to '551'\n  Update: liaison is now set to '931'\n  Update: location is now set to '792'\n  Update: dispatch is now set to '682'\n  Update: reference is now set to '396'\n  Update: delegate is now set to '665'\n  Update: contact is now set to '269'\n  Update: assignment is now set to '658'\n  Update: coordinator is now set to '366'\n  Update: liaison is now set to '630'\n  Update: location is now set to '585'\n  Update: dispatch is now set to '782'\n  Update: reference is now set to '375'\n  Update: delegate is now set to '555'\n  Update: contact is now set to '703'\n  Update: assignment is now set to '895'\n  Update: coordinator is now set to '327'\n  Update: liaison is now set to '465'\n  Update: location is now set to '446'\n  Update: dispatch is now set to '934'\n  Update: reference is now set to '304'\n  Update: delegate is now set to '240'\n  Update: contact is now set to '295'\n  Update: assignment is now set to '806'\n  Update: coordinator is now set to '189'\n  Update: liaison is now set to '954'\n  Update: location is now set to '927'\n  Update: dispatch is now set to '586'\n  Update: reference is now set to '561'\n  Update: delegate is now set to '536'\n  Update: contact is now set to '823'\n  Update: assignment is now set to '501'\n  Update: coordinator is now set to '510'\n  Update: liaison is now set to '885'\n  Update: location is now set to '426'\n  Update: dispatch is now set to '873'\n  Update: reference is now set to '844'\n  Update: delegate is now set to '646'\n  Update: contact is now set to '954'\n  Update: assignment is now set to '273'\n  Update: coordinator is now set to '171'\n  Update: liaison is now set to '866'\n  Update: location is now set to '876'\n  Update: dispatch is now set to '469'\n  Update: reference is now set to '131'\n  Update: delegate is now set to '512'\n  Update: contact is now set to '598'\n  Update: assignment is now set to '842'\n  Update: coordinator is now set to '158'\n  Update: liaison is now set to '923'\n  Update: location is now set to '591'\n  Update: dispatch is now set to '756'\n  Update: reference is now set to '461'\n  Update: delegate is now set to '968'\n  Update: contact is now set to '981'\n  Update: assignment is now set to '319'\n  Update: coordinator is now set to '880'\n  Update: liaison is now set to '488'\n  Update: location is now set to '666'\n  Update: dispatch is now set to '713'\n  Update: reference is now set to '830'\n  Update: delegate is now set to '491'\n  Update: contact is now set to '507'\n  Update: assignment is now set to '536'\n  Update: coordinator is now set to '304'\n  Update: liaison is now set to '248'\n  Update: location is now set to '499'\n  Update: dispatch is now set to '448'\n  Update: reference is now set to '736'\n  Update: delegate is now set to '647'\n  Update: contact is now set to '683'\n  Update: assignment is now set to '718'\n  Update: coordinator is now set to '234'\n  Update: liaison is now set to '394'\n  Update: location is now set to '697'\n  Update: dispatch is now set to '795'\n  Update: reference is now set to '944'\n  Update: delegate is now set to '546'\n  Update: contact is now set to '346'\n  Update: assignment is now set to '550'\n  Update: coordinator is now set to '364'\n  Update: liaison is now set to '927'\n  Update: location is now set to '474'\n  Update: dispatch is now set to '359'\n  Update: reference is now set to '828'\n  Update: delegate is now set to '511'\n  Update: contact is now set to '957'\n  Update: assignment is now set to '839'\n  Update: coordinator is now set to '689'\n  Update: liaison is now set to '579'\n  Update: location is now set to '173'\n  Update: dispatch is now set to '391'\n  Update: reference is now set to '673'\n  Update: delegate is now set to '799'\n  Update: contact is now set to '679'\n  Update: assignment is now set to '927'\n  Update: coordinator is now set to '996'\n  Update: liaison is now set to '393'\n  Update: location is now set to '129'\n  Update: dispatch is now set to '840'\n  Update: reference is now set to '615'\n  Update: delegate is now set to '685'\n  Update: contact is now set to '366'\n  Update: assignment is now set to '279'\n  Update: coordinator is now set to '192'\n  Update: liaison is now set to '325'\n  Update: location is now set to '997'\n  Update: dispatch is now set to '868'\n  Update: reference is now set to '463'\n  Update: delegate is now set to '820'\n  Update: contact is now set to '817'\n  Update: assignment is now set to '739'\n  Update: coordinator is now set to '124'\n  Update: liaison is now set to '848'\n  Update: location is now set to '824'\n  Update: dispatch is now set to '545'\n  Update: reference is now set to '319'\n  Update: delegate is now set to '927'\n  Update: contact is now set to '563'\n  Update: assignment is now set to '794'\n  Update: coordinator is now set to '479'\n  Update: liaison is now set to '876'\n  Update: location is now set to '133'\n  Update: dispatch is now set to '812'\n  Update: reference is now set to '721'\n  Update: delegate is now set to '713'\n  Update: contact is now set to '804'\n  Update: assignment is now set to '228'\n  Update: coordinator is now set to '188'\n  Update: liaison is now set to '623'\n  Update: location is now set to '444'\n  Update: dispatch is now set to '986'\n  Update: reference is now set to '200'\n  Update: delegate is now set to '790'\n  Update: contact is now set to '523'\n  Update: assignment is now set to '237'\n  Update: coordinator is now set to '412'\n  Update: liaison is now set to '395'\n  Update: location is now set to '448'\n  Update: dispatch is now set to '472'\n  Update: reference is now set to '545'\n  Update: delegate is now set to '146'\n  Update: contact is now set to '422'\n  Update: assignment is now set to '356'\n  Update: coordinator is now set to '283'\n  Update: liaison is now set to '730'\n  Update: location is now set to '975'\n  Update: dispatch is now set to '405'\n  Update: reference is now set to '736'\n  Update: delegate is now set to '483'\n  Update: contact is now set to '810'\n  Update: assignment is now set to '551'\n  Update: coordinator is now set to '694'\n  Update: liaison is now set to '131'\n  Update: location is now set to '995'\n  Update: dispatch is now set to '441'\n  Update: reference is now set to '314'\n  Update: delegate is now set to '963'\n  Update: contact is now set to '712'\n  Update: assignment is now set to '669'\n  Update: coordinator is now set to '149'\n  Update: liaison is now set to '303'\n  Update: location is now set to '419'\n  Update: dispatch is now set to '484'\n  Update: reference is now set to '813'\n  Update: delegate is now set to '488'\n  Update: contact is now set to '517'\n  Update: assignment is now set to '129'\n  Update: coordinator is now set to '316'\n  Update: liaison is now set to '478'\n  Update: location is now set to '942'\n  Update: dispatch is now set to '895'\n  Update: reference is now set to '695'\n  Update: delegate is now set to '523'\n  Update: contact is now set to '126'\n  Update: assignment is now set to '568'\n  Update: coordinator is now set to '105'\n  Update: liaison is now set to '684'\n  Update: location is now set to '227'\n  Update: dispatch is now set to '685'\n  Update: reference is now set to '688'\n  Update: delegate is now set to '957'\n  Update: contact is now set to '322'\n  Update: assignment is now set to '394'\n  Update: coordinator is now set to '759'\n  Update: liaison is now set to '320'\n  Update: location is now set to '431'\n  Update: dispatch is now set to '663'\n  Update: reference is now set to '527'\n  Update: delegate is now set to '964'\n  Update: contact is now set to '485'\n  Update: assignment is now set to '148'\n  Update: coordinator is now set to '574'\n  Update: liaison is now set to '952'\n  Update: location is now set to '207'\n  Update: dispatch is now set to '994'\n  Update: reference is now set to '542'\n  Update: delegate is now set to '555'\n  Update: contact is now set to '105'\n  Update: assignment is now set to '490'\n  Update: coordinator is now set to '629'\n  Update: liaison is now set to '617'\n  Update: location is now set to '843'\n  Update: dispatch is now set to '490'\n  Update: reference is now set to '759'\n  Update: delegate is now set to '536'\n  Update: contact is now set to '214'\n  Update: assignment is now set to '665'\n  Update: coordinator is now set to '280'\n  Update: liaison is now set to '717'\n  Update: location is now set to '488'\n  Update: dispatch is now set to '600'\n  Update: reference is now set to '961'\n  Update: delegate is now set to '508'\n  Update: contact is now set to '511'\n  Update: assignment is now set to '647'\n  Update: coordinator is now set to '372'\n  Update: liaison is now set to '572'\n  Update: location is now set to '973'\n  Update: dispatch is now set to '396'\n  Update: reference is now set to '230'\n  Update: delegate is now set to '630'\n  Update: contact is now set to '302'\n  Update: assignment is now set to '673'\n  Update: coordinator is now set to '125'\n  Update: liaison is now set to '503'\n  Update: location is now set to '552'\n  Update: dispatch is now set to '376'\n  Update: reference is now set to '157'\n  Update: delegate is now set to '974'\n  Update: contact is now set to '767'\n  Update: assignment is now set to '861'\n  Update: coordinator is now set to '370'\n  Update: liaison is now set to '247'\n  Update: location is now set to '924'\n  Update: dispatch is now set to '935'\n  Update: reference is now set to '418'\n  Update: delegate is now set to '226'\n  Update: contact is now set to '925'\n  Update: assignment is now set to '260'\n  Update: coordinator is now set to '712'\n  Update: liaison is now set to '382'\n  Update: location is now set to '818'\n  Update: dispatch is now set to '832'\n  Update: reference is now set to '864'\n  Update: delegate is now set to '269'\n  Update: contact is now set to '618'\n  Update: assignment is now set to '442'\n  Update: coordinator is now set to '556'\n  Update: liaison is now set to '269'\n  Update: location is now set to '584'\n  Update: dispatch is now set to '162'\n  Update: reference is now set to '459'\n  Update: delegate is now set to '398'\n  Update: contact is now set to '458'\n  Update: assignment is now set to '856'\n  Update: coordinator is now set to '480'\n  Update: liaison is now set to '564'\n  Update: location is now set to '154'\n  Update: dispatch is now set to '151'\n  Update: reference is now set to '327'\n  Update: delegate is now set to '960'\n  Update: contact is now set to '568'\n  Update: assignment is now set to '273'\n  Update: coordinator is now set to '741'\n  Update: liaison is now set to '991'\n  Update: location is now set to '368'\n  Update: dispatch is now set to '288'\n  Update: reference is now set to '630'\n  Update: delegate is now set to '457'\n  Update: contact is now set to '292'\n  Update: assignment is now set to '661'\n  Update: coordinator is now set to '460'\n  Update: liaison is now set to '271'\n  Update: location is now set to '583'\n  Update: dispatch is now set to '909'\n  Update: reference is now set to '537'\n  Update: delegate is now set to '644'\n  Update: contact is now set to '768'\n  Update: assignment is now set to '677'\n  Update: coordinator is now set to '186'\n  Update: liaison is now set to '154'\n  Update: location is now set to '220'\n  Update: dispatch is now set to '145'\n  Update: reference is now set to '870'\n  Update: delegate is now set to '114'\n  Update: contact is now set to '279'\n  Update: assignment is now set to '898'\n  Update: coordinator is now set to '756'\n  Update: liaison is now set to '710'\n  Update: location is now set to '391'\n  Update: dispatch is now set to '125'\n  Update: reference is now set to '451'\n  Update: delegate is now set to '837'\n  Update: contact is now set to '229'\n  Update: assignment is now set to '569'\n  Update: coordinator is now set to '735'\n  Update: liaison is now set to '995'\n  Update: location is now set to '696'\n  Update: dispatch is now set to '871'\n  Update: reference is now set to '741'\n  Update: delegate is now set to '452'\n  Update: contact is now set to '412'\n  Update: assignment is now set to '729'\n  Update: coordinator is now set to '109'\n  Update: liaison is now set to '358'\n  Update: location is now set to '258'\n  Update: dispatch is now set to '492'\n  Update: reference is now set to '236'\n  Update: delegate is now set to '939'\n  Update: contact is now set to '849'\n  Update: assignment is now set to '511'\n  Update: coordinator is now set to '255'\n  Update: liaison is now set to '827'\n  Update: location is now set to '350'\n  Update: dispatch is now set to '862'\n  Update: reference is now set to '500'\n  Update: delegate is now set to '697'\n  Update: contact is now set to '293'\n  Update: assignment is now set to '548'\n  Update: coordinator is now set to '829'\n  Update: liaison is now set to '652'\n  Update: location is now set to '990'\n  Update: dispatch is now set to '844'\n  Update: reference is now set to '848'\n  Update: delegate is now set to '174'\n  Update: contact is now set to '696'\n  Update: assignment is now set to '996'\n  Update: coordinator is now set to '202'\n  Update: liaison is now set to '233'\n  Update: location is now set to '379'\n  Update: dispatch is now set to '376'\n  Update: reference is now set to '799'\n  Update: delegate is now set to '249'\n  Update: contact is now set to '259'\n  Update: assignment is now set to '782'\n  Update: coordinator is now set to '672'\n  Update: liaison is now set to '703'\n  Update: location is now set to '195'\n  Update: dispatch is now set to '540'\n  Update: reference is now set to '439'\n  Update: delegate is now set to '973'\n  Update: contact is now set to '121'\n  Update: assignment is now set to '137'\n  Update: coordinator is now set to '519'\n  Update: liaison is now set to '736'\n  Update: location is now set to '303'\n  Update: dispatch is now set to '542'\n  Update: reference is now set to '183'\n  Update: delegate is now set to '949'\n  Update: contact is now set to '372'\n  Update: assignment is now set to '481'\n  Update: coordinator is now set to '821'\n  Update: liaison is now set to '846'\n  Update: location is now set to '684'\n  Update: dispatch is now set to '645'\n  Update: reference is now set to '437'\n  Update: delegate is now set to '504'\n  Update: contact is now set to '200'\n  Update: assignment is now set to '848'\n  Update: coordinator is now set to '232'\n  Update: liaison is now set to '336'\n  Update: location is now set to '734'\n  Update: dispatch is now set to '310'\n  Update: reference is now set to '740'\n  Update: delegate is now set to '796'\n  Update: contact is now set to '391'\n  Update: assignment is now set to '772'\n  Update: coordinator is now set to '827'\n  Update: liaison is now set to '977'\n  Update: location is now set to '360'\n  Update: dispatch is now set to '274'\n  Update: reference is now set to '171'\n  Update: delegate is now set to '890'\n  Update: contact is now set to '480'\n  Update: assignment is now set to '379'\n  Update: coordinator is now set to '643'\n  Update: liaison is now set to '806'\n  Update: location is now set to '599'\n  Update: dispatch is now set to '638'\n  Update: reference is now set to '186'\n  Update: delegate is now set to '700'\n  Update: contact is now set to '847'\n  Update: assignment is now set to '189'\n\nWhat is the FINAL value of each record?\nANSWER:\n- coordinator: [final value]\n- liaison: [final value]\n- location: [final value]\n- dispatch: [final value]\n- reference: [final value]\n- delegate: [final value]\n- contact: [final value]\n- assignment: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was '186' ever assigned to coordinator? [Yes/No]\nV2. Was '154' ever assigned to liaison? [Yes/No]\nV3. Was '666' ever assigned to location? [Yes/No]\nV4. Was '713' ever assigned to dispatch? [Yes/No]",
  "gold_json": "{\"final_values\": {\"coordinator\": \"643\", \"liaison\": \"806\", \"location\": \"599\", \"dispatch\": \"638\", \"reference\": \"186\", \"delegate\": \"700\", \"contact\": \"847\", \"assignment\": \"189\"}, \"key_names\": [\"coordinator\", \"liaison\", \"location\", \"dispatch\", \"reference\", \"delegate\", \"contact\", \"assignment\"]}"
 },
 {
  "task_id": "interference_frontier_035",
  "task_type": "interference",
  "difficulty": "Frontier",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: assignment is now set to 'Joaquin'\n  Update: location is now set to 'Orla'\n  Update: destination is now set to 'Freya'\n  Update: reference is now set to 'Ugo'\n  Update: dispatch is now set to 'Yara'\n  Update: contact is now set to 'Elara'\n  Update: registry is now set to 'Uma'\n  Update: liaison is now set to 'Paloma'\n  Update: assignment is now set to 'Willa'\n  Update: location is now set to 'Magnus'\n  Update: destination is now set to 'Celine'\n  Update: reference is now set to 'Wren'\n  Update: dispatch is now set to 'Amara'\n  Update: contact is now set to 'Willa'\n  Update: registry is now set to 'Ravi'\n  Update: liaison is now set to 'Sigrid'\n  Update: assignment is now set to 'Orla'\n  Update: location is now set to 'Bram'\n  Update: destination is now set to 'Kenji'\n  Update: reference is now set to 'Joaquin'\n  Update: dispatch is now set to 'Kaia'\n  Update: contact is now set to 'Elara'\n  Update: registry is now set to 'Xander'\n  Update: liaison is now set to 'Leif'\n  Update: assignment is now set to 'Maren'\n  Update: location is now set to 'Uma'\n  Update: destination is now set to 'Kaia'\n  Update: reference is now set to 'Lumi'\n  Update: dispatch is now set to 'Zain'\n  Update: contact is now set to 'Bashir'\n  Update: registry is now set to 'Greta'\n  Update: liaison is now set to 'Ugo'\n  Update: assignment is now set to 'Dmitri'\n  Update: location is now set to 'Orla'\n  Update: destination is now set to 'Gael'\n  Update: reference is now set to 'Magnus'\n  Update: dispatch is now set to 'Kaia'\n  Update: contact is now set to 'Xander'\n  Update: registry is now set to 'Maren'\n  Update: liaison is now set to 'Paloma'\n  Update: assignment is now set to 'Yara'\n  Update: location is now set to 'Tala'\n  Update: destination is now set to 'Nico'\n  Update: reference is now set to 'Bram'\n  Update: dispatch is now set to 'Orla'\n  Update: contact is now set to 'Soren'\n  Update: registry is now set to 'Ines'\n  Update: liaison is now set to 'Kaia'\n  Update: assignment is now set to 'Orla'\n  Update: location is now set to 'Qadir'\n  Update: destination is now set to 'Ines'\n  Update: reference is now set to 'Willa'\n  Update: dispatch is now set to 'Ines'\n  Update: contact is now set to 'Priya'\n  Update: registry is now set to 'Tariq'\n  Update: liaison is now set to 'Wren'\n  Update: assignment is now set to 'Sigrid'\n  Update: location is now set to 'Viktor'\n  Update: destination is now set to 'Haruto'\n  Update: reference is now set to 'Idris'\n  Update: dispatch is now set to 'Zora'\n  Update: contact is now set to 'Elio'\n  Update: registry is now set to 'Viktor'\n  Update: liaison is now set to 'Kenji'\n  Update: assignment is now set to 'Bram'\n  Update: location is now set to 'Vesna'\n  Update: destination is now set to 'Willa'\n  Update: reference is now set to 'Yara'\n  Update: dispatch is now set to 'Qadir'\n  Update: contact is now set to 'Joaquin'\n  Update: registry is now set to 'Maren'\n  Update: liaison is now set to 'Magnus'\n  Update: assignment is now set to 'Kenji'\n  Update: location is now set to 'Paloma'\n  Update: destination is now set to 'Dmitri'\n  Update: reference is now set to 'Tariq'\n  Update: dispatch is now set to 'Gael'\n  Update: contact is now set to 'Vesna'\n  Update: registry is now set to 'Hana'\n  Update: liaison is now set to 'Nico'\n  Update: assignment is now set to 'Nico'\n  Update: location is now set to 'Xander'\n  Update: destination is now set to 'Joelle'\n  Update: reference is now set to 'Leif'\n  Update: dispatch is now set to 'Magnus'\n  Update: contact is now set to 'Dariush'\n  Update: registry is now set to 'Amara'\n  Update: liaison is now set to 'Freya'\n  Update: assignment is now set to 'Kenji'\n  Update: location is now set to 'Hana'\n  Update: destination is now set to 'Femi'\n  Update: reference is now set to 'Lumi'\n  Update: dispatch is now set to 'Zain'\n  Update: contact is now set to 'Viktor'\n  Update: registry is now set to 'Paloma'\n  Update: liaison is now set to 'Bram'\n  Update: assignment is now set to 'Magnus'\n  Update: location is now set to 'Femi'\n  Update: destination is now set to 'Lumi'\n  Update: reference is now set to 'Zora'\n  Update: dispatch is now set to 'Vesna'\n  Update: contact is now set to 'Qadir'\n  Update: registry is now set to 'Ugo'\n  Update: liaison is now set to 'Colette'\n  Update: assignment is now set to 'Nico'\n  Update: location is now set to 'Kenji'\n  Update: destination is now set to 'Elara'\n  Update: reference is now set to 'Magnus'\n  Update: dispatch is now set to 'Dmitri'\n  Update: contact is now set to 'Soren'\n  Update: registry is now set to 'Nico'\n  Update: liaison is now set to 'Hana'\n  Update: assignment is now set to 'Sigrid'\n  Update: location is now set to 'Joelle'\n  Update: destination is now set to 'Joaquin'\n  Update: reference is now set to 'Colette'\n  Update: dispatch is now set to 'Zora'\n  Update: contact is now set to 'Yuki'\n  Update: registry is now set to 'Dariush'\n  Update: liaison is now set to 'Orla'\n  Update: assignment is now set to 'Paloma'\n  Update: location is now set to 'Paloma'\n  Update: destination is now set to 'Viktor'\n  Update: reference is now set to 'Kenji'\n  Update: dispatch is now set to 'Freya'\n  Update: contact is now set to 'Greta'\n  Update: registry is now set to 'Runa'\n  Update: liaison is now set to 'Celine'\n  Update: assignment is now set to 'Hana'\n  Update: location is now set to 'Magnus'\n  Update: destination is now set to 'Bashir'\n  Update: reference is now set to 'Tala'\n  Update: dispatch is now set to 'Bashir'\n  Update: contact is now set to 'Viktor'\n  Update: registry is now set to 'Yara'\n  Update: liaison is now set to 'Joelle'\n  Update: assignment is now set to 'Paloma'\n  Update: location is now set to 'Femi'\n  Update: destination is now set to 'Celine'\n  Update: reference is now set to 'Dmitri'\n  Update: dispatch is now set to 'Celine'\n  Update: contact is now set to 'Kaia'\n  Update: registry is now set to 'Sigrid'\n  Update: liaison is now set to 'Ugo'\n  Update: assignment is now set to 'Joelle'\n  Update: location is now set to 'Haruto'\n  Update: destination is now set to 'Idris'\n  Update: reference is now set to 'Adaeze'\n  Update: dispatch is now set to 'Haruto'\n  Update: contact is now set to 'Hana'\n  Update: registry is now set to 'Hana'\n  Update: liaison is now set to 'Xander'\n  Update: assignment is now set to 'Dariush'\n  Update: location is now set to 'Bram'\n  Update: destination is now set to 'Kaia'\n  Update: reference is now set to 'Nico'\n  Update: dispatch is now set to 'Kaia'\n  Update: contact is now set to 'Runa'\n  Update: registry is now set to 'Maren'\n  Update: liaison is now set to 'Gael'\n  Update: assignment is now set to 'Ines'\n  Update: location is now set to 'Leif'\n  Update: destination is now set to 'Joelle'\n  Update: reference is now set to 'Ravi'\n  Update: dispatch is now set to 'Sigrid'\n  Update: contact is now set to 'Yara'\n  Update: registry is now set to 'Qadir'\n  Update: liaison is now set to 'Hana'\n  Update: assignment is now set to 'Lumi'\n  Update: location is now set to 'Uma'\n  Update: destination is now set to 'Bram'\n  Update: reference is now set to 'Gael'\n  Update: dispatch is now set to 'Amara'\n  Update: contact is now set to 'Zora'\n  Update: registry is now set to 'Yara'\n  Update: liaison is now set to 'Maren'\n  Update: assignment is now set to 'Ravi'\n  Update: location is now set to 'Joaquin'\n  Update: destination is now set to 'Magnus'\n  Update: reference is now set to 'Amara'\n  Update: dispatch is now set to 'Elara'\n  Update: contact is now set to 'Yara'\n  Update: registry is now set to 'Tariq'\n  Update: liaison is now set to 'Wren'\n  Update: assignment is now set to 'Colette'\n  Update: location is now set to 'Willa'\n  Update: destination is now set to 'Celine'\n  Update: reference is now set to 'Dmitri'\n  Update: dispatch is now set to 'Tala'\n  Update: contact is now set to 'Uma'\n  Update: registry is now set to 'Xander'\n  Update: liaison is now set to 'Zain'\n  Update: assignment is now set to 'Ravi'\n  Update: location is now set to 'Zora'\n  Update: destination is now set to 'Priya'\n  Update: reference is now set to 'Wren'\n  Update: dispatch is now set to 'Orla'\n  Update: contact is now set to 'Elio'\n  Update: registry is now set to 'Tala'\n  Update: liaison is now set to 'Ravi'\n  Update: assignment is now set to 'Kenji'\n  Update: location is now set to 'Dariush'\n  Update: destination is now set to 'Tariq'\n  Update: reference is now set to 'Ravi'\n  Update: dispatch is now set to 'Elio'\n  Update: contact is now set to 'Amara'\n  Update: registry is now set to 'Bashir'\n  Update: liaison is now set to 'Dariush'\n  Update: assignment is now set to 'Ines'\n  Update: location is now set to 'Amara'\n  Update: destination is now set to 'Orla'\n  Update: reference is now set to 'Wren'\n  Update: dispatch is now set to 'Celine'\n  Update: contact is now set to 'Kaia'\n  Update: registry is now set to 'Dariush'\n  Update: liaison is now set to 'Ines'\n  Update: assignment is now set to 'Willa'\n  Update: location is now set to 'Maren'\n  Update: destination is now set to 'Qadir'\n  Update: reference is now set to 'Joelle'\n  Update: dispatch is now set to 'Kaia'\n  Update: contact is now set to 'Tala'\n  Update: registry is now set to 'Tala'\n  Update: liaison is now set to 'Kaia'\n  Update: assignment is now set to 'Dmitri'\n  Update: location is now set to 'Nalini'\n  Update: destination is now set to 'Nico'\n  Update: reference is now set to 'Sigrid'\n  Update: dispatch is now set to 'Elio'\n  Update: contact is now set to 'Hana'\n  Update: registry is now set to 'Hana'\n  Update: liaison is now set to 'Joaquin'\n  Update: assignment is now set to 'Dariush'\n  Update: location is now set to 'Tariq'\n  Update: destination is now set to 'Kenji'\n  Update: reference is now set to 'Ravi'\n  Update: dispatch is now set to 'Priya'\n  Update: contact is now set to 'Dmitri'\n  Update: registry is now set to 'Willa'\n  Update: liaison is now set to 'Ravi'\n  Update: assignment is now set to 'Bashir'\n  Update: location is now set to 'Lumi'\n  Update: destination is now set to 'Colette'\n  Update: reference is now set to 'Bashir'\n  Update: dispatch is now set to 'Leif'\n  Update: contact is now set to 'Yara'\n  Update: registry is now set to 'Sigrid'\n  Update: liaison is now set to 'Bram'\n  Update: assignment is now set to 'Femi'\n  Update: location is now set to 'Zora'\n  Update: destination is now set to 'Elio'\n  Update: reference is now set to 'Lumi'\n  Update: dispatch is now set to 'Joelle'\n  Update: contact is now set to 'Ravi'\n  Update: registry is now set to 'Willa'\n  Update: liaison is now set to 'Gael'\n  Update: assignment is now set to 'Soren'\n  Update: location is now set to 'Kenji'\n  Update: destination is now set to 'Wren'\n  Update: reference is now set to 'Olena'\n  Update: dispatch is now set to 'Bram'\n  Update: contact is now set to 'Elara'\n  Update: registry is now set to 'Leif'\n  Update: liaison is now set to 'Magnus'\n  Update: assignment is now set to 'Gael'\n  Update: location is now set to 'Qadir'\n  Update: destination is now set to 'Priya'\n  Update: reference is now set to 'Ines'\n  Update: dispatch is now set to 'Joaquin'\n  Update: contact is now set to 'Yuki'\n  Update: registry is now set to 'Bashir'\n  Update: liaison is now set to 'Zain'\n  Update: assignment is now set to 'Freya'\n  Update: location is now set to 'Idris'\n  Update: destination is now set to 'Elio'\n  Update: reference is now set to 'Ugo'\n  Update: dispatch is now set to 'Joelle'\n  Update: contact is now set to 'Willa'\n  Update: registry is now set to 'Elio'\n  Update: liaison is now set to 'Yuki'\n  Update: assignment is now set to 'Paloma'\n  Update: location is now set to 'Elio'\n  Update: destination is now set to 'Magnus'\n  Update: reference is now set to 'Tala'\n  Update: dispatch is now set to 'Runa'\n  Update: contact is now set to 'Dmitri'\n  Update: registry is now set to 'Uma'\n  Update: liaison is now set to 'Yara'\n  Update: assignment is now set to 'Idris'\n  Update: location is now set to 'Bram'\n  Update: destination is now set to 'Orla'\n  Update: reference is now set to 'Paloma'\n  Update: dispatch is now set to 'Dariush'\n  Update: contact is now set to 'Kaia'\n  Update: registry is now set to 'Priya'\n  Update: liaison is now set to 'Lumi'\n  Update: assignment is now set to 'Uma'\n  Update: location is now set to 'Nico'\n  Update: destination is now set to 'Sigrid'\n  Update: reference is now set to 'Leif'\n  Update: dispatch is now set to 'Magnus'\n  Update: contact is now set to 'Qadir'\n  Update: registry is now set to 'Bram'\n  Update: liaison is now set to 'Leif'\n  Update: assignment is now set to 'Bashir'\n  Update: location is now set to 'Tala'\n  Update: destination is now set to 'Xander'\n  Update: reference is now set to 'Wren'\n  Update: dispatch is now set to 'Femi'\n  Update: contact is now set to 'Lumi'\n  Update: registry is now set to 'Uma'\n  Update: liaison is now set to 'Amara'\n  Update: assignment is now set to 'Ravi'\n  Update: location is now set to 'Zora'\n  Update: destination is now set to 'Nalini'\n  Update: reference is now set to 'Adaeze'\n  Update: dispatch is now set to 'Bram'\n  Update: contact is now set to 'Zain'\n  Update: registry is now set to 'Yara'\n  Update: liaison is now set to 'Ugo'\n  Update: assignment is now set to 'Wren'\n  Update: location is now set to 'Haruto'\n  Update: destination is now set to 'Kenji'\n  Update: reference is now set to 'Magnus'\n  Update: dispatch is now set to 'Joaquin'\n  Update: contact is now set to 'Tala'\n  Update: registry is now set to 'Greta'\n  Update: liaison is now set to 'Kaia'\n  Update: assignment is now set to 'Zain'\n  Update: location is now set to 'Dariush'\n  Update: destination is now set to 'Priya'\n  Update: reference is now set to 'Runa'\n  Update: dispatch is now set to 'Bram'\n  Update: contact is now set to 'Bashir'\n  Update: registry is now set to 'Femi'\n  Update: liaison is now set to 'Nalini'\n  Update: assignment is now set to 'Adaeze'\n  Update: location is now set to 'Gael'\n  Update: destination is now set to 'Yara'\n  Update: reference is now set to 'Wren'\n  Update: dispatch is now set to 'Paloma'\n  Update: contact is now set to 'Adaeze'\n  Update: registry is now set to 'Maren'\n  Update: liaison is now set to 'Uma'\n  Update: assignment is now set to 'Haruto'\n  Update: location is now set to 'Amara'\n  Update: destination is now set to 'Lumi'\n  Update: reference is now set to 'Kaia'\n  Update: dispatch is now set to 'Idris'\n  Update: contact is now set to 'Tala'\n  Update: registry is now set to 'Ravi'\n  Update: liaison is now set to 'Hana'\n  Update: assignment is now set to 'Amara'\n  Update: location is now set to 'Magnus'\n  Update: destination is now set to 'Ugo'\n  Update: reference is now set to 'Ines'\n  Update: dispatch is now set to 'Dariush'\n  Update: contact is now set to 'Zora'\n  Update: registry is now set to 'Olena'\n  Update: liaison is now set to 'Ines'\n  Update: assignment is now set to 'Idris'\n  Update: location is now set to 'Runa'\n  Update: destination is now set to 'Kenji'\n  Update: reference is now set to 'Elio'\n  Update: dispatch is now set to 'Vesna'\n  Update: contact is now set to 'Idris'\n  Update: registry is now set to 'Colette'\n  Update: liaison is now set to 'Amara'\n  Update: assignment is now set to 'Femi'\n  Update: location is now set to 'Vesna'\n  Update: destination is now set to 'Ravi'\n  Update: reference is now set to 'Tala'\n  Update: dispatch is now set to 'Kenji'\n  Update: contact is now set to 'Joaquin'\n  Update: registry is now set to 'Tariq'\n  Update: liaison is now set to 'Gael'\n  Update: assignment is now set to 'Ugo'\n  Update: location is now set to 'Freya'\n  Update: destination is now set to 'Kaia'\n  Update: reference is now set to 'Magnus'\n  Update: dispatch is now set to 'Vesna'\n  Update: contact is now set to 'Zain'\n  Update: registry is now set to 'Celine'\n  Update: liaison is now set to 'Zora'\n  Update: assignment is now set to 'Colette'\n  Update: location is now set to 'Sigrid'\n  Update: destination is now set to 'Magnus'\n  Update: reference is now set to 'Uma'\n  Update: dispatch is now set to 'Ugo'\n  Update: contact is now set to 'Yuki'\n  Update: registry is now set to 'Ugo'\n  Update: liaison is now set to 'Bram'\n  Update: assignment is now set to 'Nalini'\n  Update: location is now set to 'Gael'\n  Update: destination is now set to 'Tala'\n  Update: reference is now set to 'Priya'\n  Update: dispatch is now set to 'Uma'\n  Update: contact is now set to 'Celine'\n  Update: registry is now set to 'Nico'\n  Update: liaison is now set to 'Kenji'\n\nWhat is the FINAL value of each record?\nANSWER:\n- assignment: [final value]\n- location: [final value]\n- destination: [final value]\n- reference: [final value]\n- dispatch: [final value]\n- contact: [final value]\n- registry: [final value]\n- liaison: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was 'Paloma' ever assigned to assignment? [Yes/No]\nV2. Was 'Dariush' ever assigned to location? [Yes/No]\nV3. Was 'Dmitri' ever assigned to destination? [Yes/No]\nV4. Was 'Yara' ever assigned to reference? [Yes/No]",
  "gold_json": "{\"final_values\": {\"assignment\": \"Nalini\", \"location\": \"Gael\", \"destination\": \"Tala\", \"reference\": \"Priya\", \"dispatch\": \"Uma\", \"contact\": \"Celine\", \"registry\": \"Nico\", \"liaison\": \"Kenji\"}, \"key_names\": [\"assignment\", \"location\", \"destination\", \"reference\", \"dispatch\", \"contact\", \"registry\", \"liaison\"]}"
 },
 {
  "task_id": "interference_frontier_036",
  "task_type": "interference",
  "difficulty": "Frontier",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: liaison is now set to '373'\n  Update: contact is now set to '405'\n  Update: assignment is now set to '496'\n  Update: registry is now set to '714'\n  Update: delegate is now set to '536'\n  Update: reference is now set to '959'\n  Update: dispatch is now set to '765'\n  Update: location is now set to '495'\n  Update: liaison is now set to '642'\n  Update: contact is now set to '881'\n  Update: assignment is now set to '572'\n  Update: registry is now set to '934'\n  Update: delegate is now set to '103'\n  Update: reference is now set to '505'\n  Update: dispatch is now set to '758'\n  Update: location is now set to '539'\n  Update: liaison is now set to '461'\n  Update: contact is now set to '172'\n  Update: assignment is now set to '429'\n  Update: registry is now set to '506'\n  Update: delegate is now set to '351'\n  Update: reference is now set to '166'\n  Update: dispatch is now set to '521'\n  Update: location is now set to '643'\n  Update: liaison is now set to '719'\n  Update: contact is now set to '695'\n  Update: assignment is now set to '119'\n  Update: registry is now set to '960'\n  Update: delegate is now set to '971'\n  Update: reference is now set to '375'\n  Update: dispatch is now set to '917'\n  Update: location is now set to '379'\n  Update: liaison is now set to '479'\n  Update: contact is now set to '931'\n  Update: assignment is now set to '643'\n  Update: registry is now set to '415'\n  Update: delegate is now set to '175'\n  Update: reference is now set to '711'\n  Update: dispatch is now set to '506'\n  Update: location is now set to '315'\n  Update: liaison is now set to '122'\n  Update: contact is now set to '823'\n  Update: assignment is now set to '407'\n  Update: registry is now set to '641'\n  Update: delegate is now set to '775'\n  Update: reference is now set to '683'\n  Update: dispatch is now set to '192'\n  Update: location is now set to '241'\n  Update: liaison is now set to '878'\n  Update: contact is now set to '413'\n  Update: assignment is now set to '680'\n  Update: registry is now set to '929'\n  Update: delegate is now set to '485'\n  Update: reference is now set to '632'\n  Update: dispatch is now set to '915'\n  Update: location is now set to '565'\n  Update: liaison is now set to '450'\n  Update: contact is now set to '220'\n  Update: assignment is now set to '559'\n  Update: registry is now set to '658'\n  Update: delegate is now set to '527'\n  Update: reference is now set to '752'\n  Update: dispatch is now set to '963'\n  Update: location is now set to '338'\n  Update: liaison is now set to '618'\n  Update: contact is now set to '824'\n  Update: assignment is now set to '209'\n  Update: registry is now set to '554'\n  Update: delegate is now set to '306'\n  Update: reference is now set to '310'\n  Update: dispatch is now set to '354'\n  Update: location is now set to '584'\n  Update: liaison is now set to '197'\n  Update: contact is now set to '599'\n  Update: assignment is now set to '564'\n  Update: registry is now set to '503'\n  Update: delegate is now set to '296'\n  Update: reference is now set to '295'\n  Update: dispatch is now set to '872'\n  Update: location is now set to '331'\n  Update: liaison is now set to '489'\n  Update: contact is now set to '551'\n  Update: assignment is now set to '205'\n  Update: registry is now set to '255'\n  Update: delegate is now set to '303'\n  Update: reference is now set to '848'\n  Update: dispatch is now set to '379'\n  Update: location is now set to '504'\n  Update: liaison is now set to '511'\n  Update: contact is now set to '271'\n  Update: assignment is now set to '313'\n  Update: registry is now set to '651'\n  Update: delegate is now set to '315'\n  Update: reference is now set to '638'\n  Update: dispatch is now set to '588'\n  Update: location is now set to '407'\n  Update: liaison is now set to '568'\n  Update: contact is now set to '700'\n  Update: assignment is now set to '774'\n  Update: registry is now set to '772'\n  Update: delegate is now set to '908'\n  Update: reference is now set to '261'\n  Update: dispatch is now set to '100'\n  Update: location is now set to '732'\n  Update: liaison is now set to '359'\n  Update: contact is now set to '718'\n  Update: assignment is now set to '736'\n  Update: registry is now set to '304'\n  Update: delegate is now set to '550'\n  Update: reference is now set to '444'\n  Update: dispatch is now set to '791'\n  Update: location is now set to '961'\n  Update: liaison is now set to '246'\n  Update: contact is now set to '183'\n  Update: assignment is now set to '802'\n  Update: registry is now set to '906'\n  Update: delegate is now set to '312'\n  Update: reference is now set to '486'\n  Update: dispatch is now set to '331'\n  Update: location is now set to '383'\n  Update: liaison is now set to '756'\n  Update: contact is now set to '769'\n  Update: assignment is now set to '424'\n  Update: registry is now set to '621'\n  Update: delegate is now set to '697'\n  Update: reference is now set to '689'\n  Update: dispatch is now set to '338'\n  Update: location is now set to '983'\n  Update: liaison is now set to '345'\n  Update: contact is now set to '533'\n  Update: assignment is now set to '662'\n  Update: registry is now set to '547'\n  Update: delegate is now set to '946'\n  Update: reference is now set to '980'\n  Update: dispatch is now set to '878'\n  Update: location is now set to '338'\n  Update: liaison is now set to '159'\n  Update: contact is now set to '633'\n  Update: assignment is now set to '288'\n  Update: registry is now set to '767'\n  Update: delegate is now set to '106'\n  Update: reference is now set to '285'\n  Update: dispatch is now set to '889'\n  Update: location is now set to '159'\n  Update: liaison is now set to '144'\n  Update: contact is now set to '600'\n  Update: assignment is now set to '107'\n  Update: registry is now set to '443'\n  Update: delegate is now set to '454'\n  Update: reference is now set to '487'\n  Update: dispatch is now set to '137'\n  Update: location is now set to '981'\n  Update: liaison is now set to '998'\n  Update: contact is now set to '789'\n  Update: assignment is now set to '961'\n  Update: registry is now set to '336'\n  Update: delegate is now set to '875'\n  Update: reference is now set to '428'\n  Update: dispatch is now set to '815'\n  Update: location is now set to '219'\n  Update: liaison is now set to '581'\n  Update: contact is now set to '313'\n  Update: assignment is now set to '279'\n  Update: registry is now set to '703'\n  Update: delegate is now set to '236'\n  Update: reference is now set to '172'\n  Update: dispatch is now set to '969'\n  Update: location is now set to '120'\n  Update: liaison is now set to '345'\n  Update: contact is now set to '678'\n  Update: assignment is now set to '561'\n  Update: registry is now set to '799'\n  Update: delegate is now set to '551'\n  Update: reference is now set to '688'\n  Update: dispatch is now set to '379'\n  Update: location is now set to '693'\n  Update: liaison is now set to '612'\n  Update: contact is now set to '539'\n  Update: assignment is now set to '773'\n  Update: registry is now set to '769'\n  Update: delegate is now set to '447'\n  Update: reference is now set to '223'\n  Update: dispatch is now set to '230'\n  Update: location is now set to '226'\n  Update: liaison is now set to '382'\n  Update: contact is now set to '621'\n  Update: assignment is now set to '294'\n  Update: registry is now set to '116'\n  Update: delegate is now set to '911'\n  Update: reference is now set to '810'\n  Update: dispatch is now set to '161'\n  Update: location is now set to '713'\n  Update: liaison is now set to '660'\n  Update: contact is now set to '997'\n  Update: assignment is now set to '645'\n  Update: registry is now set to '753'\n  Update: delegate is now set to '449'\n  Update: reference is now set to '539'\n  Update: dispatch is now set to '166'\n  Update: location is now set to '381'\n  Update: liaison is now set to '147'\n  Update: contact is now set to '838'\n  Update: assignment is now set to '437'\n  Update: registry is now set to '177'\n  Update: delegate is now set to '154'\n  Update: reference is now set to '574'\n  Update: dispatch is now set to '112'\n  Update: location is now set to '272'\n  Update: liaison is now set to '441'\n  Update: contact is now set to '886'\n  Update: assignment is now set to '887'\n  Update: registry is now set to '974'\n  Update: delegate is now set to '534'\n  Update: reference is now set to '166'\n  Update: dispatch is now set to '911'\n  Update: location is now set to '125'\n  Update: liaison is now set to '946'\n  Update: contact is now set to '192'\n  Update: assignment is now set to '200'\n  Update: registry is now set to '479'\n  Update: delegate is now set to '649'\n  Update: reference is now set to '675'\n  Update: dispatch is now set to '549'\n  Update: location is now set to '152'\n  Update: liaison is now set to '809'\n  Update: contact is now set to '481'\n  Update: assignment is now set to '226'\n  Update: registry is now set to '235'\n  Update: delegate is now set to '928'\n  Update: reference is now set to '885'\n  Update: dispatch is now set to '828'\n  Update: location is now set to '404'\n  Update: liaison is now set to '532'\n  Update: contact is now set to '307'\n  Update: assignment is now set to '758'\n  Update: registry is now set to '105'\n  Update: delegate is now set to '457'\n  Update: reference is now set to '211'\n  Update: dispatch is now set to '176'\n  Update: location is now set to '372'\n  Update: liaison is now set to '339'\n  Update: contact is now set to '604'\n  Update: assignment is now set to '116'\n  Update: registry is now set to '807'\n  Update: delegate is now set to '143'\n  Update: reference is now set to '558'\n  Update: dispatch is now set to '483'\n  Update: location is now set to '855'\n  Update: liaison is now set to '766'\n  Update: contact is now set to '458'\n  Update: assignment is now set to '352'\n  Update: registry is now set to '528'\n  Update: delegate is now set to '253'\n  Update: reference is now set to '206'\n  Update: dispatch is now set to '142'\n  Update: location is now set to '301'\n  Update: liaison is now set to '251'\n  Update: contact is now set to '750'\n  Update: assignment is now set to '578'\n  Update: registry is now set to '929'\n  Update: delegate is now set to '829'\n  Update: reference is now set to '518'\n  Update: dispatch is now set to '237'\n  Update: location is now set to '149'\n  Update: liaison is now set to '989'\n  Update: contact is now set to '993'\n  Update: assignment is now set to '731'\n  Update: registry is now set to '947'\n  Update: delegate is now set to '172'\n  Update: reference is now set to '606'\n  Update: dispatch is now set to '180'\n  Update: location is now set to '175'\n  Update: liaison is now set to '417'\n  Update: contact is now set to '870'\n  Update: assignment is now set to '368'\n  Update: registry is now set to '581'\n  Update: delegate is now set to '863'\n  Update: reference is now set to '743'\n  Update: dispatch is now set to '907'\n  Update: location is now set to '280'\n  Update: liaison is now set to '700'\n  Update: contact is now set to '299'\n  Update: assignment is now set to '120'\n  Update: registry is now set to '532'\n  Update: delegate is now set to '170'\n  Update: reference is now set to '718'\n  Update: dispatch is now set to '425'\n  Update: location is now set to '722'\n  Update: liaison is now set to '997'\n  Update: contact is now set to '592'\n  Update: assignment is now set to '222'\n  Update: registry is now set to '127'\n  Update: delegate is now set to '597'\n  Update: reference is now set to '680'\n  Update: dispatch is now set to '986'\n  Update: location is now set to '773'\n  Update: liaison is now set to '434'\n  Update: contact is now set to '633'\n  Update: assignment is now set to '381'\n  Update: registry is now set to '165'\n  Update: delegate is now set to '246'\n  Update: reference is now set to '707'\n  Update: dispatch is now set to '710'\n  Update: location is now set to '222'\n  Update: liaison is now set to '243'\n  Update: contact is now set to '393'\n  Update: assignment is now set to '628'\n  Update: registry is now set to '285'\n  Update: delegate is now set to '606'\n  Update: reference is now set to '422'\n  Update: dispatch is now set to '825'\n  Update: location is now set to '144'\n  Update: liaison is now set to '868'\n  Update: contact is now set to '492'\n  Update: assignment is now set to '496'\n  Update: registry is now set to '719'\n  Update: delegate is now set to '192'\n  Update: reference is now set to '317'\n  Update: dispatch is now set to '649'\n  Update: location is now set to '243'\n  Update: liaison is now set to '305'\n  Update: contact is now set to '920'\n  Update: assignment is now set to '321'\n  Update: registry is now set to '908'\n  Update: delegate is now set to '612'\n  Update: reference is now set to '749'\n  Update: dispatch is now set to '702'\n  Update: location is now set to '782'\n  Update: liaison is now set to '128'\n  Update: contact is now set to '166'\n  Update: assignment is now set to '688'\n  Update: registry is now set to '439'\n  Update: delegate is now set to '955'\n  Update: reference is now set to '325'\n  Update: dispatch is now set to '644'\n  Update: location is now set to '150'\n  Update: liaison is now set to '786'\n  Update: contact is now set to '732'\n  Update: assignment is now set to '242'\n  Update: registry is now set to '840'\n  Update: delegate is now set to '257'\n  Update: reference is now set to '167'\n  Update: dispatch is now set to '860'\n  Update: location is now set to '594'\n  Update: liaison is now set to '893'\n  Update: contact is now set to '309'\n  Update: assignment is now set to '729'\n  Update: registry is now set to '928'\n  Update: delegate is now set to '307'\n  Update: reference is now set to '566'\n  Update: dispatch is now set to '234'\n  Update: location is now set to '498'\n  Update: liaison is now set to '492'\n  Update: contact is now set to '458'\n  Update: assignment is now set to '215'\n  Update: registry is now set to '807'\n  Update: delegate is now set to '448'\n  Update: reference is now set to '420'\n  Update: dispatch is now set to '492'\n  Update: location is now set to '344'\n  Update: liaison is now set to '425'\n  Update: contact is now set to '226'\n  Update: assignment is now set to '355'\n  Update: registry is now set to '273'\n  Update: delegate is now set to '264'\n  Update: reference is now set to '527'\n  Update: dispatch is now set to '844'\n  Update: location is now set to '680'\n  Update: liaison is now set to '601'\n  Update: contact is now set to '758'\n  Update: assignment is now set to '590'\n  Update: registry is now set to '699'\n  Update: delegate is now set to '960'\n  Update: reference is now set to '909'\n  Update: dispatch is now set to '146'\n  Update: location is now set to '451'\n  Update: liaison is now set to '341'\n  Update: contact is now set to '497'\n  Update: assignment is now set to '415'\n  Update: registry is now set to '933'\n  Update: delegate is now set to '438'\n  Update: reference is now set to '952'\n  Update: dispatch is now set to '707'\n  Update: location is now set to '596'\n  Update: liaison is now set to '149'\n  Update: contact is now set to '590'\n  Update: assignment is now set to '904'\n  Update: registry is now set to '817'\n  Update: delegate is now set to '288'\n  Update: reference is now set to '530'\n  Update: dispatch is now set to '535'\n  Update: location is now set to '348'\n  Update: liaison is now set to '187'\n  Update: contact is now set to '416'\n  Update: assignment is now set to '693'\n  Update: registry is now set to '841'\n  Update: delegate is now set to '551'\n  Update: reference is now set to '188'\n  Update: dispatch is now set to '480'\n  Update: location is now set to '512'\n\nWhat is the FINAL value of each record?\nANSWER:\n- liaison: [final value]\n- contact: [final value]\n- assignment: [final value]\n- registry: [final value]\n- delegate: [final value]\n- reference: [final value]\n- dispatch: [final value]\n- location: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was '809' ever assigned to liaison? [Yes/No]\nV2. Was '700' ever assigned to contact? [Yes/No]\nV3. Was '564' ever assigned to assignment? [Yes/No]\nV4. Was '479' ever assigned to registry? [Yes/No]",
  "gold_json": "{\"final_values\": {\"liaison\": \"187\", \"contact\": \"416\", \"assignment\": \"693\", \"registry\": \"841\", \"delegate\": \"551\", \"reference\": \"188\", \"dispatch\": \"480\", \"location\": \"512\"}, \"key_names\": [\"liaison\", \"contact\", \"assignment\", \"registry\", \"delegate\", \"reference\", \"dispatch\", \"location\"]}"
 },
 {
  "task_id": "interference_frontier_037",
  "task_type": "interference",
  "difficulty": "Frontier",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: assignment is now set to '254'\n  Update: location is now set to '243'\n  Update: destination is now set to '948'\n  Update: dispatch is now set to '385'\n  Update: contact is now set to '350'\n  Update: coordinator is now set to '320'\n  Update: registry is now set to '250'\n  Update: delegate is now set to '139'\n  Update: assignment is now set to '577'\n  Update: location is now set to '826'\n  Update: destination is now set to '929'\n  Update: dispatch is now set to '531'\n  Update: contact is now set to '827'\n  Update: coordinator is now set to '185'\n  Update: registry is now set to '492'\n  Update: delegate is now set to '206'\n  Update: assignment is now set to '502'\n  Update: location is now set to '614'\n  Update: destination is now set to '715'\n  Update: dispatch is now set to '983'\n  Update: contact is now set to '657'\n  Update: coordinator is now set to '709'\n  Update: registry is now set to '419'\n  Update: delegate is now set to '512'\n  Update: assignment is now set to '123'\n  Update: location is now set to '150'\n  Update: destination is now set to '834'\n  Update: dispatch is now set to '561'\n  Update: contact is now set to '283'\n  Update: coordinator is now set to '472'\n  Update: registry is now set to '224'\n  Update: delegate is now set to '967'\n  Update: assignment is now set to '497'\n  Update: location is now set to '432'\n  Update: destination is now set to '368'\n  Update: dispatch is now set to '542'\n  Update: contact is now set to '191'\n  Update: coordinator is now set to '950'\n  Update: registry is now set to '742'\n  Update: delegate is now set to '200'\n  Update: assignment is now set to '683'\n  Update: location is now set to '195'\n  Update: destination is now set to '468'\n  Update: dispatch is now set to '856'\n  Update: contact is now set to '250'\n  Update: coordinator is now set to '827'\n  Update: registry is now set to '996'\n  Update: delegate is now set to '646'\n  Update: assignment is now set to '701'\n  Update: location is now set to '939'\n  Update: destination is now set to '283'\n  Update: dispatch is now set to '982'\n  Update: contact is now set to '630'\n  Update: coordinator is now set to '547'\n  Update: registry is now set to '419'\n  Update: delegate is now set to '737'\n  Update: assignment is now set to '138'\n  Update: location is now set to '764'\n  Update: destination is now set to '545'\n  Update: dispatch is now set to '810'\n  Update: contact is now set to '159'\n  Update: coordinator is now set to '919'\n  Update: registry is now set to '804'\n  Update: delegate is now set to '559'\n  Update: assignment is now set to '581'\n  Update: location is now set to '671'\n  Update: destination is now set to '994'\n  Update: dispatch is now set to '343'\n  Update: contact is now set to '246'\n  Update: coordinator is now set to '906'\n  Update: registry is now set to '238'\n  Update: delegate is now set to '158'\n  Update: assignment is now set to '590'\n  Update: location is now set to '683'\n  Update: destination is now set to '635'\n  Update: dispatch is now set to '122'\n  Update: contact is now set to '984'\n  Update: coordinator is now set to '146'\n  Update: registry is now set to '839'\n  Update: delegate is now set to '404'\n  Update: assignment is now set to '384'\n  Update: location is now set to '988'\n  Update: destination is now set to '357'\n  Update: dispatch is now set to '932'\n  Update: contact is now set to '443'\n  Update: coordinator is now set to '431'\n  Update: registry is now set to '932'\n  Update: delegate is now set to '238'\n  Update: assignment is now set to '852'\n  Update: location is now set to '925'\n  Update: destination is now set to '346'\n  Update: dispatch is now set to '108'\n  Update: contact is now set to '937'\n  Update: coordinator is now set to '464'\n  Update: registry is now set to '317'\n  Update: delegate is now set to '972'\n  Update: assignment is now set to '318'\n  Update: location is now set to '192'\n  Update: destination is now set to '831'\n  Update: dispatch is now set to '562'\n  Update: contact is now set to '613'\n  Update: coordinator is now set to '884'\n  Update: registry is now set to '628'\n  Update: delegate is now set to '606'\n  Update: assignment is now set to '789'\n  Update: location is now set to '784'\n  Update: destination is now set to '160'\n  Update: dispatch is now set to '512'\n  Update: contact is now set to '905'\n  Update: coordinator is now set to '735'\n  Update: registry is now set to '255'\n  Update: delegate is now set to '984'\n  Update: assignment is now set to '787'\n  Update: location is now set to '365'\n  Update: destination is now set to '879'\n  Update: dispatch is now set to '797'\n  Update: contact is now set to '137'\n  Update: coordinator is now set to '786'\n  Update: registry is now set to '530'\n  Update: delegate is now set to '763'\n  Update: assignment is now set to '449'\n  Update: location is now set to '538'\n  Update: destination is now set to '477'\n  Update: dispatch is now set to '268'\n  Update: contact is now set to '796'\n  Update: coordinator is now set to '848'\n  Update: registry is now set to '570'\n  Update: delegate is now set to '571'\n  Update: assignment is now set to '414'\n  Update: location is now set to '444'\n  Update: destination is now set to '738'\n  Update: dispatch is now set to '219'\n  Update: contact is now set to '914'\n  Update: coordinator is now set to '284'\n  Update: registry is now set to '698'\n  Update: delegate is now set to '901'\n  Update: assignment is now set to '593'\n  Update: location is now set to '172'\n  Update: destination is now set to '927'\n  Update: dispatch is now set to '973'\n  Update: contact is now set to '109'\n  Update: coordinator is now set to '826'\n  Update: registry is now set to '212'\n  Update: delegate is now set to '990'\n  Update: assignment is now set to '992'\n  Update: location is now set to '523'\n  Update: destination is now set to '904'\n  Update: dispatch is now set to '905'\n  Update: contact is now set to '110'\n  Update: coordinator is now set to '870'\n  Update: registry is now set to '810'\n  Update: delegate is now set to '545'\n  Update: assignment is now set to '429'\n  Update: location is now set to '269'\n  Update: destination is now set to '757'\n  Update: dispatch is now set to '549'\n  Update: contact is now set to '493'\n  Update: coordinator is now set to '868'\n  Update: registry is now set to '531'\n  Update: delegate is now set to '577'\n  Update: assignment is now set to '353'\n  Update: location is now set to '744'\n  Update: destination is now set to '740'\n  Update: dispatch is now set to '938'\n  Update: contact is now set to '355'\n  Update: coordinator is now set to '899'\n  Update: registry is now set to '526'\n  Update: delegate is now set to '509'\n  Update: assignment is now set to '860'\n  Update: location is now set to '656'\n  Update: destination is now set to '648'\n  Update: dispatch is now set to '620'\n  Update: contact is now set to '720'\n  Update: coordinator is now set to '155'\n  Update: registry is now set to '444'\n  Update: delegate is now set to '823'\n  Update: assignment is now set to '465'\n  Update: location is now set to '925'\n  Update: destination is now set to '284'\n  Update: dispatch is now set to '604'\n  Update: contact is now set to '983'\n  Update: coordinator is now set to '545'\n  Update: registry is now set to '377'\n  Update: delegate is now set to '799'\n  Update: assignment is now set to '309'\n  Update: location is now set to '851'\n  Update: destination is now set to '304'\n  Update: dispatch is now set to '821'\n  Update: contact is now set to '497'\n  Update: coordinator is now set to '880'\n  Update: registry is now set to '955'\n  Update: delegate is now set to '886'\n  Update: assignment is now set to '196'\n  Update: location is now set to '961'\n  Update: destination is now set to '869'\n  Update: dispatch is now set to '242'\n  Update: contact is now set to '659'\n  Update: coordinator is now set to '805'\n  Update: registry is now set to '858'\n  Update: delegate is now set to '815'\n  Update: assignment is now set to '669'\n  Update: location is now set to '835'\n  Update: destination is now set to '235'\n  Update: dispatch is now set to '153'\n  Update: contact is now set to '490'\n  Update: coordinator is now set to '627'\n  Update: registry is now set to '640'\n  Update: delegate is now set to '581'\n  Update: assignment is now set to '690'\n  Update: location is now set to '713'\n  Update: destination is now set to '139'\n  Update: dispatch is now set to '972'\n  Update: contact is now set to '133'\n  Update: coordinator is now set to '973'\n  Update: registry is now set to '144'\n  Update: delegate is now set to '400'\n  Update: assignment is now set to '861'\n  Update: location is now set to '850'\n  Update: destination is now set to '581'\n  Update: dispatch is now set to '338'\n  Update: contact is now set to '619'\n  Update: coordinator is now set to '666'\n  Update: registry is now set to '211'\n  Update: delegate is now set to '288'\n  Update: assignment is now set to '215'\n  Update: location is now set to '288'\n  Update: destination is now set to '387'\n  Update: dispatch is now set to '469'\n  Update: contact is now set to '860'\n  Update: coordinator is now set to '876'\n  Update: registry is now set to '993'\n  Update: delegate is now set to '608'\n  Update: assignment is now set to '761'\n  Update: location is now set to '199'\n  Update: destination is now set to '357'\n  Update: dispatch is now set to '994'\n  Update: contact is now set to '337'\n  Update: coordinator is now set to '505'\n  Update: registry is now set to '876'\n  Update: delegate is now set to '198'\n  Update: assignment is now set to '454'\n  Update: location is now set to '329'\n  Update: destination is now set to '454'\n  Update: dispatch is now set to '799'\n  Update: contact is now set to '292'\n  Update: coordinator is now set to '165'\n  Update: registry is now set to '849'\n  Update: delegate is now set to '558'\n  Update: assignment is now set to '219'\n  Update: location is now set to '310'\n  Update: destination is now set to '607'\n  Update: dispatch is now set to '586'\n  Update: contact is now set to '634'\n  Update: coordinator is now set to '962'\n  Update: registry is now set to '815'\n  Update: delegate is now set to '616'\n  Update: assignment is now set to '421'\n  Update: location is now set to '266'\n  Update: destination is now set to '236'\n  Update: dispatch is now set to '326'\n  Update: contact is now set to '534'\n  Update: coordinator is now set to '164'\n  Update: registry is now set to '228'\n  Update: delegate is now set to '308'\n  Update: assignment is now set to '121'\n  Update: location is now set to '156'\n  Update: destination is now set to '511'\n  Update: dispatch is now set to '314'\n  Update: contact is now set to '289'\n  Update: coordinator is now set to '282'\n  Update: registry is now set to '661'\n  Update: delegate is now set to '913'\n  Update: assignment is now set to '255'\n  Update: location is now set to '174'\n  Update: destination is now set to '465'\n  Update: dispatch is now set to '518'\n  Update: contact is now set to '273'\n  Update: coordinator is now set to '758'\n  Update: registry is now set to '409'\n  Update: delegate is now set to '789'\n  Update: assignment is now set to '839'\n  Update: location is now set to '975'\n  Update: destination is now set to '827'\n  Update: dispatch is now set to '210'\n  Update: contact is now set to '638'\n  Update: coordinator is now set to '788'\n  Update: registry is now set to '754'\n  Update: delegate is now set to '672'\n  Update: assignment is now set to '806'\n  Update: location is now set to '930'\n  Update: destination is now set to '394'\n  Update: dispatch is now set to '529'\n  Update: contact is now set to '169'\n  Update: coordinator is now set to '982'\n  Update: registry is now set to '421'\n  Update: delegate is now set to '914'\n  Update: assignment is now set to '337'\n  Update: location is now set to '588'\n  Update: destination is now set to '952'\n  Update: dispatch is now set to '277'\n  Update: contact is now set to '297'\n  Update: coordinator is now set to '141'\n  Update: registry is now set to '286'\n  Update: delegate is now set to '175'\n  Update: assignment is now set to '149'\n  Update: location is now set to '511'\n  Update: destination is now set to '607'\n  Update: dispatch is now set to '592'\n  Update: contact is now set to '569'\n  Update: coordinator is now set to '734'\n  Update: registry is now set to '328'\n  Update: delegate is now set to '824'\n  Update: assignment is now set to '828'\n  Update: location is now set to '572'\n  Update: destination is now set to '525'\n  Update: dispatch is now set to '423'\n  Update: contact is now set to '440'\n  Update: coordinator is now set to '443'\n  Update: registry is now set to '564'\n  Update: delegate is now set to '453'\n  Update: assignment is now set to '817'\n  Update: location is now set to '501'\n  Update: destination is now set to '831'\n  Update: dispatch is now set to '779'\n  Update: contact is now set to '631'\n  Update: coordinator is now set to '160'\n  Update: registry is now set to '755'\n  Update: delegate is now set to '525'\n  Update: assignment is now set to '979'\n  Update: location is now set to '505'\n  Update: destination is now set to '314'\n  Update: dispatch is now set to '272'\n  Update: contact is now set to '469'\n  Update: coordinator is now set to '528'\n  Update: registry is now set to '324'\n  Update: delegate is now set to '351'\n  Update: assignment is now set to '443'\n  Update: location is now set to '606'\n  Update: destination is now set to '245'\n  Update: dispatch is now set to '693'\n  Update: contact is now set to '808'\n  Update: coordinator is now set to '482'\n  Update: registry is now set to '809'\n  Update: delegate is now set to '391'\n  Update: assignment is now set to '936'\n  Update: location is now set to '760'\n  Update: destination is now set to '331'\n  Update: dispatch is now set to '762'\n  Update: contact is now set to '776'\n  Update: coordinator is now set to '789'\n  Update: registry is now set to '732'\n  Update: delegate is now set to '837'\n  Update: assignment is now set to '206'\n  Update: location is now set to '989'\n  Update: destination is now set to '153'\n  Update: dispatch is now set to '676'\n  Update: contact is now set to '642'\n  Update: coordinator is now set to '377'\n  Update: registry is now set to '736'\n  Update: delegate is now set to '384'\n  Update: assignment is now set to '701'\n  Update: location is now set to '459'\n  Update: destination is now set to '329'\n  Update: dispatch is now set to '980'\n  Update: contact is now set to '574'\n  Update: coordinator is now set to '577'\n  Update: registry is now set to '564'\n  Update: delegate is now set to '387'\n  Update: assignment is now set to '783'\n  Update: location is now set to '328'\n  Update: destination is now set to '984'\n  Update: dispatch is now set to '917'\n  Update: contact is now set to '604'\n  Update: coordinator is now set to '720'\n  Update: registry is now set to '918'\n  Update: delegate is now set to '774'\n  Update: assignment is now set to '538'\n  Update: location is now set to '520'\n  Update: destination is now set to '990'\n  Update: dispatch is now set to '183'\n  Update: contact is now set to '869'\n  Update: coordinator is now set to '103'\n  Update: registry is now set to '392'\n  Update: delegate is now set to '419'\n  Update: assignment is now set to '739'\n  Update: location is now set to '748'\n  Update: destination is now set to '839'\n  Update: dispatch is now set to '776'\n  Update: contact is now set to '901'\n  Update: coordinator is now set to '328'\n  Update: registry is now set to '862'\n  Update: delegate is now set to '288'\n  Update: assignment is now set to '986'\n  Update: location is now set to '143'\n  Update: destination is now set to '279'\n  Update: dispatch is now set to '531'\n  Update: contact is now set to '290'\n  Update: coordinator is now set to '872'\n  Update: registry is now set to '929'\n  Update: delegate is now set to '285'\n\nWhat is the FINAL value of each record?\nANSWER:\n- assignment: [final value]\n- location: [final value]\n- destination: [final value]\n- dispatch: [final value]\n- contact: [final value]\n- coordinator: [final value]\n- registry: [final value]\n- delegate: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was '538' ever assigned to assignment? [Yes/No]\nV2. Was '243' ever assigned to location? [Yes/No]\nV3. Was '990' ever assigned to destination? [Yes/No]\nV4. Was '518' ever assigned to dispatch? [Yes/No]",
  "gold_json": "{\"final_values\": {\"assignment\": \"986\", \"location\": \"143\", \"destination\": \"279\", \"dispatch\": \"531\", \"contact\": \"290\", \"coordinator\": \"872\", \"registry\": \"929\", \"delegate\": \"285\"}, \"key_names\": [\"assignment\", \"location\", \"destination\", \"dispatch\", \"contact\", \"coordinator\", \"registry\", \"delegate\"]}"
 },
 {
  "task_id": "interference_frontier_038",
  "task_type": "interference",
  "difficulty": "Frontier",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: location is now set to 'Joelle'\n  Update: reference is now set to 'Hana'\n  Update: liaison is now set to 'Willa'\n  Update: assignment is now set to 'Olena'\n  Update: delegate is now set to 'Paloma'\n  Update: registry is now set to 'Idris'\n  Update: dispatch is now set to 'Amara'\n  Update: contact is now set to 'Lumi'\n  Update: location is now set to 'Runa'\n  Update: reference is now set to 'Joelle'\n  Update: liaison is now set to 'Lumi'\n  Update: assignment is now set to 'Zain'\n  Update: delegate is now set to 'Bashir'\n  Update: registry is now set to 'Celine'\n  Update: dispatch is now set to 'Soren'\n  Update: contact is now set to 'Tariq'\n  Update: location is now set to 'Greta'\n  Update: reference is now set to 'Leif'\n  Update: liaison is now set to 'Tariq'\n  Update: assignment is now set to 'Vesna'\n  Update: delegate is now set to 'Orla'\n  Update: registry is now set to 'Orla'\n  Update: dispatch is now set to 'Femi'\n  Update: contact is now set to 'Paloma'\n  Update: location is now set to 'Priya'\n  Update: reference is now set to 'Wren'\n  Update: liaison is now set to 'Runa'\n  Update: assignment is now set to 'Nalini'\n  Update: delegate is now set to 'Olena'\n  Update: registry is now set to 'Tariq'\n  Update: dispatch is now set to 'Tala'\n  Update: contact is now set to 'Soren'\n  Update: location is now set to 'Willa'\n  Update: reference is now set to 'Olena'\n  Update: liaison is now set to 'Sigrid'\n  Update: assignment is now set to 'Vesna'\n  Update: delegate is now set to 'Gael'\n  Update: registry is now set to 'Leif'\n  Update: dispatch is now set to 'Yuki'\n  Update: contact is now set to 'Amara'\n  Update: location is now set to 'Qadir'\n  Update: reference is now set to 'Zora'\n  Update: liaison is now set to 'Adaeze'\n  Update: assignment is now set to 'Runa'\n  Update: delegate is now set to 'Adaeze'\n  Update: registry is now set to 'Nalini'\n  Update: dispatch is now set to 'Adaeze'\n  Update: contact is now set to 'Tala'\n  Update: location is now set to 'Orla'\n  Update: reference is now set to 'Adaeze'\n  Update: liaison is now set to 'Gael'\n  Update: assignment is now set to 'Joaquin'\n  Update: delegate is now set to 'Femi'\n  Update: registry is now set to 'Joelle'\n  Update: dispatch is now set to 'Tala'\n  Update: contact is now set to 'Gael'\n  Update: location is now set to 'Bashir'\n  Update: reference is now set to 'Ravi'\n  Update: liaison is now set to 'Viktor'\n  Update: assignment is now set to 'Priya'\n  Update: delegate is now set to 'Magnus'\n  Update: registry is now set to 'Ines'\n  Update: dispatch is now set to 'Gael'\n  Update: contact is now set to 'Zora'\n  Update: location is now set to 'Wren'\n  Update: reference is now set to 'Lumi'\n  Update: liaison is now set to 'Leif'\n  Update: assignment is now set to 'Soren'\n  Update: delegate is now set to 'Joaquin'\n  Update: registry is now set to 'Orla'\n  Update: dispatch is now set to 'Haruto'\n  Update: contact is now set to 'Hana'\n  Update: location is now set to 'Yuki'\n  Update: reference is now set to 'Idris'\n  Update: liaison is now set to 'Colette'\n  Update: assignment is now set to 'Yara'\n  Update: delegate is now set to 'Amara'\n  Update: registry is now set to 'Maren'\n  Update: dispatch is now set to 'Tala'\n  Update: contact is now set to 'Wren'\n  Update: location is now set to 'Leif'\n  Update: reference is now set to 'Gael'\n  Update: liaison is now set to 'Sigrid'\n  Update: assignment is now set to 'Kaia'\n  Update: delegate is now set to 'Freya'\n  Update: registry is now set to 'Qadir'\n  Update: dispatch is now set to 'Runa'\n  Update: contact is now set to 'Magnus'\n  Update: location is now set to 'Nalini'\n  Update: reference is now set to 'Uma'\n  Update: liaison is now set to 'Kaia'\n  Update: assignment is now set to 'Sigrid'\n  Update: delegate is now set to 'Ines'\n  Update: registry is now set to 'Paloma'\n  Update: dispatch is now set to 'Haruto'\n  Update: contact is now set to 'Runa'\n  Update: location is now set to 'Elara'\n  Update: reference is now set to 'Joaquin'\n  Update: liaison is now set to 'Runa'\n  Update: assignment is now set to 'Xander'\n  Update: delegate is now set to 'Kaia'\n  Update: registry is now set to 'Xander'\n  Update: dispatch is now set to 'Elio'\n  Update: contact is now set to 'Soren'\n  Update: location is now set to 'Colette'\n  Update: reference is now set to 'Olena'\n  Update: liaison is now set to 'Soren'\n  Update: assignment is now set to 'Joaquin'\n  Update: delegate is now set to 'Ines'\n  Update: registry is now set to 'Greta'\n  Update: dispatch is now set to 'Wren'\n  Update: contact is now set to 'Nalini'\n  Update: location is now set to 'Dariush'\n  Update: reference is now set to 'Tala'\n  Update: liaison is now set to 'Orla'\n  Update: assignment is now set to 'Qadir'\n  Update: delegate is now set to 'Runa'\n  Update: registry is now set to 'Zora'\n  Update: dispatch is now set to 'Vesna'\n  Update: contact is now set to 'Zora'\n  Update: location is now set to 'Adaeze'\n  Update: reference is now set to 'Elio'\n  Update: liaison is now set to 'Maren'\n  Update: assignment is now set to 'Joelle'\n  Update: delegate is now set to 'Dariush'\n  Update: registry is now set to 'Bram'\n  Update: dispatch is now set to 'Gael'\n  Update: contact is now set to 'Nico'\n  Update: location is now set to 'Freya'\n  Update: reference is now set to 'Nalini'\n  Update: liaison is now set to 'Colette'\n  Update: assignment is now set to 'Nalini'\n  Update: delegate is now set to 'Runa'\n  Update: registry is now set to 'Zora'\n  Update: dispatch is now set to 'Dmitri'\n  Update: contact is now set to 'Maren'\n  Update: location is now set to 'Nalini'\n  Update: reference is now set to 'Bashir'\n  Update: liaison is now set to 'Ines'\n  Update: assignment is now set to 'Maren'\n  Update: delegate is now set to 'Dariush'\n  Update: registry is now set to 'Idris'\n  Update: dispatch is now set to 'Qadir'\n  Update: contact is now set to 'Paloma'\n  Update: location is now set to 'Orla'\n  Update: reference is now set to 'Sigrid'\n  Update: liaison is now set to 'Orla'\n  Update: assignment is now set to 'Nalini'\n  Update: delegate is now set to 'Elio'\n  Update: registry is now set to 'Hana'\n  Update: dispatch is now set to 'Adaeze'\n  Update: contact is now set to 'Maren'\n  Update: location is now set to 'Vesna'\n  Update: reference is now set to 'Femi'\n  Update: liaison is now set to 'Nalini'\n  Update: assignment is now set to 'Nico'\n  Update: delegate is now set to 'Ugo'\n  Update: registry is now set to 'Haruto'\n  Update: dispatch is now set to 'Greta'\n  Update: contact is now set to 'Olena'\n  Update: location is now set to 'Elara'\n  Update: reference is now set to 'Joaquin'\n  Update: liaison is now set to 'Greta'\n  Update: assignment is now set to 'Lumi'\n  Update: delegate is now set to 'Colette'\n  Update: registry is now set to 'Yara'\n  Update: dispatch is now set to 'Elara'\n  Update: contact is now set to 'Nalini'\n  Update: location is now set to 'Femi'\n  Update: reference is now set to 'Zora'\n  Update: liaison is now set to 'Yuki'\n  Update: assignment is now set to 'Colette'\n  Update: delegate is now set to 'Elara'\n  Update: registry is now set to 'Priya'\n  Update: dispatch is now set to 'Dariush'\n  Update: contact is now set to 'Vesna'\n  Update: location is now set to 'Amara'\n  Update: reference is now set to 'Bram'\n  Update: liaison is now set to 'Willa'\n  Update: assignment is now set to 'Elio'\n  Update: delegate is now set to 'Soren'\n  Update: registry is now set to 'Magnus'\n  Update: dispatch is now set to 'Amara'\n  Update: contact is now set to 'Amara'\n  Update: location is now set to 'Greta'\n  Update: reference is now set to 'Zain'\n  Update: liaison is now set to 'Dmitri'\n  Update: assignment is now set to 'Vesna'\n  Update: delegate is now set to 'Magnus'\n  Update: registry is now set to 'Priya'\n  Update: dispatch is now set to 'Lumi'\n  Update: contact is now set to 'Gael'\n  Update: location is now set to 'Leif'\n  Update: reference is now set to 'Tala'\n  Update: liaison is now set to 'Tala'\n  Update: assignment is now set to 'Elio'\n  Update: delegate is now set to 'Kenji'\n  Update: registry is now set to 'Paloma'\n  Update: dispatch is now set to 'Femi'\n  Update: contact is now set to 'Ines'\n  Update: location is now set to 'Adaeze'\n  Update: reference is now set to 'Uma'\n  Update: liaison is now set to 'Ugo'\n  Update: assignment is now set to 'Dmitri'\n  Update: delegate is now set to 'Bram'\n  Update: registry is now set to 'Greta'\n  Update: dispatch is now set to 'Elio'\n  Update: contact is now set to 'Colette'\n  Update: location is now set to 'Qadir'\n  Update: reference is now set to 'Yuki'\n  Update: liaison is now set to 'Freya'\n  Update: assignment is now set to 'Dariush'\n  Update: delegate is now set to 'Yara'\n  Update: registry is now set to 'Amara'\n  Update: dispatch is now set to 'Tala'\n  Update: contact is now set to 'Soren'\n  Update: location is now set to 'Joaquin'\n  Update: reference is now set to 'Qadir'\n  Update: liaison is now set to 'Zain'\n  Update: assignment is now set to 'Leif'\n  Update: delegate is now set to 'Olena'\n  Update: registry is now set to 'Bram'\n  Update: dispatch is now set to 'Ugo'\n  Update: contact is now set to 'Zora'\n  Update: location is now set to 'Femi'\n  Update: reference is now set to 'Ines'\n  Update: liaison is now set to 'Uma'\n  Update: assignment is now set to 'Tariq'\n  Update: delegate is now set to 'Yuki'\n  Update: registry is now set to 'Sigrid'\n  Update: dispatch is now set to 'Elio'\n  Update: contact is now set to 'Olena'\n  Update: location is now set to 'Hana'\n  Update: reference is now set to 'Lumi'\n  Update: liaison is now set to 'Soren'\n  Update: assignment is now set to 'Adaeze'\n  Update: delegate is now set to 'Kaia'\n  Update: registry is now set to 'Bashir'\n  Update: dispatch is now set to 'Yara'\n  Update: contact is now set to 'Priya'\n  Update: location is now set to 'Yuki'\n  Update: reference is now set to 'Kenji'\n  Update: liaison is now set to 'Kenji'\n  Update: assignment is now set to 'Dmitri'\n  Update: delegate is now set to 'Ravi'\n  Update: registry is now set to 'Greta'\n  Update: dispatch is now set to 'Nalini'\n  Update: contact is now set to 'Amara'\n  Update: location is now set to 'Bram'\n  Update: reference is now set to 'Haruto'\n  Update: liaison is now set to 'Femi'\n  Update: assignment is now set to 'Priya'\n  Update: delegate is now set to 'Greta'\n  Update: registry is now set to 'Elara'\n  Update: dispatch is now set to 'Tariq'\n  Update: contact is now set to 'Freya'\n  Update: location is now set to 'Ines'\n  Update: reference is now set to 'Ines'\n  Update: liaison is now set to 'Willa'\n  Update: assignment is now set to 'Xander'\n  Update: delegate is now set to 'Qadir'\n  Update: registry is now set to 'Zain'\n  Update: dispatch is now set to 'Amara'\n  Update: contact is now set to 'Uma'\n  Update: location is now set to 'Kaia'\n  Update: reference is now set to 'Hana'\n  Update: liaison is now set to 'Kaia'\n  Update: assignment is now set to 'Elara'\n  Update: delegate is now set to 'Idris'\n  Update: registry is now set to 'Celine'\n  Update: dispatch is now set to 'Soren'\n  Update: contact is now set to 'Zora'\n  Update: location is now set to 'Zora'\n  Update: reference is now set to 'Orla'\n  Update: liaison is now set to 'Bram'\n  Update: assignment is now set to 'Yara'\n  Update: delegate is now set to 'Bashir'\n  Update: registry is now set to 'Bram'\n  Update: dispatch is now set to 'Viktor'\n  Update: contact is now set to 'Lumi'\n  Update: location is now set to 'Adaeze'\n  Update: reference is now set to 'Leif'\n  Update: liaison is now set to 'Zora'\n  Update: assignment is now set to 'Yuki'\n  Update: delegate is now set to 'Nico'\n  Update: registry is now set to 'Tariq'\n  Update: dispatch is now set to 'Maren'\n  Update: contact is now set to 'Olena'\n  Update: location is now set to 'Olena'\n  Update: reference is now set to 'Joaquin'\n  Update: liaison is now set to 'Idris'\n  Update: assignment is now set to 'Kaia'\n  Update: delegate is now set to 'Nalini'\n  Update: registry is now set to 'Wren'\n  Update: dispatch is now set to 'Dmitri'\n  Update: contact is now set to 'Qadir'\n  Update: location is now set to 'Hana'\n  Update: reference is now set to 'Gael'\n  Update: liaison is now set to 'Femi'\n  Update: assignment is now set to 'Viktor'\n  Update: delegate is now set to 'Amara'\n  Update: registry is now set to 'Bashir'\n  Update: dispatch is now set to 'Ugo'\n  Update: contact is now set to 'Runa'\n  Update: location is now set to 'Ines'\n  Update: reference is now set to 'Ines'\n  Update: liaison is now set to 'Elio'\n  Update: assignment is now set to 'Sigrid'\n  Update: delegate is now set to 'Nico'\n  Update: registry is now set to 'Kenji'\n  Update: dispatch is now set to 'Femi'\n  Update: contact is now set to 'Lumi'\n  Update: location is now set to 'Adaeze'\n  Update: reference is now set to 'Maren'\n  Update: liaison is now set to 'Greta'\n  Update: assignment is now set to 'Olena'\n  Update: delegate is now set to 'Willa'\n  Update: registry is now set to 'Sigrid'\n  Update: dispatch is now set to 'Celine'\n  Update: contact is now set to 'Viktor'\n  Update: location is now set to 'Maren'\n  Update: reference is now set to 'Sigrid'\n  Update: liaison is now set to 'Zora'\n  Update: assignment is now set to 'Haruto'\n  Update: delegate is now set to 'Adaeze'\n  Update: registry is now set to 'Lumi'\n  Update: dispatch is now set to 'Willa'\n  Update: contact is now set to 'Maren'\n  Update: location is now set to 'Zain'\n  Update: reference is now set to 'Greta'\n  Update: liaison is now set to 'Greta'\n  Update: assignment is now set to 'Joelle'\n  Update: delegate is now set to 'Joelle'\n  Update: registry is now set to 'Wren'\n  Update: dispatch is now set to 'Idris'\n  Update: contact is now set to 'Leif'\n  Update: location is now set to 'Priya'\n  Update: reference is now set to 'Hana'\n  Update: liaison is now set to 'Colette'\n  Update: assignment is now set to 'Tala'\n  Update: delegate is now set to 'Lumi'\n  Update: registry is now set to 'Elara'\n  Update: dispatch is now set to 'Ravi'\n  Update: contact is now set to 'Haruto'\n  Update: location is now set to 'Elara'\n  Update: reference is now set to 'Olena'\n  Update: liaison is now set to 'Lumi'\n  Update: assignment is now set to 'Orla'\n  Update: delegate is now set to 'Kaia'\n  Update: registry is now set to 'Ravi'\n  Update: dispatch is now set to 'Magnus'\n  Update: contact is now set to 'Vesna'\n  Update: location is now set to 'Bashir'\n  Update: reference is now set to 'Adaeze'\n  Update: liaison is now set to 'Joaquin'\n  Update: assignment is now set to 'Vesna'\n  Update: delegate is now set to 'Adaeze'\n  Update: registry is now set to 'Idris'\n  Update: dispatch is now set to 'Soren'\n  Update: contact is now set to 'Leif'\n  Update: location is now set to 'Sigrid'\n  Update: reference is now set to 'Yuki'\n  Update: liaison is now set to 'Tariq'\n  Update: assignment is now set to 'Willa'\n  Update: delegate is now set to 'Viktor'\n  Update: registry is now set to 'Magnus'\n  Update: dispatch is now set to 'Tariq'\n  Update: contact is now set to 'Celine'\n  Update: location is now set to 'Colette'\n  Update: reference is now set to 'Lumi'\n  Update: liaison is now set to 'Nalini'\n  Update: assignment is now set to 'Haruto'\n  Update: delegate is now set to 'Ravi'\n  Update: registry is now set to 'Uma'\n  Update: dispatch is now set to 'Femi'\n  Update: contact is now set to 'Wren'\n  Update: location is now set to 'Elara'\n  Update: reference is now set to 'Yuki'\n  Update: liaison is now set to 'Viktor'\n  Update: assignment is now set to 'Ines'\n  Update: delegate is now set to 'Xander'\n  Update: registry is now set to 'Elara'\n  Update: dispatch is now set to 'Hana'\n  Update: contact is now set to 'Femi'\n  Update: location is now set to 'Kaia'\n  Update: reference is now set to 'Qadir'\n  Update: liaison is now set to 'Maren'\n  Update: assignment is now set to 'Elio'\n  Update: delegate is now set to 'Idris'\n  Update: registry is now set to 'Bram'\n  Update: dispatch is now set to 'Dmitri'\n  Update: contact is now set to 'Kaia'\n  Update: location is now set to 'Xander'\n  Update: reference is now set to 'Paloma'\n  Update: liaison is now set to 'Leif'\n  Update: assignment is now set to 'Haruto'\n  Update: delegate is now set to 'Haruto'\n  Update: registry is now set to 'Nalini'\n  Update: dispatch is now set to 'Amara'\n  Update: contact is now set to 'Dmitri'\n\nWhat is the FINAL value of each record?\nANSWER:\n- location: [final value]\n- reference: [final value]\n- liaison: [final value]\n- assignment: [final value]\n- delegate: [final value]\n- registry: [final value]\n- dispatch: [final value]\n- contact: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was 'Amara' ever assigned to location? [Yes/No]\nV2. Was 'Hana' ever assigned to reference? [Yes/No]\nV3. Was 'Ines' ever assigned to liaison? [Yes/No]\nV4. Was 'Dariush' ever assigned to assignment? [Yes/No]",
  "gold_json": "{\"final_values\": {\"location\": \"Xander\", \"reference\": \"Paloma\", \"liaison\": \"Leif\", \"assignment\": \"Haruto\", \"delegate\": \"Haruto\", \"registry\": \"Nalini\", \"dispatch\": \"Amara\", \"contact\": \"Dmitri\"}, \"key_names\": [\"location\", \"reference\", \"liaison\", \"assignment\", \"delegate\", \"registry\", \"dispatch\", \"contact\"]}"
 },
 {
  "task_id": "interference_frontier_039",
  "task_type": "interference",
  "difficulty": "Frontier",
  "prompt": "You will receive a series of updates to one or more records. Each update REPLACES the previous value. After reading ALL updates, report ONLY the FINAL value for each record.\n\nUpdates (in order):\n  Update: contact is now set to 'Reykjavik'\n  Update: coordinator is now set to 'Cusco'\n  Update: liaison is now set to 'Recife'\n  Update: delegate is now set to 'Mandalay'\n  Update: location is now set to 'Tallinn'\n  Update: reference is now set to 'Valetta'\n  Update: assignment is now set to 'Kotor'\n  Update: registry is now set to 'Oulu'\n  Update: contact is now set to 'Zanzibar'\n  Update: coordinator is now set to 'Cartagena'\n  Update: liaison is now set to 'Fez'\n  Update: delegate is now set to 'Gdansk'\n  Update: location is now set to 'Luang Prabang'\n  Update: reference is now set to 'Reykjavik'\n  Update: assignment is now set to 'Kumasi'\n  Update: registry is now set to 'Reykjavik'\n  Update: contact is now set to 'Valetta'\n  Update: coordinator is now set to 'Fez'\n  Update: liaison is now set to 'Zanzibar'\n  Update: delegate is now set to 'Oulu'\n  Update: location is now set to 'Kumasi'\n  Update: reference is now set to 'Cartagena'\n  Update: assignment is now set to 'Oulu'\n  Update: registry is now set to 'Kotor'\n  Update: contact is now set to 'Mandalay'\n  Update: coordinator is now set to 'Zanzibar'\n  Update: liaison is now set to 'Plovdiv'\n  Update: delegate is now set to 'Fez'\n  Update: location is now set to 'Oulu'\n  Update: reference is now set to 'Ulaanbaatar'\n  Update: assignment is now set to 'Zanzibar'\n  Update: registry is now set to 'Tbilisi'\n  Update: contact is now set to 'Cusco'\n  Update: coordinator is now set to 'Tbilisi'\n  Update: liaison is now set to 'Tbilisi'\n  Update: delegate is now set to 'Gdansk'\n  Update: location is now set to 'Bruges'\n  Update: reference is now set to 'Plovdiv'\n  Update: assignment is now set to 'Kumasi'\n  Update: registry is now set to 'Jaipur'\n  Update: contact is now set to 'Cartagena'\n  Update: coordinator is now set to 'Kumasi'\n  Update: liaison is now set to 'Luang Prabang'\n  Update: delegate is now set to 'Ulaanbaatar'\n  Update: location is now set to 'Ulaanbaatar'\n  Update: reference is now set to 'Ulaanbaatar'\n  Update: assignment is now set to 'Gdansk'\n  Update: registry is now set to 'Kumasi'\n  Update: contact is now set to 'Cusco'\n  Update: coordinator is now set to 'Jaipur'\n  Update: liaison is now set to 'Trieste'\n  Update: delegate is now set to 'Reykjavik'\n  Update: location is now set to 'Fez'\n  Update: reference is now set to 'Tallinn'\n  Update: assignment is now set to 'Tbilisi'\n  Update: registry is now set to 'Jaipur'\n  Update: contact is now set to 'Tallinn'\n  Update: coordinator is now set to 'Mandalay'\n  Update: liaison is now set to 'Jaipur'\n  Update: delegate is now set to 'Cartagena'\n  Update: location is now set to 'Kotor'\n  Update: reference is now set to 'Kotor'\n  Update: assignment is now set to 'Cusco'\n  Update: registry is now set to 'Mandalay'\n  Update: contact is now set to 'Jaipur'\n  Update: coordinator is now set to 'Luang Prabang'\n  Update: liaison is now set to 'Reykjavik'\n  Update: delegate is now set to 'Tbilisi'\n  Update: location is now set to 'Ulaanbaatar'\n  Update: reference is now set to 'Valetta'\n  Update: assignment is now set to 'Reykjavik'\n  Update: registry is now set to 'Trieste'\n  Update: contact is now set to 'Zanzibar'\n  Update: coordinator is now set to 'Oulu'\n  Update: liaison is now set to 'Recife'\n  Update: delegate is now set to 'Mandalay'\n  Update: location is now set to 'Cartagena'\n  Update: reference is now set to 'Ulaanbaatar'\n  Update: assignment is now set to 'Fez'\n  Update: registry is now set to 'Jaipur'\n  Update: contact is now set to 'Plovdiv'\n  Update: coordinator is now set to 'Tbilisi'\n  Update: liaison is now set to 'Reykjavik'\n  Update: delegate is now set to 'Luang Prabang'\n  Update: location is now set to 'Reykjavik'\n  Update: reference is now set to 'Reykjavik'\n  Update: assignment is now set to 'Luang Prabang'\n  Update: registry is now set to 'Luang Prabang'\n  Update: contact is now set to 'Mandalay'\n  Update: coordinator is now set to 'Tallinn'\n  Update: liaison is now set to 'Mandalay'\n  Update: delegate is now set to 'Tbilisi'\n  Update: location is now set to 'Tbilisi'\n  Update: reference is now set to 'Gdansk'\n  Update: assignment is now set to 'Fez'\n  Update: registry is now set to 'Valetta'\n  Update: contact is now set to 'Ulaanbaatar'\n  Update: coordinator is now set to 'Kumasi'\n  Update: liaison is now set to 'Luang Prabang'\n  Update: delegate is now set to 'Jaipur'\n  Update: location is now set to 'Fez'\n  Update: reference is now set to 'Fez'\n  Update: assignment is now set to 'Tallinn'\n  Update: registry is now set to 'Trieste'\n  Update: contact is now set to 'Valetta'\n  Update: coordinator is now set to 'Gdansk'\n  Update: liaison is now set to 'Gdansk'\n  Update: delegate is now set to 'Fez'\n  Update: location is now set to 'Recife'\n  Update: reference is now set to 'Luang Prabang'\n  Update: assignment is now set to 'Kumasi'\n  Update: registry is now set to 'Recife'\n  Update: contact is now set to 'Kumasi'\n  Update: coordinator is now set to 'Bruges'\n  Update: liaison is now set to 'Luang Prabang'\n  Update: delegate is now set to 'Recife'\n  Update: location is now set to 'Mandalay'\n  Update: reference is now set to 'Fez'\n  Update: assignment is now set to 'Reykjavik'\n  Update: registry is now set to 'Gdansk'\n  Update: contact is now set to 'Reykjavik'\n  Update: coordinator is now set to 'Ulaanbaatar'\n  Update: liaison is now set to 'Tbilisi'\n  Update: delegate is now set to 'Kotor'\n  Update: location is now set to 'Tallinn'\n  Update: reference is now set to 'Cartagena'\n  Update: assignment is now set to 'Oulu'\n  Update: registry is now set to 'Cartagena'\n  Update: contact is now set to 'Cartagena'\n  Update: coordinator is now set to 'Luang Prabang'\n  Update: liaison is now set to 'Bruges'\n  Update: delegate is now set to 'Kumasi'\n  Update: location is now set to 'Fez'\n  Update: reference is now set to 'Ulaanbaatar'\n  Update: assignment is now set to 'Cartagena'\n  Update: registry is now set to 'Gdansk'\n  Update: contact is now set to 'Jaipur'\n  Update: coordinator is now set to 'Bruges'\n  Update: liaison is now set to 'Kotor'\n  Update: delegate is now set to 'Bruges'\n  Update: location is now set to 'Kumasi'\n  Update: reference is now set to 'Kumasi'\n  Update: assignment is now set to 'Mandalay'\n  Update: registry is now set to 'Cartagena'\n  Update: contact is now set to 'Kumasi'\n  Update: coordinator is now set to 'Valetta'\n  Update: liaison is now set to 'Valetta'\n  Update: delegate is now set to 'Mandalay'\n  Update: location is now set to 'Jaipur'\n  Update: reference is now set to 'Cartagena'\n  Update: assignment is now set to 'Ulaanbaatar'\n  Update: registry is now set to 'Fez'\n  Update: contact is now set to 'Bruges'\n  Update: coordinator is now set to 'Tallinn'\n  Update: liaison is now set to 'Fez'\n  Update: delegate is now set to 'Fez'\n  Update: location is now set to 'Fez'\n  Update: reference is now set to 'Cusco'\n  Update: assignment is now set to 'Kotor'\n  Update: registry is now set to 'Oulu'\n  Update: contact is now set to 'Plovdiv'\n  Update: coordinator is now set to 'Gdansk'\n  Update: liaison is now set to 'Mandalay'\n  Update: delegate is now set to 'Bruges'\n  Update: location is now set to 'Luang Prabang'\n  Update: reference is now set to 'Ulaanbaatar'\n  Update: assignment is now set to 'Valetta'\n  Update: registry is now set to 'Reykjavik'\n  Update: contact is now set to 'Cusco'\n  Update: coordinator is now set to 'Ulaanbaatar'\n  Update: liaison is now set to 'Reykjavik'\n  Update: delegate is now set to 'Cartagena'\n  Update: location is now set to 'Zanzibar'\n  Update: reference is now set to 'Cusco'\n  Update: assignment is now set to 'Bruges'\n  Update: registry is now set to 'Mandalay'\n  Update: contact is now set to 'Oulu'\n  Update: coordinator is now set to 'Tbilisi'\n  Update: liaison is now set to 'Mandalay'\n  Update: delegate is now set to 'Cusco'\n  Update: location is now set to 'Cartagena'\n  Update: reference is now set to 'Gdansk'\n  Update: assignment is now set to 'Trieste'\n  Update: registry is now set to 'Fez'\n  Update: contact is now set to 'Cusco'\n  Update: coordinator is now set to 'Recife'\n  Update: liaison is now set to 'Recife'\n  Update: delegate is now set to 'Ulaanbaatar'\n  Update: location is now set to 'Ulaanbaatar'\n  Update: reference is now set to 'Zanzibar'\n  Update: assignment is now set to 'Jaipur'\n  Update: registry is now set to 'Kumasi'\n  Update: contact is now set to 'Luang Prabang'\n  Update: coordinator is now set to 'Oulu'\n  Update: liaison is now set to 'Kotor'\n  Update: delegate is now set to 'Kumasi'\n  Update: location is now set to 'Luang Prabang'\n  Update: reference is now set to 'Mandalay'\n  Update: assignment is now set to 'Cartagena'\n  Update: registry is now set to 'Trieste'\n  Update: contact is now set to 'Tbilisi'\n  Update: coordinator is now set to 'Jaipur'\n  Update: liaison is now set to 'Tallinn'\n  Update: delegate is now set to 'Cartagena'\n  Update: location is now set to 'Jaipur'\n  Update: reference is now set to 'Zanzibar'\n  Update: assignment is now set to 'Oulu'\n  Update: registry is now set to 'Recife'\n  Update: contact is now set to 'Mandalay'\n  Update: coordinator is now set to 'Zanzibar'\n  Update: liaison is now set to 'Zanzibar'\n  Update: delegate is now set to 'Ulaanbaatar'\n  Update: location is now set to 'Tallinn'\n  Update: reference is now set to 'Bruges'\n  Update: assignment is now set to 'Recife'\n  Update: registry is now set to 'Kotor'\n  Update: contact is now set to 'Kumasi'\n  Update: coordinator is now set to 'Oulu'\n  Update: liaison is now set to 'Recife'\n  Update: delegate is now set to 'Tbilisi'\n  Update: location is now set to 'Ulaanbaatar'\n  Update: reference is now set to 'Tbilisi'\n  Update: assignment is now set to 'Cusco'\n  Update: registry is now set to 'Ulaanbaatar'\n  Update: contact is now set to 'Kotor'\n  Update: coordinator is now set to 'Kotor'\n  Update: liaison is now set to 'Oulu'\n  Update: delegate is now set to 'Recife'\n  Update: location is now set to 'Bruges'\n  Update: reference is now set to 'Jaipur'\n  Update: assignment is now set to 'Tbilisi'\n  Update: registry is now set to 'Cartagena'\n  Update: contact is now set to 'Fez'\n  Update: coordinator is now set to 'Tbilisi'\n  Update: liaison is now set to 'Fez'\n  Update: delegate is now set to 'Zanzibar'\n  Update: location is now set to 'Gdansk'\n  Update: reference is now set to 'Cartagena'\n  Update: assignment is now set to 'Zanzibar'\n  Update: registry is now set to 'Cusco'\n  Update: contact is now set to 'Mandalay'\n  Update: coordinator is now set to 'Fez'\n  Update: liaison is now set to 'Gdansk'\n  Update: delegate is now set to 'Kumasi'\n  Update: location is now set to 'Cusco'\n  Update: reference is now set to 'Trieste'\n  Update: assignment is now set to 'Luang Prabang'\n  Update: registry is now set to 'Valetta'\n  Update: contact is now set to 'Luang Prabang'\n  Update: coordinator is now set to 'Plovdiv'\n  Update: liaison is now set to 'Luang Prabang'\n  Update: delegate is now set to 'Jaipur'\n  Update: location is now set to 'Recife'\n  Update: reference is now set to 'Reykjavik'\n  Update: assignment is now set to 'Cusco'\n  Update: registry is now set to 'Ulaanbaatar'\n  Update: contact is now set to 'Ulaanbaatar'\n  Update: coordinator is now set to 'Tallinn'\n  Update: liaison is now set to 'Ulaanbaatar'\n  Update: delegate is now set to 'Trieste'\n  Update: location is now set to 'Tallinn'\n  Update: reference is now set to 'Oulu'\n  Update: assignment is now set to 'Jaipur'\n  Update: registry is now set to 'Tbilisi'\n  Update: contact is now set to 'Mandalay'\n  Update: coordinator is now set to 'Bruges'\n  Update: liaison is now set to 'Trieste'\n  Update: delegate is now set to 'Tbilisi'\n  Update: location is now set to 'Zanzibar'\n  Update: reference is now set to 'Recife'\n  Update: assignment is now set to 'Fez'\n  Update: registry is now set to 'Kumasi'\n  Update: contact is now set to 'Oulu'\n  Update: coordinator is now set to 'Plovdiv'\n  Update: liaison is now set to 'Valetta'\n  Update: delegate is now set to 'Mandalay'\n  Update: location is now set to 'Kotor'\n  Update: reference is now set to 'Tallinn'\n  Update: assignment is now set to 'Bruges'\n  Update: registry is now set to 'Recife'\n  Update: contact is now set to 'Tallinn'\n  Update: coordinator is now set to 'Bruges'\n  Update: liaison is now set to 'Trieste'\n  Update: delegate is now set to 'Oulu'\n  Update: location is now set to 'Valetta'\n  Update: reference is now set to 'Jaipur'\n  Update: assignment is now set to 'Reykjavik'\n  Update: registry is now set to 'Reykjavik'\n  Update: contact is now set to 'Kotor'\n  Update: coordinator is now set to 'Tallinn'\n  Update: liaison is now set to 'Tbilisi'\n  Update: delegate is now set to 'Tallinn'\n  Update: location is now set to 'Tbilisi'\n  Update: reference is now set to 'Bruges'\n  Update: assignment is now set to 'Cartagena'\n  Update: registry is now set to 'Trieste'\n  Update: contact is now set to 'Tbilisi'\n  Update: coordinator is now set to 'Gdansk'\n  Update: liaison is now set to 'Tallinn'\n  Update: delegate is now set to 'Cusco'\n  Update: location is now set to 'Cartagena'\n  Update: reference is now set to 'Ulaanbaatar'\n  Update: assignment is now set to 'Ulaanbaatar'\n  Update: registry is now set to 'Recife'\n  Update: contact is now set to 'Bruges'\n  Update: coordinator is now set to 'Kotor'\n  Update: liaison is now set to 'Gdansk'\n  Update: delegate is now set to 'Plovdiv'\n  Update: location is now set to 'Luang Prabang'\n  Update: reference is now set to 'Oulu'\n  Update: assignment is now set to 'Jaipur'\n  Update: registry is now set to 'Fez'\n  Update: contact is now set to 'Plovdiv'\n  Update: coordinator is now set to 'Cartagena'\n  Update: liaison is now set to 'Fez'\n  Update: delegate is now set to 'Gdansk'\n  Update: location is now set to 'Plovdiv'\n  Update: reference is now set to 'Luang Prabang'\n  Update: assignment is now set to 'Kumasi'\n  Update: registry is now set to 'Mandalay'\n  Update: contact is now set to 'Oulu'\n  Update: coordinator is now set to 'Valetta'\n  Update: liaison is now set to 'Reykjavik'\n  Update: delegate is now set to 'Luang Prabang'\n  Update: location is now set to 'Trieste'\n  Update: reference is now set to 'Jaipur'\n  Update: assignment is now set to 'Trieste'\n  Update: registry is now set to 'Reykjavik'\n  Update: contact is now set to 'Trieste'\n  Update: coordinator is now set to 'Bruges'\n  Update: liaison is now set to 'Recife'\n  Update: delegate is now set to 'Reykjavik'\n  Update: location is now set to 'Recife'\n  Update: reference is now set to 'Fez'\n  Update: assignment is now set to 'Zanzibar'\n  Update: registry is now set to 'Plovdiv'\n  Update: contact is now set to 'Tallinn'\n  Update: coordinator is now set to 'Gdansk'\n  Update: liaison is now set to 'Kumasi'\n  Update: delegate is now set to 'Cartagena'\n  Update: location is now set to 'Fez'\n  Update: reference is now set to 'Trieste'\n  Update: assignment is now set to 'Recife'\n  Update: registry is now set to 'Valetta'\n  Update: contact is now set to 'Recife'\n  Update: coordinator is now set to 'Ulaanbaatar'\n  Update: liaison is now set to 'Tbilisi'\n  Update: delegate is now set to 'Bruges'\n  Update: location is now set to 'Jaipur'\n  Update: reference is now set to 'Oulu'\n  Update: assignment is now set to 'Tallinn'\n  Update: registry is now set to 'Mandalay'\n  Update: contact is now set to 'Reykjavik'\n  Update: coordinator is now set to 'Fez'\n  Update: liaison is now set to 'Recife'\n  Update: delegate is now set to 'Mandalay'\n  Update: location is now set to 'Oulu'\n  Update: reference is now set to 'Valetta'\n  Update: assignment is now set to 'Zanzibar'\n  Update: registry is now set to 'Cartagena'\n  Update: contact is now set to 'Fez'\n  Update: coordinator is now set to 'Gdansk'\n  Update: liaison is now set to 'Reykjavik'\n  Update: delegate is now set to 'Recife'\n  Update: location is now set to 'Mandalay'\n  Update: reference is now set to 'Reykjavik'\n  Update: assignment is now set to 'Recife'\n  Update: registry is now set to 'Luang Prabang'\n  Update: contact is now set to 'Oulu'\n  Update: coordinator is now set to 'Ulaanbaatar'\n  Update: liaison is now set to 'Mandalay'\n  Update: delegate is now set to 'Bruges'\n  Update: location is now set to 'Trieste'\n  Update: reference is now set to 'Tallinn'\n  Update: assignment is now set to 'Cartagena'\n  Update: registry is now set to 'Kumasi'\n  Update: contact is now set to 'Gdansk'\n  Update: coordinator is now set to 'Recife'\n  Update: liaison is now set to 'Recife'\n  Update: delegate is now set to 'Luang Prabang'\n  Update: location is now set to 'Kotor'\n  Update: reference is now set to 'Tbilisi'\n  Update: assignment is now set to 'Cusco'\n  Update: registry is now set to 'Cartagena'\n  Update: contact is now set to 'Mandalay'\n  Update: coordinator is now set to 'Mandalay'\n  Update: liaison is now set to 'Cartagena'\n  Update: delegate is now set to 'Bruges'\n  Update: location is now set to 'Kumasi'\n  Update: reference is now set to 'Gdansk'\n  Update: assignment is now set to 'Valetta'\n  Update: registry is now set to 'Valetta'\n  Update: contact is now set to 'Oulu'\n  Update: coordinator is now set to 'Zanzibar'\n  Update: liaison is now set to 'Recife'\n  Update: delegate is now set to 'Luang Prabang'\n  Update: location is now set to 'Kotor'\n  Update: reference is now set to 'Valetta'\n  Update: assignment is now set to 'Reykjavik'\n  Update: registry is now set to 'Fez'\n\nWhat is the FINAL value of each record?\nANSWER:\n- contact: [final value]\n- coordinator: [final value]\n- liaison: [final value]\n- delegate: [final value]\n- location: [final value]\n- reference: [final value]\n- assignment: [final value]\n- registry: [final value]\n\nAlso answer these verification questions (Yes or No):\nV1. Was 'Tbilisi' ever assigned to contact? [Yes/No]\nV2. Was 'Tbilisi' ever assigned to coordinator? [Yes/No]\nV3. Was 'Tbilisi' ever assigned to liaison? [Yes/No]\nV4. Was 'Tallinn' ever assigned to delegate? [Yes/No]",
  "gold_json": "{\"final_values\": {\"contact\": \"Oulu\", \"coordinator\": \"Zanzibar\", \"liaison\": \"Recife\", \"delegate\": \"Luang Prabang\", \"location\": \"Kotor\", \"reference\": \"Valetta\", \"assignment\": \"Reykjavik\", \"registry\": \"Fez\"}, \"key_names\": [\"contact\", \"coordinator\", \"liaison\", \"delegate\", \"location\", \"reference\", \"assignment\", \"registry\"]}"
 },
 {
  "task_id": "blink_easy_000",
  "task_type": "blink",
  "difficulty": "Easy",
  "prompt": "Below is a rapid word stream of 20 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word\n  - Target 2 (T2): appears shortly after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. chapter\n2. stairway\n3. monument\n4. chimney\n5. EMERALD\n6. balcony\n7. pavilion\n8. corridor\n9. curtain\n10. fountain\n11. pattern\n12. table\n13. forty-five\n14. street\n15. bridge\n16. station\n17. lantern\n18. market\n19. village\n20. window\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"EMERALD\", \"t2\": \"forty-five\"}"
 },
 {
  "task_id": "blink_easy_001",
  "task_type": "blink",
  "difficulty": "Easy",
  "prompt": "Below is a rapid word stream of 20 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word\n  - Target 2 (T2): appears shortly after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. highway\n2. blanket\n3. market\n4. signal\n5. DIAMOND\n6. column\n7. river\n8. pattern\n9. kitchen\n10. curtain\n11. corner\n12. bridge\n13. nine-million\n14. corridor\n15. garden\n16. doorway\n17. chimney\n18. factory\n19. library\n20. monument\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"DIAMOND\", \"t2\": \"nine-million\"}"
 },
 {
  "task_id": "blink_easy_002",
  "task_type": "blink",
  "difficulty": "Easy",
  "prompt": "Below is a rapid word stream of 20 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word\n  - Target 2 (T2): appears shortly after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. window\n2. balcony\n3. gallery\n4. district\n5. RUBY\n6. corner\n7. pattern\n8. stairway\n9. surface\n10. fountain\n11. blanket\n12. river\n13. nine-million\n14. chapter\n15. curtain\n16. cabinet\n17. library\n18. harbor\n19. garden\n20. platform\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"RUBY\", \"t2\": \"nine-million\"}"
 },
 {
  "task_id": "blink_easy_003",
  "task_type": "blink",
  "difficulty": "Easy",
  "prompt": "Below is a rapid word stream of 20 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word\n  - Target 2 (T2): appears shortly after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. balcony\n2. table\n3. morning\n4. fountain\n5. TOPAZ\n6. river\n7. kitchen\n8. cabinet\n9. market\n10. highway\n11. curtain\n12. chamber\n13. four-thousand\n14. library\n15. factory\n16. ceiling\n17. stairway\n18. platform\n19. doorway\n20. surface\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"TOPAZ\", \"t2\": \"four-thousand\"}"
 },
 {
  "task_id": "blink_easy_004",
  "task_type": "blink",
  "difficulty": "Easy",
  "prompt": "Below is a rapid word stream of 20 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word\n  - Target 2 (T2): appears shortly after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. library\n2. surface\n3. curtain\n4. doorway\n5. TOPAZ\n6. bridge\n7. passage\n8. terrace\n9. blanket\n10. building\n11. gallery\n12. garden\n13. thirteen\n14. window\n15. factory\n16. balcony\n17. chimney\n18. column\n19. ceiling\n20. kitchen\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"TOPAZ\", \"t2\": \"thirteen\"}"
 },
 {
  "task_id": "blink_easy_005",
  "task_type": "blink",
  "difficulty": "Easy",
  "prompt": "Below is a rapid word stream of 20 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word\n  - Target 2 (T2): appears shortly after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. kitchen\n2. evening\n3. monument\n4. bridge\n5. TOPAZ\n6. shelter\n7. factory\n8. corner\n9. stairway\n10. curtain\n11. table\n12. highway\n13. twenty-eight\n14. pattern\n15. passage\n16. ceiling\n17. gallery\n18. surface\n19. cabinet\n20. blanket\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"TOPAZ\", \"t2\": \"twenty-eight\"}"
 },
 {
  "task_id": "blink_easy_006",
  "task_type": "blink",
  "difficulty": "Easy",
  "prompt": "Below is a rapid word stream of 20 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word\n  - Target 2 (T2): appears shortly after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. passage\n2. gallery\n3. column\n4. highway\n5. RUBY\n6. curtain\n7. shelter\n8. street\n9. district\n10. cabinet\n11. surface\n12. table\n13. twenty-eight\n14. chimney\n15. library\n16. doorway\n17. bridge\n18. village\n19. evening\n20. balcony\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"RUBY\", \"t2\": \"twenty-eight\"}"
 },
 {
  "task_id": "blink_easy_007",
  "task_type": "blink",
  "difficulty": "Easy",
  "prompt": "Below is a rapid word stream of 20 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word\n  - Target 2 (T2): appears shortly after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. curtain\n2. column\n3. terrace\n4. gallery\n5. AMETHYST\n6. bridge\n7. ceiling\n8. table\n9. balcony\n10. river\n11. library\n12. chapter\n13. nine-million\n14. kitchen\n15. evening\n16. window\n17. market\n18. harbor\n19. street\n20. platform\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"AMETHYST\", \"t2\": \"nine-million\"}"
 },
 {
  "task_id": "blink_medium_008",
  "task_type": "blink",
  "difficulty": "Medium",
  "prompt": "Below is a rapid word stream of 30 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word\n  - Target 2 (T2): appears shortly after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. shelter\n2. chimney\n3. curtain\n4. morning\n5. stairway\n6. market\n7. DIAMOND\n8. pavilion\n9. library\n10. kitchen\n11. balcony\n12. seven-hundred\n13. street\n14. monument\n15. chamber\n16. garden\n17. platform\n18. ceiling\n19. terrace\n20. building\n21. lantern\n22. cabinet\n23. surface\n24. fountain\n25. river\n26. district\n27. passage\n28. factory\n29. pattern\n30. bridge\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"DIAMOND\", \"t2\": \"seven-hundred\"}"
 },
 {
  "task_id": "blink_medium_009",
  "task_type": "blink",
  "difficulty": "Medium",
  "prompt": "Below is a rapid word stream of 30 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word\n  - Target 2 (T2): appears shortly after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. district\n2. factory\n3. surface\n4. ceiling\n5. window\n6. platform\n7. EMERALD\n8. building\n9. village\n10. chapter\n11. table\n12. sixty-three\n13. monument\n14. corridor\n15. shelter\n16. gallery\n17. garden\n18. corner\n19. doorway\n20. stairway\n21. evening\n22. lantern\n23. curtain\n24. column\n25. chimney\n26. highway\n27. pavilion\n28. bridge\n29. terrace\n30. library\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"EMERALD\", \"t2\": \"sixty-three\"}"
 },
 {
  "task_id": "blink_medium_010",
  "task_type": "blink",
  "difficulty": "Medium",
  "prompt": "Below is a rapid word stream of 30 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word\n  - Target 2 (T2): appears shortly after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. ceiling\n2. chapter\n3. stairway\n4. passage\n5. platform\n6. evening\n7. morning\n8. building\n9. terrace\n10. EMERALD\n11. blanket\n12. fountain\n13. market\n14. river\n15. sixty-three\n16. corner\n17. garden\n18. doorway\n19. chimney\n20. library\n21. table\n22. kitchen\n23. gallery\n24. factory\n25. pattern\n26. column\n27. corridor\n28. monument\n29. signal\n30. station\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"EMERALD\", \"t2\": \"sixty-three\"}"
 },
 {
  "task_id": "blink_medium_011",
  "task_type": "blink",
  "difficulty": "Medium",
  "prompt": "Below is a rapid word stream of 30 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word\n  - Target 2 (T2): appears shortly after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. station\n2. corner\n3. market\n4. gallery\n5. balcony\n6. corridor\n7. monument\n8. highway\n9. pattern\n10. DIAMOND\n11. morning\n12. platform\n13. garden\n14. lantern\n15. sixty-three\n16. stairway\n17. library\n18. chimney\n19. table\n20. surface\n21. district\n22. signal\n23. pavilion\n24. column\n25. blanket\n26. chamber\n27. street\n28. bridge\n29. passage\n30. ceiling\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"DIAMOND\", \"t2\": \"sixty-three\"}"
 },
 {
  "task_id": "blink_medium_012",
  "task_type": "blink",
  "difficulty": "Medium",
  "prompt": "Below is a rapid word stream of 30 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word\n  - Target 2 (T2): appears shortly after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. building\n2. district\n3. highway\n4. chamber\n5. river\n6. kitchen\n7. cabinet\n8. pattern\n9. evening\n10. EMERALD\n11. market\n12. terrace\n13. harbor\n14. chapter\n15. seven-hundred\n16. doorway\n17. table\n18. morning\n19. library\n20. balcony\n21. stairway\n22. street\n23. bridge\n24. corner\n25. fountain\n26. blanket\n27. gallery\n28. factory\n29. ceiling\n30. signal\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"EMERALD\", \"t2\": \"seven-hundred\"}"
 },
 {
  "task_id": "blink_medium_013",
  "task_type": "blink",
  "difficulty": "Medium",
  "prompt": "Below is a rapid word stream of 30 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word\n  - Target 2 (T2): appears shortly after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. balcony\n2. lantern\n3. garden\n4. surface\n5. shelter\n6. factory\n7. RUBY\n8. platform\n9. bridge\n10. cabinet\n11. market\n12. sixty-three\n13. river\n14. blanket\n15. evening\n16. street\n17. library\n18. doorway\n19. district\n20. monument\n21. signal\n22. morning\n23. passage\n24. window\n25. highway\n26. column\n27. corridor\n28. building\n29. stairway\n30. gallery\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"RUBY\", \"t2\": \"sixty-three\"}"
 },
 {
  "task_id": "blink_medium_014",
  "task_type": "blink",
  "difficulty": "Medium",
  "prompt": "Below is a rapid word stream of 30 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word\n  - Target 2 (T2): appears shortly after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. chapter\n2. building\n3. kitchen\n4. platform\n5. library\n6. evening\n7. GARNET\n8. balcony\n9. lantern\n10. fountain\n11. blanket\n12. forty-five\n13. table\n14. corner\n15. factory\n16. curtain\n17. village\n18. river\n19. corridor\n20. street\n21. surface\n22. terrace\n23. harbor\n24. doorway\n25. garden\n26. chamber\n27. gallery\n28. market\n29. station\n30. highway\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"GARNET\", \"t2\": \"forty-five\"}"
 },
 {
  "task_id": "blink_medium_015",
  "task_type": "blink",
  "difficulty": "Medium",
  "prompt": "Below is a rapid word stream of 30 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word\n  - Target 2 (T2): appears shortly after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. corner\n2. library\n3. harbor\n4. village\n5. curtain\n6. pattern\n7. GARNET\n8. cabinet\n9. kitchen\n10. column\n11. terrace\n12. forty-five\n13. gallery\n14. ceiling\n15. market\n16. passage\n17. fountain\n18. lantern\n19. chapter\n20. shelter\n21. signal\n22. station\n23. street\n24. doorway\n25. blanket\n26. morning\n27. chimney\n28. bridge\n29. building\n30. stairway\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"GARNET\", \"t2\": \"forty-five\"}"
 },
 {
  "task_id": "blink_hard_016",
  "task_type": "blink",
  "difficulty": "Hard",
  "prompt": "Below is a rapid word stream of 40 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word\n  - Target 2 (T2): appears shortly after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. chimney\n2. chapter\n3. stairway\n4. lantern\n5. signal\n6. harbor\n7. station\n8. column\n9. table\n10. morning\n11. factory\n12. terrace\n13. library\n14. curtain\n15. surface\n16. doorway\n17. ceiling\n18. DIAMOND\n19. river\n20. pattern\n21. PENGUIN\n22. kitchen\n23. platform\n24. blanket\n25. window\n26. building\n27. corner\n28. pavilion\n29. bridge\n30. passage\n31. gallery\n32. chamber\n33. district\n34. evening\n35. monument\n36. market\n37. cabinet\n38. street\n39. village\n40. highway\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"DIAMOND\", \"t2\": \"PENGUIN\"}"
 },
 {
  "task_id": "blink_hard_017",
  "task_type": "blink",
  "difficulty": "Hard",
  "prompt": "Below is a rapid word stream of 40 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word\n  - Target 2 (T2): appears shortly after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. market\n2. pattern\n3. signal\n4. morning\n5. factory\n6. highway\n7. window\n8. platform\n9. balcony\n10. stairway\n11. shelter\n12. monument\n13. chamber\n14. pavilion\n15. village\n16. EMERALD\n17. ceiling\n18. column\n19. PENGUIN\n20. evening\n21. district\n22. fountain\n23. curtain\n24. corner\n25. bridge\n26. library\n27. terrace\n28. passage\n29. building\n30. street\n31. harbor\n32. station\n33. kitchen\n34. river\n35. corridor\n36. chimney\n37. table\n38. gallery\n39. blanket\n40. lantern\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"EMERALD\", \"t2\": \"PENGUIN\"}"
 },
 {
  "task_id": "blink_hard_018",
  "task_type": "blink",
  "difficulty": "Hard",
  "prompt": "Below is a rapid word stream of 40 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word\n  - Target 2 (T2): appears shortly after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. harbor\n2. platform\n3. signal\n4. window\n5. river\n6. corridor\n7. chapter\n8. building\n9. station\n10. evening\n11. OPAL\n12. stairway\n13. library\n14. OCTOPUS\n15. curtain\n16. fountain\n17. surface\n18. market\n19. bridge\n20. chamber\n21. doorway\n22. factory\n23. pavilion\n24. kitchen\n25. shelter\n26. terrace\n27. blanket\n28. garden\n29. street\n30. cabinet\n31. monument\n32. district\n33. morning\n34. lantern\n35. gallery\n36. chimney\n37. highway\n38. village\n39. ceiling\n40. passage\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"OPAL\", \"t2\": \"OCTOPUS\"}"
 },
 {
  "task_id": "blink_hard_019",
  "task_type": "blink",
  "difficulty": "Hard",
  "prompt": "Below is a rapid word stream of 40 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word\n  - Target 2 (T2): appears shortly after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. cabinet\n2. river\n3. fountain\n4. doorway\n5. chamber\n6. morning\n7. corner\n8. passage\n9. market\n10. corridor\n11. harbor\n12. RUBY\n13. lantern\n14. factory\n15. LEOPARD\n16. gallery\n17. platform\n18. window\n19. curtain\n20. blanket\n21. garden\n22. chimney\n23. chapter\n24. library\n25. terrace\n26. signal\n27. balcony\n28. village\n29. district\n30. bridge\n31. pattern\n32. station\n33. evening\n34. street\n35. highway\n36. surface\n37. table\n38. monument\n39. column\n40. ceiling\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"RUBY\", \"t2\": \"LEOPARD\"}"
 },
 {
  "task_id": "blink_hard_020",
  "task_type": "blink",
  "difficulty": "Hard",
  "prompt": "Below is a rapid word stream of 40 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word\n  - Target 2 (T2): appears shortly after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. factory\n2. chapter\n3. market\n4. signal\n5. pavilion\n6. station\n7. lantern\n8. ceiling\n9. corridor\n10. TOPAZ\n11. chamber\n12. shelter\n13. ELEPHANT\n14. street\n15. platform\n16. garden\n17. kitchen\n18. blanket\n19. highway\n20. passage\n21. bridge\n22. window\n23. curtain\n24. table\n25. gallery\n26. surface\n27. pattern\n28. river\n29. fountain\n30. library\n31. chimney\n32. stairway\n33. cabinet\n34. building\n35. monument\n36. morning\n37. doorway\n38. balcony\n39. evening\n40. district\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"TOPAZ\", \"t2\": \"ELEPHANT\"}"
 },
 {
  "task_id": "blink_hard_021",
  "task_type": "blink",
  "difficulty": "Hard",
  "prompt": "Below is a rapid word stream of 40 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word\n  - Target 2 (T2): appears shortly after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. kitchen\n2. factory\n3. market\n4. highway\n5. gallery\n6. blanket\n7. cabinet\n8. chapter\n9. station\n10. AMETHYST\n11. fountain\n12. river\n13. LEOPARD\n14. table\n15. doorway\n16. pavilion\n17. terrace\n18. platform\n19. column\n20. corner\n21. bridge\n22. surface\n23. village\n24. chimney\n25. monument\n26. stairway\n27. lantern\n28. curtain\n29. library\n30. pattern\n31. district\n32. morning\n33. passage\n34. evening\n35. balcony\n36. building\n37. signal\n38. garden\n39. ceiling\n40. shelter\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"AMETHYST\", \"t2\": \"LEOPARD\"}"
 },
 {
  "task_id": "blink_hard_022",
  "task_type": "blink",
  "difficulty": "Hard",
  "prompt": "Below is a rapid word stream of 40 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word\n  - Target 2 (T2): appears shortly after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. garden\n2. kitchen\n3. evening\n4. gallery\n5. window\n6. passage\n7. chimney\n8. corner\n9. table\n10. chapter\n11. bridge\n12. factory\n13. platform\n14. monument\n15. stairway\n16. OPAL\n17. market\n18. lantern\n19. PEACOCK\n20. building\n21. column\n22. street\n23. chamber\n24. highway\n25. doorway\n26. library\n27. balcony\n28. pavilion\n29. cabinet\n30. curtain\n31. blanket\n32. morning\n33. village\n34. surface\n35. shelter\n36. terrace\n37. ceiling\n38. corridor\n39. pattern\n40. station\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"OPAL\", \"t2\": \"PEACOCK\"}"
 },
 {
  "task_id": "blink_hard_023",
  "task_type": "blink",
  "difficulty": "Hard",
  "prompt": "Below is a rapid word stream of 40 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word\n  - Target 2 (T2): appears shortly after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. pattern\n2. platform\n3. terrace\n4. pavilion\n5. stairway\n6. fountain\n7. column\n8. doorway\n9. table\n10. highway\n11. market\n12. passage\n13. monument\n14. EMERALD\n15. window\n16. chamber\n17. BUFFALO\n18. harbor\n19. chapter\n20. blanket\n21. signal\n22. chimney\n23. library\n24. bridge\n25. building\n26. cabinet\n27. station\n28. corner\n29. corridor\n30. street\n31. district\n32. curtain\n33. river\n34. morning\n35. shelter\n36. garden\n37. ceiling\n38. kitchen\n39. surface\n40. village\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"EMERALD\", \"t2\": \"BUFFALO\"}"
 },
 {
  "task_id": "blink_expert_024",
  "task_type": "blink",
  "difficulty": "Expert",
  "prompt": "Below is a rapid word stream of 50 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word\n  - Target 2 (T2): appears shortly after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. fountain\n2. ceiling\n3. gallery\n4. kitchen\n5. doorway\n6. signal\n7. library\n8. highway\n9. platform\n10. chimney\n11. stairway\n12. river\n13. street\n14. corridor\n15. GARNET\n16. surface\n17. twenty-eight\n18. lantern\n19. village\n20. building\n21. corner\n22. market\n23. blanket\n24. factory\n25. garden\n26. chapter\n27. district\n28. cabinet\n29. monument\n30. passage\n31. terrace\n32. window\n33. shelter\n34. morning\n35. evening\n36. harbor\n37. pattern\n38. station\n39. curtain\n40. table\n41. bridge\n42. chamber\n43. pavilion\n44. column\n45. balcony\n46. street\n47. column\n48. street\n49. ceiling\n50. station\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"GARNET\", \"t2\": \"twenty-eight\"}"
 },
 {
  "task_id": "blink_expert_025",
  "task_type": "blink",
  "difficulty": "Expert",
  "prompt": "Below is a rapid word stream of 50 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word\n  - Target 2 (T2): appears shortly after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. signal\n2. chimney\n3. bridge\n4. harbor\n5. cabinet\n6. fountain\n7. ceiling\n8. monument\n9. morning\n10. kitchen\n11. village\n12. column\n13. OPAL\n14. pavilion\n15. nine-million\n16. lantern\n17. window\n18. doorway\n19. balcony\n20. corridor\n21. river\n22. platform\n23. building\n24. factory\n25. stairway\n26. gallery\n27. table\n28. terrace\n29. district\n30. corner\n31. shelter\n32. garden\n33. street\n34. passage\n35. blanket\n36. highway\n37. chapter\n38. evening\n39. chamber\n40. pattern\n41. market\n42. curtain\n43. library\n44. station\n45. surface\n46. signal\n47. evening\n48. curtain\n49. signal\n50. chimney\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"OPAL\", \"t2\": \"nine-million\"}"
 },
 {
  "task_id": "blink_expert_026",
  "task_type": "blink",
  "difficulty": "Expert",
  "prompt": "Below is a rapid word stream of 50 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word\n  - Target 2 (T2): appears shortly after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. balcony\n2. column\n3. cabinet\n4. blanket\n5. shelter\n6. evening\n7. fountain\n8. doorway\n9. table\n10. market\n11. library\n12. lantern\n13. building\n14. highway\n15. street\n16. district\n17. station\n18. bridge\n19. corridor\n20. chamber\n21. terrace\n22. EMERALD\n23. stairway\n24. thirteen\n25. monument\n26. river\n27. village\n28. pavilion\n29. chapter\n30. garden\n31. surface\n32. factory\n33. corner\n34. chimney\n35. harbor\n36. ceiling\n37. signal\n38. morning\n39. pattern\n40. platform\n41. kitchen\n42. gallery\n43. curtain\n44. window\n45. passage\n46. stairway\n47. street\n48. blanket\n49. market\n50. passage\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"EMERALD\", \"t2\": \"thirteen\"}"
 },
 {
  "task_id": "blink_expert_027",
  "task_type": "blink",
  "difficulty": "Expert",
  "prompt": "Below is a rapid word stream of 50 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word\n  - Target 2 (T2): appears shortly after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. table\n2. surface\n3. chamber\n4. fountain\n5. evening\n6. station\n7. blanket\n8. passage\n9. kitchen\n10. river\n11. RUBY\n12. column\n13. twenty-eight\n14. library\n15. platform\n16. doorway\n17. factory\n18. terrace\n19. market\n20. lantern\n21. shelter\n22. highway\n23. monument\n24. signal\n25. corridor\n26. stairway\n27. ceiling\n28. bridge\n29. window\n30. garden\n31. gallery\n32. chapter\n33. curtain\n34. pattern\n35. village\n36. corner\n37. morning\n38. chimney\n39. district\n40. pavilion\n41. street\n42. balcony\n43. cabinet\n44. harbor\n45. building\n46. chamber\n47. highway\n48. station\n49. library\n50. window\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"RUBY\", \"t2\": \"twenty-eight\"}"
 },
 {
  "task_id": "blink_expert_028",
  "task_type": "blink",
  "difficulty": "Expert",
  "prompt": "Below is a rapid word stream of 50 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word\n  - Target 2 (T2): appears shortly after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. village\n2. harbor\n3. table\n4. doorway\n5. factory\n6. station\n7. corridor\n8. garden\n9. balcony\n10. passage\n11. terrace\n12. pavilion\n13. monument\n14. ceiling\n15. OPAL\n16. shelter\n17. sixty-three\n18. building\n19. platform\n20. cabinet\n21. street\n22. lantern\n23. fountain\n24. highway\n25. district\n26. stairway\n27. pattern\n28. bridge\n29. gallery\n30. river\n31. library\n32. chapter\n33. curtain\n34. signal\n35. chamber\n36. evening\n37. surface\n38. corner\n39. market\n40. kitchen\n41. morning\n42. column\n43. blanket\n44. chimney\n45. window\n46. ceiling\n47. column\n48. pavilion\n49. building\n50. platform\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"OPAL\", \"t2\": \"sixty-three\"}"
 },
 {
  "task_id": "blink_expert_029",
  "task_type": "blink",
  "difficulty": "Expert",
  "prompt": "Below is a rapid word stream of 50 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word\n  - Target 2 (T2): appears shortly after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. curtain\n2. evening\n3. pavilion\n4. corridor\n5. factory\n6. gallery\n7. morning\n8. surface\n9. fountain\n10. chapter\n11. market\n12. bridge\n13. GARNET\n14. river\n15. sixty-three\n16. district\n17. table\n18. pattern\n19. balcony\n20. harbor\n21. doorway\n22. stairway\n23. ceiling\n24. cabinet\n25. monument\n26. village\n27. building\n28. corner\n29. passage\n30. chimney\n31. station\n32. column\n33. chamber\n34. terrace\n35. garden\n36. window\n37. highway\n38. platform\n39. street\n40. blanket\n41. library\n42. lantern\n43. signal\n44. shelter\n45. kitchen\n46. river\n47. monument\n48. harbor\n49. evening\n50. pavilion\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"GARNET\", \"t2\": \"sixty-three\"}"
 },
 {
  "task_id": "blink_expert_030",
  "task_type": "blink",
  "difficulty": "Expert",
  "prompt": "Below is a rapid word stream of 50 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word\n  - Target 2 (T2): appears shortly after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. chimney\n2. highway\n3. terrace\n4. market\n5. harbor\n6. table\n7. district\n8. signal\n9. corridor\n10. passage\n11. garden\n12. pavilion\n13. curtain\n14. doorway\n15. kitchen\n16. library\n17. street\n18. morning\n19. river\n20. station\n21. monument\n22. shelter\n23. RUBY\n24. surface\n25. seven-hundred\n26. corner\n27. chapter\n28. balcony\n29. stairway\n30. evening\n31. lantern\n32. bridge\n33. chamber\n34. ceiling\n35. gallery\n36. platform\n37. blanket\n38. village\n39. window\n40. fountain\n41. building\n42. factory\n43. cabinet\n44. pattern\n45. column\n46. passage\n47. surface\n48. library\n49. district\n50. curtain\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"RUBY\", \"t2\": \"seven-hundred\"}"
 },
 {
  "task_id": "blink_expert_031",
  "task_type": "blink",
  "difficulty": "Expert",
  "prompt": "Below is a rapid word stream of 50 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word\n  - Target 2 (T2): appears shortly after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. chapter\n2. factory\n3. evening\n4. column\n5. street\n6. cabinet\n7. garden\n8. monument\n9. station\n10. chimney\n11. chamber\n12. ceiling\n13. signal\n14. AMETHYST\n15. corner\n16. thirteen\n17. corridor\n18. kitchen\n19. morning\n20. terrace\n21. bridge\n22. blanket\n23. village\n24. building\n25. platform\n26. market\n27. curtain\n28. gallery\n29. passage\n30. highway\n31. library\n32. doorway\n33. pattern\n34. fountain\n35. pavilion\n36. surface\n37. window\n38. shelter\n39. river\n40. harbor\n41. stairway\n42. district\n43. lantern\n44. balcony\n45. table\n46. building\n47. highway\n48. surface\n49. river\n50. village\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"AMETHYST\", \"t2\": \"thirteen\"}"
 },
 {
  "task_id": "blink_frontier_032",
  "task_type": "blink",
  "difficulty": "Frontier",
  "prompt": "Below is a rapid word stream of 80 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word\n  - Target 2 (T2): appears shortly after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. shelter\n2. pavilion\n3. table\n4. factory\n5. district\n6. lantern\n7. chapter\n8. harbor\n9. market\n10. doorway\n11. village\n12. bridge\n13. highway\n14. chamber\n15. pattern\n16. street\n17. sixty-three\n18. DIAMOND\n19. passage\n20. corner\n21. building\n22. surface\n23. curtain\n24. stairway\n25. evening\n26. fountain\n27. signal\n28. gallery\n29. garden\n30. chimney\n31. river\n32. morning\n33. column\n34. monument\n35. balcony\n36. blanket\n37. kitchen\n38. station\n39. window\n40. library\n41. terrace\n42. corridor\n43. platform\n44. ceiling\n45. cabinet\n46. surface\n47. district\n48. pavilion\n49. highway\n50. cabinet\n51. platform\n52. chamber\n53. market\n54. corner\n55. pavilion\n56. morning\n57. ceiling\n58. fountain\n59. window\n60. platform\n61. garden\n62. monument\n63. station\n64. lantern\n65. balcony\n66. garden\n67. street\n68. stairway\n69. corridor\n70. street\n71. surface\n72. shelter\n73. village\n74. chamber\n75. monument\n76. kitchen\n77. street\n78. cabinet\n79. chamber\n80. passage\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"sixty-three\", \"t2\": \"DIAMOND\"}"
 },
 {
  "task_id": "blink_frontier_033",
  "task_type": "blink",
  "difficulty": "Frontier",
  "prompt": "Below is a rapid word stream of 80 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word\n  - Target 2 (T2): appears shortly after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. chapter\n2. blanket\n3. surface\n4. station\n5. stairway\n6. ceiling\n7. library\n8. lantern\n9. pavilion\n10. curtain\n11. doorway\n12. table\n13. evening\n14. chimney\n15. balcony\n16. factory\n17. harbor\n18. corridor\n19. passage\n20. signal\n21. gallery\n22. highway\n23. district\n24. kitchen\n25. garden\n26. bridge\n27. village\n28. window\n29. street\n30. chamber\n31. column\n32. corner\n33. sixty-three\n34. OPAL\n35. cabinet\n36. morning\n37. market\n38. pattern\n39. river\n40. platform\n41. building\n42. fountain\n43. terrace\n44. monument\n45. shelter\n46. harbor\n47. pattern\n48. bridge\n49. chimney\n50. ceiling\n51. signal\n52. gallery\n53. street\n54. window\n55. market\n56. balcony\n57. pavilion\n58. ceiling\n59. morning\n60. surface\n61. pattern\n62. chamber\n63. balcony\n64. morning\n65. table\n66. table\n67. passage\n68. evening\n69. building\n70. window\n71. pattern\n72. chapter\n73. fountain\n74. shelter\n75. factory\n76. column\n77. evening\n78. platform\n79. kitchen\n80. window\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"sixty-three\", \"t2\": \"OPAL\"}"
 },
 {
  "task_id": "blink_frontier_034",
  "task_type": "blink",
  "difficulty": "Frontier",
  "prompt": "Below is a rapid word stream of 80 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word\n  - Target 2 (T2): appears shortly after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. window\n2. building\n3. fountain\n4. curtain\n5. chapter\n6. passage\n7. district\n8. shelter\n9. pavilion\n10. blanket\n11. morning\n12. station\n13. chamber\n14. column\n15. terrace\n16. garden\n17. doorway\n18. market\n19. highway\n20. cabinet\n21. twenty-eight\n22. OPAL\n23. surface\n24. street\n25. river\n26. library\n27. gallery\n28. corner\n29. ceiling\n30. lantern\n31. kitchen\n32. monument\n33. stairway\n34. factory\n35. evening\n36. bridge\n37. platform\n38. chimney\n39. table\n40. signal\n41. pattern\n42. village\n43. corridor\n44. balcony\n45. harbor\n46. street\n47. gallery\n48. street\n49. balcony\n50. window\n51. lantern\n52. lantern\n53. window\n54. evening\n55. surface\n56. gallery\n57. cabinet\n58. table\n59. factory\n60. evening\n61. village\n62. table\n63. fountain\n64. column\n65. lantern\n66. balcony\n67. window\n68. ceiling\n69. cabinet\n70. stairway\n71. river\n72. pattern\n73. lantern\n74. pattern\n75. fountain\n76. highway\n77. curtain\n78. factory\n79. shelter\n80. kitchen\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"twenty-eight\", \"t2\": \"OPAL\"}"
 },
 {
  "task_id": "blink_frontier_035",
  "task_type": "blink",
  "difficulty": "Frontier",
  "prompt": "Below is a rapid word stream of 80 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word\n  - Target 2 (T2): appears shortly after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. terrace\n2. kitchen\n3. village\n4. curtain\n5. station\n6. shelter\n7. lantern\n8. platform\n9. morning\n10. balcony\n11. ceiling\n12. garden\n13. river\n14. highway\n15. street\n16. blanket\n17. monument\n18. chapter\n19. doorway\n20. eighty-one\n21. OPAL\n22. factory\n23. pavilion\n24. harbor\n25. pattern\n26. evening\n27. table\n28. passage\n29. district\n30. chimney\n31. window\n32. chamber\n33. building\n34. stairway\n35. surface\n36. corridor\n37. fountain\n38. market\n39. corner\n40. signal\n41. library\n42. bridge\n43. cabinet\n44. column\n45. gallery\n46. kitchen\n47. stairway\n48. curtain\n49. library\n50. monument\n51. pattern\n52. platform\n53. curtain\n54. window\n55. morning\n56. stairway\n57. gallery\n58. station\n59. chamber\n60. column\n61. factory\n62. table\n63. surface\n64. platform\n65. surface\n66. chimney\n67. evening\n68. cabinet\n69. window\n70. kitchen\n71. ceiling\n72. window\n73. gallery\n74. highway\n75. corridor\n76. market\n77. surface\n78. curtain\n79. passage\n80. pavilion\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"eighty-one\", \"t2\": \"OPAL\"}"
 },
 {
  "task_id": "blink_frontier_036",
  "task_type": "blink",
  "difficulty": "Frontier",
  "prompt": "Below is a rapid word stream of 80 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word\n  - Target 2 (T2): appears shortly after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. gallery\n2. building\n3. corner\n4. column\n5. market\n6. garden\n7. chapter\n8. chimney\n9. morning\n10. street\n11. terrace\n12. passage\n13. table\n14. surface\n15. curtain\n16. platform\n17. evening\n18. shelter\n19. stairway\n20. balcony\n21. fountain\n22. doorway\n23. bridge\n24. corridor\n25. harbor\n26. chamber\n27. four-thousand\n28. DIAMOND\n29. river\n30. factory\n31. station\n32. library\n33. pavilion\n34. pattern\n35. monument\n36. village\n37. window\n38. ceiling\n39. blanket\n40. signal\n41. cabinet\n42. kitchen\n43. lantern\n44. district\n45. highway\n46. harbor\n47. pattern\n48. station\n49. market\n50. kitchen\n51. monument\n52. surface\n53. fountain\n54. pavilion\n55. harbor\n56. doorway\n57. surface\n58. pattern\n59. shelter\n60. blanket\n61. district\n62. library\n63. factory\n64. terrace\n65. village\n66. morning\n67. lantern\n68. evening\n69. library\n70. gallery\n71. chimney\n72. pavilion\n73. monument\n74. kitchen\n75. district\n76. lantern\n77. garden\n78. river\n79. highway\n80. river\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"four-thousand\", \"t2\": \"DIAMOND\"}"
 },
 {
  "task_id": "blink_frontier_037",
  "task_type": "blink",
  "difficulty": "Frontier",
  "prompt": "Below is a rapid word stream of 80 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word\n  - Target 2 (T2): appears shortly after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. corridor\n2. balcony\n3. gallery\n4. factory\n5. ceiling\n6. blanket\n7. district\n8. lantern\n9. stairway\n10. window\n11. morning\n12. pavilion\n13. shelter\n14. platform\n15. column\n16. evening\n17. passage\n18. doorway\n19. river\n20. garden\n21. kitchen\n22. library\n23. harbor\n24. building\n25. table\n26. curtain\n27. signal\n28. chapter\n29. street\n30. fountain\n31. cabinet\n32. surface\n33. thirteen\n34. GARNET\n35. village\n36. market\n37. station\n38. terrace\n39. monument\n40. bridge\n41. corner\n42. pattern\n43. chamber\n44. highway\n45. chimney\n46. ceiling\n47. kitchen\n48. district\n49. corridor\n50. station\n51. platform\n52. corner\n53. building\n54. curtain\n55. passage\n56. doorway\n57. ceiling\n58. street\n59. garden\n60. market\n61. bridge\n62. factory\n63. market\n64. chamber\n65. passage\n66. balcony\n67. highway\n68. balcony\n69. surface\n70. factory\n71. lantern\n72. chapter\n73. bridge\n74. cabinet\n75. monument\n76. corner\n77. stairway\n78. table\n79. ceiling\n80. signal\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"thirteen\", \"t2\": \"GARNET\"}"
 },
 {
  "task_id": "blink_frontier_038",
  "task_type": "blink",
  "difficulty": "Frontier",
  "prompt": "Below is a rapid word stream of 80 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word\n  - Target 2 (T2): appears shortly after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. gallery\n2. chamber\n3. window\n4. pavilion\n5. factory\n6. lantern\n7. pattern\n8. blanket\n9. kitchen\n10. market\n11. shelter\n12. terrace\n13. doorway\n14. corridor\n15. evening\n16. corner\n17. stairway\n18. monument\n19. garden\n20. seven-hundred\n21. EMERALD\n22. passage\n23. ceiling\n24. chimney\n25. cabinet\n26. signal\n27. river\n28. column\n29. harbor\n30. surface\n31. curtain\n32. district\n33. library\n34. platform\n35. village\n36. station\n37. building\n38. fountain\n39. table\n40. balcony\n41. highway\n42. street\n43. morning\n44. bridge\n45. chapter\n46. stairway\n47. village\n48. shelter\n49. lantern\n50. factory\n51. village\n52. chimney\n53. market\n54. doorway\n55. factory\n56. river\n57. ceiling\n58. cabinet\n59. passage\n60. harbor\n61. doorway\n62. morning\n63. evening\n64. lantern\n65. street\n66. highway\n67. passage\n68. shelter\n69. ceiling\n70. signal\n71. station\n72. monument\n73. harbor\n74. garden\n75. balcony\n76. surface\n77. village\n78. doorway\n79. lantern\n80. corner\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"seven-hundred\", \"t2\": \"EMERALD\"}"
 },
 {
  "task_id": "blink_frontier_039",
  "task_type": "blink",
  "difficulty": "Frontier",
  "prompt": "Below is a rapid word stream of 80 items. Most words are common nouns. Two special target items are hidden in the stream:\n  - Target 1 (T1): an ALL-CAPS word\n  - Target 2 (T2): appears shortly after T1\n\nRead the entire stream carefully, then report both targets.\n\nWord stream:\n1. garden\n2. passage\n3. building\n4. gallery\n5. village\n6. balcony\n7. pavilion\n8. factory\n9. fountain\n10. district\n11. monument\n12. platform\n13. cabinet\n14. harbor\n15. station\n16. table\n17. surface\n18. kitchen\n19. evening\n20. chamber\n21. column\n22. stairway\n23. curtain\n24. chapter\n25. lantern\n26. blanket\n27. corridor\n28. pattern\n29. morning\n30. window\n31. doorway\n32. four-thousand\n33. OPAL\n34. terrace\n35. ceiling\n36. bridge\n37. chimney\n38. street\n39. corner\n40. library\n41. market\n42. shelter\n43. signal\n44. highway\n45. river\n46. gallery\n47. monument\n48. district\n49. shelter\n50. monument\n51. library\n52. monument\n53. doorway\n54. curtain\n55. chapter\n56. street\n57. cabinet\n58. window\n59. surface\n60. signal\n61. street\n62. building\n63. chimney\n64. column\n65. garden\n66. platform\n67. factory\n68. table\n69. gallery\n70. district\n71. district\n72. morning\n73. river\n74. surface\n75. street\n76. lantern\n77. cabinet\n78. monument\n79. column\n80. harbor\n\nANSWER:\nT1: [the first target word]\nT2: [the second target word]",
  "gold_json": "{\"t1\": \"four-thousand\", \"t2\": \"OPAL\"}"
 }
]
''')

print(f"Loaded {len(DATASET)} items")
for tt in ['capacity', 'interference', 'blink']:
    count = sum(1 for d in DATASET if d["task_type"] == tt)
    print(f"  {tt}: {count} items")


In [ ]:
# ══════════════════════════════════════════════════════════════════════
# Cell 4: Execution Loop
# ══════════════════════════════════════════════════════════════════════

TASK_DISPATCH = {
    "capacity": cogattention_capacity,
    "interference": cogattention_interference,
    "blink": cogattention_blink,
}

n_total = len(DATASET)
for i, item in enumerate(DATASET):
    task_fn = TASK_DISPATCH[item["task_type"]]
    print(f"[{i+1}/{n_total}] {item['task_id']} ({item['difficulty']})")
    task_fn.run(
        llm=kbench.llm,
        prompt=item["prompt"],
        gold_json=item["gold_json"],
        task_id=item["task_id"],
        difficulty=item["difficulty"],
    )

print(f"\nCompleted {n_total} items for Attention Capacity")
